# Parcel Update
* Amy Fish, afish@trpa.gov
* Andy McClary, amcclary@trpa.gov
* Mason Bindl, mbindl@trpa.gov

## Setup

### Imports, Functions, and Global Variables

In [19]:
# import packages
import urllib
import json
import requests
import os
import shutil
import sys
import re
import logging

from datetime import datetime 
import time
from zipfile import ZipFile
from io import BytesIO

import pandas as pd
import pyodbc

import arcpy
from arcgis.features import GeoAccessor, GeoSeriesAccessor
from arcgis.gis import GIS

from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# import traceback
# from pytz import timezone
# import pytz
# import pathlib
# from IPython.display import display
# import getpass
# from time import strftime
# import linecache
# import ssl

# environment settings
arcpy.env.workspace = "//Trpa-fs01/GIS/PARCELUPDATE/Workspace/ParcelStaging.gdb"
arcpy.env.overwriteOutput = True
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(26910)

# set workspace and sde connections 
workspace = "//Trpa-fs01/GIS/PARCELUPDATE/Workspace/Staging"

# network path to connection files
filePath = "C:\GIS\DB_CONNECT"
# database file path 
sdeBase    = os.path.join(filePath, "Vector.sde")
sdeCollect = os.path.join(filePath, "Collection.sde")
sdeTabular = os.path.join(filePath, "Tabular.sde")


### Functions ###

# set none to '' for all cells
def replace_null_values_with_blank(fc): 
    with arcpy.da.UpdateCursor(fc, "*") as cursor: 
        for row in cursor: 
            for i in range(len(row)): 
                if row[i] is None: 
                    row[i] = "" 
                    cursor.updateRow(row)
                    

# combine duplicate records 
def CombineAPNs(fc, fld_dissolve):    
    from time import strftime  
    print ("Started combining APNs: " + strftime("%Y-%m-%d %H:%M:%S"))

    # get unique values from field
    value_list = [r[0] for r in arcpy.da.SearchCursor(fc, (fld_dissolve))]
    unique_vals = list(set(value_list))
    
    if len(value_list) !=len(unique_vals):
        seen = set()
        dup_vals = set()
        for x in value_list:
            if x in seen:
                dup_vals.add(x)
            else:
                seen.add(x)
        print(dup_vals)
        dup_vals.remove('')
        for unique_val in dup_vals:
            geoms = [r[0] for r in arcpy.da.SearchCursor(fc, ('SHAPE@', fld_dissolve)) if r[1] == unique_val]
#Probably don't need this as there will always be more than one geometry
            if len(geoms) > 1:
                print(unique_val)    
                diss_geom = DissolveGeoms(geoms)

                # update the first feature with new geometry and delete the others
                where = "{} = '{}'".format(fld_dissolve, unique_val)
                cnt = 0
                with arcpy.da.UpdateCursor(fc, ('SHAPE@'), where) as curs:
                    for row in curs:
                        cnt += 1
                        if cnt == 1:
                            row[0] = diss_geom
                            curs.updateRow(row)
                        else:
                            curs.deleteRow()
    else:
        print("No duplicates!")
    print ("Finished combining APNs: " + strftime("%Y-%m-%d %H:%M:%S"))
    
#
def DissolveGeoms(geoms):
    cnt = 0
    for geom in geoms:
        cnt += 1
        if cnt == 1:
            diss_geom = geom
        else:
            diss_geom = diss_geom.union(geom)
    return diss_geom

# moves attribute values from one feature class to the other using an aspatial join
def fieldJoinCalc(updateFC, updateFieldsList, sourceFC, sourceFieldsList):
    from time import strftime  
    print ("Started data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))
#     log.info("Started data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))
    # Use list comprehension to build a dictionary from arcpy SearchCursor  
    valueDict = {r[0]:(r[1:]) for r in arcpy.da.SearchCursor(sourceFC, sourceFieldsList)}  
   
    with arcpy.da.UpdateCursor(updateFC, updateFieldsList) as updateRows:  
        for updateRow in updateRows:  
            # store the Join value of the row being updated in a keyValue variable  
            keyValue = updateRow[0]  
            # verify that the keyValue is in the Dictionary  
            if keyValue in valueDict:  
                # transfer the value stored under the keyValue from the dictionary to the updated field.  
                updateRow[1] = valueDict[keyValue][0]  
                updateRows.updateRow(updateRow)    
    del valueDict  
    print ("Finished data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))
#     log.info("Finished data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))

def differenceDictionary(df1, df2, key_field):
#Generate a list of columns in common
    common_columns = list(set(df1.columns) & set(df2.columns))

# keep only the common columns in both dataframes
    df1 = df1[common_columns]
    df2 = df2[common_columns]
    df1 = df1.set_index(key_field)
    df2 = df2.set_index(key_field)
    df1.sort_index(inplace=True)
    df2.sort_index(inplace=True)

    diff_df = df1.compare(df2)

    new_values =diff_df.loc[:,pd.IndexSlice[:,'other']].droplevel(1,axis=1)

    dict_update = new_values.to_dict('index')
    new_dict = {k: {a: b for a, b in v.items() if not pd.isnull(b)} for k, v in dict_update.items()}
    return new_dict    

def update_fc_from_dict(update_dict,key_field, fc):
    #This gets our update cursor down to fields that need to be updated
    update_fields = set(field for values in update_dict.values() for field in values.keys())
    # create a SQL query to filter the feature class based on the key field values
    key_field_values = tuple(update_dict.keys())
    #Had to get rid of the query for branch versioning because of restiricitons on the length of GET Request
    #Maybe worth thinking about trying to get it to work at some point
    #sql_query = f"{arcpy.AddFieldDelimiters(fc, key_field)} IN {key_field_values}"
    
    #print(sql_query)
    print("Updating Attributes started: " + strftime("%Y-%m-%d %H:%M:%S"))
    # update the attributes using the nested dictionary
    with arcpy.da.UpdateCursor(fc, [key_field] + list(update_fields)) as cursor:
        for row in cursor:
            key_field_value = row[0]
            if key_field_value in update_dict:
                update_values = update_dict[key_field_value]
                for field, value in update_values.items():
                    index = cursor.fields.index(field)
                    row[index] = value
                cursor.updateRow(row)
    print("Updating Attributes Finished: " + strftime("%Y-%m-%d %H:%M:%S"))
    
# Parcel AOI to select parcels to keep (includes TRPA Boundary and Olympic Valley Watershed)
parcelAOI = "Parcel_AOI"

#sde feature classes to use in attribution stage
sde_Impervious       = sdeBase + "\\sde.SDE.Impervious\\sde.SDE.Impervious_2019"
sde_Bailey           = sdeBase + "\\sde.SDE.Soils\sde.SDE.land_capability_Bailey_Soils"
sde_RegionalLandUse  = sdeBase + "\\sde.SDE.Planning\\sde.SDE.RegionalLandUse"
sde_NRCSSoils1974    = sdeBase + "\\sde.SDE.Soils\\sde.SDE.NRCS_Soils_1974"
sde_NRCSSoils2003    = sdeBase + "\\sde.SDE.Soils\\sde.SDE.NRCS_Soils_2003"
sde_Catchment        = sdeBase + "\\sde.SDE.WaterQuality\\sde.SDE.TMDL_Catchment"
sde_HydroArea        = sdeBase + "\\sde.SDE.Water\\sde.SDE.Hydro_Areas"
sde_Watershed        = sdeBase + "\\sde.SDE.Water\\sde.SDE.Watershed"
sde_FireDistrict     = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.FireDistricts"
sde_LocalPlan        = sdeBase + "\\sde.SDE.Planning\\sde.SDE.LocalPlan"
sde_SpecialDistrict  = sdeBase + "\\sde.SDE.Planning\\sde.SDE.SpecialPlanningDistrict"
sde_CSLT             = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.CSLT"
sde_CurrentParcels   = sdeBase + "\\sde.SDE.Parcels\\sde.SDE.Parcel_Master"
sde_Zoning           = sdeBase + "\\sde.SDE.Planning\\sde.SDE.District"
sde_TownCenter       = sdeBase + "\\sde.SDE.Planning\\sde.SDE.TownCenter"
sde_TownCenterBuffer = sdeBase + "\\sde.SDE.Planning\\sde.SDE.TownCenter_Buffer"
sde_Index1987        = sdeBase + "\\sde.SDE.Index\\sde.SDE.AssessorMapIndex_1987"
sde_TRPAboundary     = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.TRPA_bdy"
sde_BonusUnitboundary= sdeBase + "\\sde.SDE.Planning\\sde.SDE.Bonus_unit_boundary"
sde_UrbanArea        = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.UrbanAreas"
sde_Zip              = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.Postal_ZIP"
sde_TAZ              = sdeBase + "\\sde.SDE.Transportation\\sde.SDE.Transportation_Analysis_Zone"
sde_Littoral         = sdeBase + "\\sde.SDE.Shorezone\\sde.SDE.LittoralParcel"
sde_Tolerance        = sdeBase + "\\sde.SDE.Shorezone\\sde.SDE.Tolerance_District"

# in memory fcs to use in the attribution stage
memory = "memory" + "\\"
ParcelPoint_RegionalLandUse = memory + "ParcelPoint_RegionalLandUse"
ParcelPoint_Soils74         = memory + "ParcelPoint_Soils74"
ParcelPoint_Soils03         = memory + "ParcelPoint_Soils03"
ParcelPoint_Catchment       = memory + "ParcelPoint_Catchment"
ParcelPoint_HydroArea       = memory + "ParcelPoint_HydroArea"
ParcelPoint_Watershed       = memory + "ParcelPoint_Watershed"
ParcelPoint_FireDistrict    = memory + "ParcelPoint_FireDistrict"
ParcelPoint_LocalPlan       = memory + "ParcelPoint_LocalPlan"
ParcelPoint_TownCenter      = memory + "ParcelPoint_TownCenter"
ParcelPoint_TownCenterBuffer= memory + "ParcelPoint_TownCenterBuffer"
ParcelPoint_Zoning          = memory + "ParcelPoint_Zoning"
ParcelPoint_SpecialDistrict = memory + "ParcelPoint_SpecialDistrict"
ParcelPoint_Index1987       = memory + "ParcelPoint_Index1987"
ParcelPoint_PstlTown        = memory + "ParcelPoint_PstlTown"
ParcelPoint_PstlZip         = memory + "ParcelPoint_PstlZip"
ParcelPoint_CSLT            = memory + "ParcelPoint_CSLT"
ParcelPoint_TAZ             = memory + "ParcelPoint_TAZ"
ParcelPoint_Design          = memory + "ParcelPoint_Design"
ParcelPoint_Littoral        = memory + "ParcelPoint_Littoral"
ParcelPoint_Tolerance       = memory + "ParcelPoint_Tolerance"

# Set up fields to add to FGDB.
baseFields = [
# apn ppno
['APN_TRPA', 'TEXT', 'APN', 50],
['PPNO_TRPA', 'DOUBLE','PPNO'],
['JURISDICTION_TRPA', 'TEXT', 'Jurisdiction', 4],
['COUNTY_TRPA', 'TEXT', 'County', 2],
 # parcel address   
['HSE_NUMBR_TRPA', 'TEXT', 'House Number', 25],
['UNIT_NUMBR_TRPA', 'TEXT', 'Unit Number', 50],
['STR_DIR_TRPA', 'TEXT','Street Direction', 5],
['STR_NAME_TRPA', 'TEXT', 'Street Name', 100],
['STR_SUFFIX_TRPA', 'TEXT', 'Street Suffix', 6],
['APO_ADDRESS_TRPA', 'TEXT', 'Full Address', 100],
['PSTL_TOWN_TRPA', 'TEXT', 'Postal Town', 25],
['PSTL_STATE_TRPA', 'TEXT', 'Postal State', 2],
['PSTL_ZIP5_TRPA', 'TEXT', 'Postal Zip Code', 5],
# owner info
['OWN_FIRST_TRPA', 'TEXT', 'Owner First Name', 255],
['OWN_LAST_TRPA', 'TEXT', 'Owner Last Name', 255],
['OWN_FULL_TRPA', 'TEXT', 'Owner Name', 255],
    # swap this in soon
# ['OWNER_NAME_TRPA', 'TEXT', 'Owner Name', 255],
['MAIL_ADD1_TRPA', 'TEXT', 'Mailing Address', 100],
['MAIL_CITY_TRPA', 'TEXT', 'Mailing City', 50],
['MAIL_STATE_TRPA', 'TEXT', 'Mailing State', 25],
['MAIL_ZIP5_TRPA', 'TEXT', 'Mailing Zip Code', 50],
# value fields  
['AS_LANDVALUE_TRPA', 'LONG','Assessed Land Value'],
['AS_IMPROVALUE_TRPA', 'LONG','Assessed Improved Value'],
['AS_SUM_TRPA', 'LONG', 'Assessed Sum Value'],
['TAX_LANDVALUE_TRPA', 'LONG','Tax Land Value'],
['TAX_IMPROVALUE_TRPA', 'LONG','Tax Improved Value'],
['TAX_SUM_TRPA', 'LONG','Tax Sum'],
['TAX_YEAR_TRPA', 'TEXT','Tax Year', 5],
# jurisdiction land use fields
['COUNTY_LANDUSE_CODE_TRPA', 'TEXT', 'County Landuse Code', 50],
['COUNTY_LANDUSE_TRPA', 'TEXT', 'County Landuse', 250],
# Fields for building info
["YEAR_BUILT_TRPA", "SHORT", 'Year Built', 5],
['UNITS_TRPA', 'DOUBLE', 'Units', 5],
["BEDROOMS_TRPA", "DOUBLE",'Bedrooms'],
['BATHROOMS_TRPA', 'DOUBLE', 'Bathrooms'],
['BUILDING_SQFT_TRPA', 'DOUBLE', 'Building Size'],
# fields to add? 
["VHR_TRPA", "TEXT", "Vacation Home Rental", 3],
["HOA_TRPA", "TEXT", "Home Owners Association", 3]
]

trpaFields = [
# land use
['OWNERSHIP_TYPE_TRPA', 'TEXT', 'Ownership Type', 50],
['EXISTING_LANDUSE_TRPA', 'TEXT', 'Existing Landuse', 50],
['REGIONAL_LANDUSE_TRPA', 'TEXT', 'Regional Landuse', 50], 
# Fields for soil, watershed, etc...
['ESTIMATED_COVERAGE_ALLOWED_TRPA', 'DOUBLE', "Estimate of Coverage Allowed (Bailey, sq.ft.)"],
['IMPERVIOUS_SURFACE_SQFT_TRPA', 'DOUBLE', "Impervious Surface (Remote Sensing, sq.ft.)"],
['SOIL_1974_TRPA', 'TEXT','NRCS Soils 1974', 5],
["SOIL_2003_TRPA", "TEXT", "NRCS Soils 2003", 5],
["CATCHMENT_TRPA", "TEXT", "Catchment", 150],
["HRA_NAME_TRPA", "TEXT", "Hydrologic Resource Area", 30],
["WATERSHED_NUMBER_TRPA", "SHORT", "Watershed Number"],
["WATERSHED_NAME_TRPA", "TEXT", "Watershed Name", 30],
["PRIORITY_WATERSHED_TRPA", "TEXT", "Priority Watershed", 2],
["FIREPD_TRPA", "TEXT", "Fire Protection District", 25],
# Fields for Planning purposes
["PLAN_ID_TRPA", "TEXT", 'Plan ID',8],
["PLAN_NAME_TRPA", "TEXT", 'Plan Name', 40],
["PLAN_TYPE_TRPA", "TEXT", 'Plan Type', 40],
["ZONING_ID_TRPA", "TEXT", 'Zoning ID', 50],
["ZONING_DESCRIPTION_TRPA", "TEXT", 'Zoning Description',500],
["TOWN_CENTER_TRPA", "TEXT",'Town Center', 50],
["LOCATION_TO_TOWNCENTER_TRPA", "TEXT", 'Location Relative to Town Center', 50],
["TOLERANCE_ID_TRPA", "TEXT", 'Tolerance ID', 50],
["TAZ_TRPA", "DOUBLE",'Transportation Analysis Zone'],
["INDEX_1987_TRPA", "TEXT", "1987 Parcel Map Index",10],
["LITTORAL_TRPA", "SHORT", "Littoral"],
["WITHIN_TRPA_BNDY_TRPA", "SHORT","Within TRPA Boundary?"],
["WITHIN_BONUSUNIT_BNDY_TRPA", "SHORT", "Within Bonus Unit Boundary"],
["LOCAL_PLAN_HYPERLINK_TRPA", "TEXT", "Local Plan Hyperlink", 255],
["DESIGN_GUIDELINES_HYPERLINK_TRPA", "TEXT", "Design Guidelines", 255],
["LTINFO_HYPERLINK_TRPA", "TEXT", "LTinfo Parcel Details", 255],
["INDEX_1987_HYPERLINK_TRPA", "TEXT", "Index 1987 Hyperlink", 255],
# Fields for Parcel Size
["PARCEL_ACRES_TRPA", "DOUBLE", "Acres"],
["PARCEL_SQFT_TRPA", "DOUBLE", "Square Feet"] 
]

### Logging

In [ ]:
# create logger
logger = logging.getLogger(__name__)
# set log level for all handlers to debug
logger.setLevel(logging.DEBUG)

# create console handler and set level to debug
# best for development or debugging
consoleHandler = logging.StreamHandler()
consoleHandler.setLevel(logging.DEBUG)

# create formatter
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')

# add formatter to ch
consoleHandler.setFormatter(formatter)

# add ch to logger
logger.addHandler(consoleHandler)

###################
# USE LOGGER
###################

# example usage
logger.debug('debug message')
logger.info('info message')
logger.warning('warn message')
logger.error('error message')
logger.critical('critical message')

In [44]:
parcelLog = 'ParcelETL_'+str(strftime("%m%d%Y"))+'.log'
print(parcelLog)

ParcelETL_05092023.log


In [54]:

###################
# SETUP LOGGER
###################
# create logger
logger = logging.getLogger(__name__)
# set log level for all handlers to debug
logger.setLevel(logging.DEBUG)

# create console handler and set level to debug
consoleHandler = logging.StreamHandler()
consoleHandler.setLevel(logging.DEBUG)

# create file handler and set level to debug
parcelLog = 'ParcelETL_'+str(strftime("%m%d%Y"))+'.log'
fileHandler = logging.FileHandler(parcelLog)
fileHandler.setLevel(logging.DEBUG)

# create formatter
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')

# add formatter to handlers
consoleHandler.setFormatter(formatter)
fileHandler.setFormatter(formatter)

# add handlers to logger
logger.addHandler(consoleHandler)
logger.addHandler(fileHandler)

# set log level for development
logger.setLevel(logging.INFO)

###################
# USE LOGGER
###################

# context/ session information
context_user = "User XYZ"
a = 50
b = 50

# calculation
logger.debug("User %s provided the numbers %s and %s for calculation", context_user, a, b)
try:
    result = a / b
    logger.info("Calculation successful for user %s with a result of: %s", context_user, result)
except ZeroDivisionError as error:
    logger.error("Calculation was unsuccessful for user %s with the inputs %s and %s", context_user, a, b)
    logger.error(error)

2023-05-09 21:35:40,595 - __main__ - INFO - Calculation successful for user User XYZ with a result of: 1.0
2023-05-09 21:35:40,595 - __main__ - INFO - Calculation successful for user User XYZ with a result of: 1.0
2023-05-09 21:35:40,595 - __main__ - INFO - Calculation successful for user User XYZ with a result of: 1.0
2023-05-09 21:35:40,595 - __main__ - INFO - Calculation successful for user User XYZ with a result of: 1.0


In [52]:
# Set logging.
def setup_logging(log_filename,logging_level):
    log_format = "%(asctime)s %(levelname)-8s %(message)s"
    log_date_format = "%a, %d %b %Y %H:%M:%S"
    logging.basicConfig(filename=log_filename,
                        filemode='a',
                        level=logging_level,
                        format=log_format,
                        datefmt=log_date_format)
    
# Set up logging.
logDate = strftime("%Y%m%d")
logFilename = os.path.join(workspace, parcelLog)
setup_logging(logFilename, 'INFO')
logging.info('Starting Script')

In [56]:
logging.info("What")
logger.debug("how")

## Extract

### Carson City County

In [27]:
## Carson City County GET Data
# parameters for get data from rest service
params = {'where': '1=1', 'outFields': '*', 'f': 'pjson', 'returnGeometry': True}
r = requests.get('https://gis.carson.org/arcgis/rest/services/CarsonCity/CarsonCityNV_OpenData/FeatureServer/36/query', params)
data = r.json()

# save JSON as a Feature class
json_path = os.path.join(workspace,'CCtemp.json')

# delete existing/old json file
os.remove(json_path)

# open and write data to json file
with open(json_path, 'w') as f:
    json.dump(data, f)

# delete the existing table
arcpy.management.Delete('Parcel_CC_Features')
print("Deleted existing table")

# json object to table
arcpy.JSONToFeatures_conversion(json_path, 'Parcel_CC_Features')
print("Saved CC Staging Feature class")

# get data from rest service
params = {'where': '1=1', 'outFields': '*', 'f': 'pjson', 'returnGeometry': True}
r = requests.get('https://gis.carson.org/arcgis/rest/services/CarsonCity/CarsonCityNV_OpenData/FeatureServer/42/query', params)
data = r.json()

# save JSON as a Feature class
json_path = os.path.join(workspace,'CCtemp.json')

# delete existing/old json file
os.remove(json_path)

# open and write data to json file
with open(json_path, 'w') as f:
    json.dump(data, f)

# delete the existing table
arcpy.management.Delete('Parcel_CC_Table')
print("Deleted existing table")

# json object to table
arcpy.JSONToFeatures_conversion(json_path, 'Parcel_CC_Table')
print("Saved CC Staging Table")


# The qualifiedFieldNames environment is used by Copy Features when persisting 
# the join field names.
arcpy.env.qualifiedFieldNames = False

# Set local variables
inFeatures = "Parcel_CC_Features"
joinTable  = "Parcel_CC_Table"
joinField  = "APN"
outFeature = "Parcel_CC_Extracted"

# Join the feature layer to a table
cc_join = arcpy.management.AddJoin(inFeatures, 
                                           joinField, 
                                           joinTable, 
                                           joinField)

# Copy the joined layer to a new permanent feature class
arcpy.management.CopyFeatures(cc_join, outFeature)
print("Carson Parcels Extracted")

Deleted existing table
Saved CC Staging Feature class
Deleted existing table
Saved CC Staging Table
Carson Parcels Extracted


### Douglas County

In [28]:
baseURL = "https://gisservices.douglasnv.us/server/rest/services/TRPA_Parcels/FeatureServer/0"
fields = "*"
outfc = "Parcel_DG_Extracted"

# Get record extract limit
urlstring = baseURL + "?f=json"
j = urllib.request.urlopen(urlstring)
js = json.load(j)
maxrc = int(js["maxRecordCount"])
print("Record extract limit: %s" % maxrc)

# Get object ids of features
where = "1=1"
urlstring = baseURL + "/query?where={}&returnIdsOnly=true&f=json".format(where)
j = urllib.request.urlopen(urlstring)
js = json.load(j)
idfield = js["objectIdFieldName"]
idlist = js["objectIds"]
idlist.sort()
numrec = len(idlist)
print("Number of target records: %s" % numrec)

# Gather features
print ("Gathering records...")
fs = dict()
for i in range(0, numrec, maxrc):
    torec = i + (maxrc - 1)
    if torec > numrec:
        torec = numrec - 1
    fromid = idlist[i]
    toid = idlist[torec]
    where = "{} >= {} and {} <= {}".format(idfield, fromid, idfield, toid)
    print ("  {}".format(where))
    urlstring = baseURL + "/query?where={}&returnGeometry=true&outFields={}&f=json".format(where,fields)
    # build that feature set!
    fs[i] = arcpy.FeatureSet()
    fs[i].load(urlstring)

# Save features
print("Saving features...")
fslist = []
for key,value in fs.items():
    fslist.append(value)
arcpy.Merge_management(fslist, outfc)

print("Douglas Parcels Extracted")

Record extract limit: 10000
Number of target records: 30130
Gathering records...
  OBJECTID >= 512177 and OBJECTID <= 522176
  OBJECTID >= 522177 and OBJECTID <= 532176
  OBJECTID >= 532177 and OBJECTID <= 542176
  OBJECTID >= 542177 and OBJECTID <= 542306
Saving features...
Douglas Parcels Extracted


### El Dorado County

In [26]:
# Set up Zip path.
zipPath = workspace
# setup output feature class
outfc = "Parcel_EL_Extracted"

# Check if zip from failed attempt still exists
existingZip = pathlib.Path(zipPath + r"\zipfolder")
if existingZip.exists():
    shutil.rmtree(zipPath + r"\zipfolder")
    logging.info('Previous zip folder deleted')

# Setup the params for the extraction GP tool. The boundary is a polygon the grabs the whole county.
payload = {'f': 'json', 'env:outSR': '6418', 'Layers_to_Clip': '["Parcels"]', 'Area_of_Interest': '{"geometryType":"esriGeometryPolygon","features":[{"geometry":{"rings":[[[-13490599.294393552,4646257.881632805],[-13490599.294393552,4735689.204726496],[-13336502.24537058,4735689.204726496],[-13336502.24537058,4646257.881632805],[-13490599.294393552,4646257.881632805]]],"spatialReference":{"wkid":102100}}}],"sr":{"wkid":102100}}', 'Feature_Format': 'File Geodatabase - GDB - .gdb'}

# Make the request to the GP service.
logging.info('Requesting parcels from EDC')
job = requests.get(r"https://see-eldorado.edcgov.us/arcgis/rest/services/uGOTNETandEXTRACTS/geoservices/GPServer/Extract%20Data%20Task/submitJob",params=payload)
jobJson = job.json()

# Check to make sure the job was accepted and get the JobID.
if 'jobId' in jobJson:
    jobID = jobJson['jobId']
    jobStatus = jobJson['jobStatus']
    jobURL = r"https://see-eldorado.edcgov.us/arcgis/rest/services/uGOTNETandEXTRACTS/geoservices/GPServer/Extract%20Data%20Task/jobs"
    if jobStatus == 'esriJobSubmitted' or jobStatus == 'esriJobExecuting':
        logging.info('EDC job submitted')

    # Check the status of the job, when done grab the resulting ZIP file link.
    while jobStatus == 'esriJobSubmitted' or jobStatus == 'esriJobExecuting':
        time.sleep(5)
        jobCheck = requests.get(jobURL+"/"+jobID+"?f=json")
        jobJson = jobCheck.json()
        if 'jobStatus' in jobJson:
            jobStatus = jobJson['jobStatus']
            if jobStatus == "esriJobSucceeded":
                if 'results' in jobJson:
                    logging.info('EDC server job completed')
                    resultURL = jobJson['results']['Output_Zip_File']['paramUrl']

                    # Grab the ZIP link.
                    logging.info('Downloading ZIP from EDC')
                    jobResult = requests.get(jobURL+"/"+jobID+r"/"+resultURL+r"?f=json&returnType=data")
            if jobStatus == "esriJobFailed":
                logging.error('EDC server job failure')
                if 'messages' in jobJson:
                    logging.error(jobJson['messages'])
                raise ValueError('EDC job failed!')

# Get the ZIP file.
parcelsZip = requests.get(jobResult.json()['value']['url'])
logging.info('Downloaded ZIP from EDC')

# Save the ZIP into memory.
zipFile = ZipFile(BytesIO(parcelsZip.content))

# Unzip the ZIP to the defined path.
for each in zipFile.namelist():
    if not each.endswith('/'):
        root, name = os.path.split(each)
        directory = os.path.normpath(os.path.join(zipPath, root))
        if not os.path.isdir(directory):
            os.makedirs(directory)
        open(os.path.join(directory, name), 'wb').write(zipFile.read(each))
logging.info('Unzipped files in ' + str(zipPath))

# Setup env for parcel FGDB and set overwrite to true.
zipFolder = zipPath + r"\zipfolder"
# arcpy.env.overwriteOutput = True
in_features = os.path.join(workspace, "zipfolder\data.gdb\Parcels")

# Export to staging gdb
arcpy.management.CopyFeatures(in_features, outfc)
print("El Dorado Parcels Extracted")

El Dorado Parcels Extracted


### Placer County 

In [29]:
#Parameters
hostedFeatureService = 'true'
agsService = 'false'

# ## Need to use TRPA Admin user for this ###
# username = 'mbindl'
# password = getpass.getpass()
username = 'TRPA_ADMIN'
password = 'TRP@g1sT3am'

baseURL = "https://services9.arcgis.com/NENkjkswKTzMfG3A/arcgis/rest/services/Placer_County_Assessment_Master_view/FeatureServer/2"
fields = "*"
outdata = "Parcel_PL_Table"
token = ''

# Disable warnings
requests.packages.urllib3.disable_warnings()

#Report error function
def PrintException():
    exc_type, exc_obj, tb = sys.exc_info()
    f = tb.tb_frame
    lineno = tb.tb_lineno
    filename = f.f_code.co_filename
    linecache.checkcache(filename)
    line = linecache.getline(filename, lineno, f.f_globals)
    arcpy.AddError('Error:  Line {} -- "{}": {}'.format(lineno, line.strip(), exc_obj))
    sys.exit()

#generate token for AGOL Hosted Feature Service

if username and password:
    try:
        tokenURL = 'https://www.arcgis.com/sharing/rest/generateToken'
        params = {'f': 'pjson', 'username': username, 'password': password, 'referer': 'https://www.arcgis.com', 'expiration': str(21600)}
        response = requests.post(tokenURL, data = params, verify = False)
        token = response.json()['token']
    except:
        PrintException()
else:
    token = ''

print('Token: '+token)

# Get record extract limit 
urlstring = baseURL + "?token="+token+"&f=json" 
j = requests.get(urlstring, verify=False)
js = j.json() 
maxrc = int(js["maxRecordCount"]) 
print("Record extract limit: %s" % maxrc)

# Get object ids of features
where = "1%3D1"
urlstring = baseURL + "/query?where=1%3D1&returnIdsOnly=true&f=json&token="+token
j = requests.get(urlstring, verify=True)
js = j.json() 
idfield = js["objectIdFieldName"]
idlist = js["objectIds"]
idlist.sort()
numrec = len(idlist)
print("Number of target records: %s" % numrec)

# Gather features
print ("Gathering records...")
fs = {}
for i in range(0, numrec, maxrc):
    torec = i + (maxrc - 1)
    if torec > numrec:
        torec = numrec - 1
    fromid = idlist[i]
    toid = idlist[torec]
    where = "{} >= {} and {} <= {}".format(idfield, fromid, idfield, toid)
    print ("  {}".format(where))
    urlstring = baseURL + f'/query?where={where}&outFields={fields}&f=json&token='+token
    fs[i] = arcpy.RecordSet()
    fs[i].load(urlstring)

# Save features
print("Saving features...")
fslist = []
for key,value in fs.items():
    fslist.append(value)
arcpy.Merge_management(fslist, outdata)
print("Done")


Token: bfgBu-lsW5uGQT2EdlKJg6hGOCuYy5d9gZmq64bs45xM4RcWL5aVkyU0CdqyP4zzuI7BGgpw5EeDzmbD0Uva3Terg8ZIYfx1THUzphGoV1nU4_IR7D56gZ45uOBC1x8HM4WOCihXXk6Pp1fn6ShFd05giw7NvbCKgQiVfqeeD0tjlKG5LOZFWyL8eMI4b_CU
Record extract limit: 2000
Number of target records: 172420
Gathering records...
  OBJECTID >= 1 and OBJECTID <= 2000
  OBJECTID >= 2001 and OBJECTID <= 4000
  OBJECTID >= 4001 and OBJECTID <= 6000
  OBJECTID >= 6001 and OBJECTID <= 8000
  OBJECTID >= 8001 and OBJECTID <= 10000
  OBJECTID >= 10001 and OBJECTID <= 12000
  OBJECTID >= 12001 and OBJECTID <= 14000
  OBJECTID >= 14001 and OBJECTID <= 16000
  OBJECTID >= 16001 and OBJECTID <= 18000
  OBJECTID >= 18001 and OBJECTID <= 20000
  OBJECTID >= 20001 and OBJECTID <= 22000
  OBJECTID >= 22001 and OBJECTID <= 24000
  OBJECTID >= 24001 and OBJECTID <= 26000
  OBJECTID >= 26001 and OBJECTID <= 28000
  OBJECTID >= 28001 and OBJECTID <= 30000
  OBJECTID >= 30001 and OBJECTID <= 32000
  OBJECTID >= 32001 and OBJECTID <= 34000
  OBJECTID >= 340

In [30]:
#Parameters
hostedFeatureService = 'true'
agsService = 'false'

# username and password to get the token via the AGOL shared group
# username = 'mbindl'
# password = getpass.getpass()
username = 'TRPA_ADMIN'
password = 'TRP@g1sT3am'

baseURL = "https://services9.arcgis.com/NENkjkswKTzMfG3A/arcgis/rest/services/Placer_County_Assessment_Master_view/FeatureServer/0"
fields = "*"
outdata = 'Parcel_PL_Features'
token = ''

# Disable warnings
requests.packages.urllib3.disable_warnings()

#Report error function
def PrintException():
    exc_type, exc_obj, tb = sys.exc_info()
    f = tb.tb_frame
    lineno = tb.tb_lineno
    filename = f.f_code.co_filename
    linecache.checkcache(filename)
    line = linecache.getline(filename, lineno, f.f_globals)
    arcpy.AddError('Error:  Line {} -- "{}": {}'.format(lineno, line.strip(), exc_obj))
    sys.exit()

#generate token for AGOL Hosted Feature Service

if username and password:
    try:
        tokenURL = 'https://www.arcgis.com/sharing/rest/generateToken'
        params = {'f': 'pjson', 'username': username, 'password': password, 'referer': 'https://www.arcgis.com', 'expiration': str(21600)}
        response = requests.post(tokenURL, data = params, verify = False)
        token = response.json()['token']
    except:
        PrintException()
else:
    token = ''

print('Token: '+token)

# Get record extract limit 
urlstring = baseURL + "?token="+token+"&f=json" 
j = requests.get(urlstring, verify=False)
js = j.json() 
maxrc = int(js["maxRecordCount"]) 
print("Record extract limit: %s" % maxrc)

# Get object ids of features
where = "1%3D1"
urlstring = baseURL + "/query?where=1%3D1&returnIdsOnly=true&f=json&token="+token
j = requests.get(urlstring, verify=True)
js = j.json() 
idfield = js["objectIdFieldName"]
idlist = js["objectIds"]
idlist.sort()
numrec = len(idlist)
print("Number of target records: %s" % numrec)

# Gather features
print ("Gathering records...")
fs = {}
for i in range(0, numrec, maxrc):
    torec = i + (maxrc - 1)
    if torec > numrec:
        torec = numrec - 1
    fromid = idlist[i]
    toid = idlist[torec]
    where = "{} >= {} and {} <= {}".format(idfield, fromid, idfield, toid)
    print ("  {}".format(where))
    urlstring = baseURL + f'/query?where={where}&returnGeometry=true&outFields={fields}&f=json&token='+token
    fs[i] = arcpy.FeatureSet()
    fs[i].load(urlstring)

# Save features
print("Saving features...")
fslist = []
for key,value in fs.items():
    fslist.append(value)
arcpy.Merge_management(fslist, outdata)
print("Done")


Token: 6RFshjt35c3i3rZwgbB32mA7fhpaazd0Xh09fFoWvJ-gTfY2z379fjTC-dCHoVXrdPzThHg9gRiXlyfTUGWraeHwvVX0A5Zc9G7xQIzPM14pIisSIf7I5e9K4c3SHuUPz6KlUSdcoZMo-THyPbPMBbM-G3WV1wISBS3DgbLZdeo8r81WHYzuG5lw3Pkhw5qw
Record extract limit: 2000
Number of target records: 192507
Gathering records...
  OBJECTID >= 1 and OBJECTID <= 2000
  OBJECTID >= 2001 and OBJECTID <= 4000
  OBJECTID >= 4001 and OBJECTID <= 6000
  OBJECTID >= 6001 and OBJECTID <= 8000
  OBJECTID >= 8001 and OBJECTID <= 10000
  OBJECTID >= 10001 and OBJECTID <= 12000
  OBJECTID >= 12001 and OBJECTID <= 14000
  OBJECTID >= 14001 and OBJECTID <= 16000
  OBJECTID >= 16001 and OBJECTID <= 18000
  OBJECTID >= 18001 and OBJECTID <= 20000
  OBJECTID >= 20001 and OBJECTID <= 22000
  OBJECTID >= 22001 and OBJECTID <= 24000
  OBJECTID >= 24001 and OBJECTID <= 26000
  OBJECTID >= 26001 and OBJECTID <= 28000
  OBJECTID >= 28001 and OBJECTID <= 30000
  OBJECTID >= 30001 and OBJECTID <= 32000
  OBJECTID >= 32001 and OBJECTID <= 34000
  OBJECTID >= 340

In [31]:
#Parameters
hostedFeatureService = 'true'
agsService = 'false'

## Need to use TRPA Admin user for this ###
username = 'TRPA_ADMIN'
password = 'TRP@g1sT3am'

baseURL = "https://services9.arcgis.com/NENkjkswKTzMfG3A/arcgis/rest/services/Placer_County_Assessment_Master_view/FeatureServer/3"
fields = "*"
outdata = "Parcel_PL_Table_Mega"
token = ''

# Disable warnings
requests.packages.urllib3.disable_warnings()

#Report error function
def PrintException():
    exc_type, exc_obj, tb = sys.exc_info()
    f = tb.tb_frame
    lineno = tb.tb_lineno
    filename = f.f_code.co_filename
    linecache.checkcache(filename)
    line = linecache.getline(filename, lineno, f.f_globals)
    arcpy.AddError('Error:  Line {} -- "{}": {}'.format(lineno, line.strip(), exc_obj))
    sys.exit()

#generate token for AGOL Hosted Feature Service

if username and password:
    try:
        tokenURL = 'https://www.arcgis.com/sharing/rest/generateToken'
        params = {'f': 'pjson', 'username': username, 'password': password, 'referer': 'https://www.arcgis.com', 'expiration': str(21600)}
        response = requests.post(tokenURL, data = params, verify = False)
        token = response.json()['token']
    except:
        PrintException()
else:
    token = ''

print('Token: '+token)

# Get record extract limit 
urlstring = baseURL + "?token="+token+"&f=json" 
j = requests.get(urlstring, verify=False)
js = j.json() 
maxrc = int(js["maxRecordCount"]) 
print("Record extract limit: %s" % maxrc)

# Get object ids of features
where = "1%3D1"
urlstring = baseURL + "/query?where=1%3D1&returnIdsOnly=true&f=json&token="+token
j = requests.get(urlstring, verify=True)
js = j.json() 
idfield = js["objectIdFieldName"]
idlist = js["objectIds"]
idlist.sort()
numrec = len(idlist)
print("Number of target records: %s" % numrec)

# Gather features
print ("Gathering records...")
fs = {}
for i in range(0, numrec, maxrc):
    torec = i + (maxrc - 1)
    if torec > numrec:
        torec = numrec - 1
    fromid = idlist[i]
    toid = idlist[torec]
    where = "{} >= {} and {} <= {}".format(idfield, fromid, idfield, toid)
    print ("  {}".format(where))
    urlstring = baseURL + f'/query?where={where}&outFields={fields}&f=json&token='+token
    fs[i] = arcpy.RecordSet()
    fs[i].load(urlstring)

# Save features
print("Saving features...")
fslist = []
for key,value in fs.items():
    fslist.append(value)
arcpy.Merge_management(fslist, outdata)
print("Done")


Token: uEgkbT_OG2GLtHBMahRFr7OLF8_mfPvDzR2JjQB7IAnqet9obkRq4z46X27XgR7n-j9eIjQukRe8xjP3oTBeix5vu4tebVEZeKaq9ShMjS-y7xscPUAlt6dx67bt9Mq9_VKeYlC6RKCr0GaV2enG4CysLVsvuH75ONdeT36uc1kRDWwPv6EilwiqeKqXgkG1
Record extract limit: 2000
Number of target records: 201713
Gathering records...
  OBJECTID >= 1 and OBJECTID <= 2000
  OBJECTID >= 2001 and OBJECTID <= 4000
  OBJECTID >= 4001 and OBJECTID <= 6000
  OBJECTID >= 6001 and OBJECTID <= 8000
  OBJECTID >= 8001 and OBJECTID <= 10000
  OBJECTID >= 10001 and OBJECTID <= 12000
  OBJECTID >= 12001 and OBJECTID <= 14000
  OBJECTID >= 14001 and OBJECTID <= 16000
  OBJECTID >= 16001 and OBJECTID <= 18000
  OBJECTID >= 18001 and OBJECTID <= 20000
  OBJECTID >= 20001 and OBJECTID <= 22000
  OBJECTID >= 22001 and OBJECTID <= 24000
  OBJECTID >= 24001 and OBJECTID <= 26000
  OBJECTID >= 26001 and OBJECTID <= 28000
  OBJECTID >= 28001 and OBJECTID <= 30000
  OBJECTID >= 30001 and OBJECTID <= 32000
  OBJECTID >= 32001 and OBJECTID <= 34000
  OBJECTID >= 340

In [32]:


# The qualifiedFieldNames environment is used by Copy Features when persisting 
# the join field names.
arcpy.env.qualifiedFieldNames = False

# Set local variables
inFeatures = "Parcel_PL_Features"
joinTable  = "Parcel_PL_Table_Mega"
joinField  = "ASMT"
outFeature = "Parcel_PL_Extracted"

# arcpy.management.CalculateField(inFeatures, "APN", 
#                                 '!APN!.replace("-","")', "PYTHON3")

# Join the feature layer to a table
pl_joined_table = arcpy.management.AddJoin(inFeatures, 
                                           joinField, 
                                           joinTable, 
                                           joinField)

# Copy the layer to a new permanent feature class
arcpy.management.CopyFeatures(pl_joined_table, outFeature)

# See field names and aliases
totalRecords = arcpy.management.GetCount(outFeature)
print('{} has {} records'.format(outFeature, totalRecords[0]))
resultFields = arcpy.ListFields(result)
print([field.name for field in resultFields])
print([field.aliasName for field in resultFields])

Parcel_PL_Extracted has 197408 records
['OBJECTID', 'Shape', 'FEEPARCEL', 'APN', 'GISAPN', 'BOOK', 'BOOK_PAGE', 'ROLL_YEAR', 'JURISDICTION', 'PARCEL_LEVEL', 'TRANSACTIO', 'TRA', 'PARCELTYPE', 'TAX_CD', 'TAXABLEX', 'TAX_DESC', 'USE_CD', 'USE_CD_N', 'ACRES', 'EFFECTIVEY', 'STR_SQFT', 'OWNER1', 'OWNER2', 'ADR1', 'ADR2', 'CITY', 'STATE', 'ZIP', 'STREETNUM', 'STREETNAME', 'STREETTYPE', 'STREETDIR', 'SP_APT', 'COMMUNITY', 'ASMT_DESC', 'LANDVALUE', 'STRUCTURE', 'NEIGHBORHOODCODE', 'APPRAISERID', 'SITUSID', 'ASMT', 'GIS_ACRES', 'OBJECTID_1', 'FEEPARCEL_1', 'TRANSACTIO_1', 'TRA_1', 'PARCELTYPE_1', 'TAX_CD_1', 'TAXABLEX_1', 'TAX_DESC_1', 'USE_CD_1', 'USE_CD_N_1', 'ACRES_1', 'EFFECTIVEY_1', 'STR_SQFT_1', 'OWNER1_1', 'OWNER2_1', 'ADR1_1', 'ADR2_1', 'CITY_1', 'STATE_1', 'ZIP_1', 'STREETNUM_1', 'STREETNAME_1', 'STREETTYPE_1', 'STREETDIR_1', 'SP_APT_1', 'COMMUNITY_1', 'ASMT_DESC_1', 'LANDVALUE_1', 'STRUCTURE_1', 'NEIGHBORHOODCODE_1', 'APPRAISERID_1', 'SITUSID_1', 'ASMT_1', 'Shape_Length', 'Shape_Area

### Washoe County 

In [33]:
baseURL = "https://wcgisweb.washoecounty.us/arcgis/rest/services/OpenData/OpenData/FeatureServer/0"
fields = "*"
outfc = "Parcel_WA_Extracted"

# Get record extract limit
urlstring = baseURL + "?f=json"
j = urllib.request.urlopen(urlstring)
js = json.load(j)
maxrc = int(js["maxRecordCount"])
print("Record extract limit: %s" % maxrc)

# Get object ids of features
where = "1=1"
urlstring = baseURL + "/query?where={}&returnIdsOnly=true&f=json".format(where)
j = urllib.request.urlopen(urlstring)
js = json.load(j)
idfield = js["objectIdFieldName"]
idlist = js["objectIds"]
idlist.sort()
numrec = len(idlist)
print("Number of target records: %s" % numrec)

# Gather features
print ("Gathering records...")
fs = dict()
for i in range(0, numrec, maxrc):
    torec = i + (maxrc - 1)
    if torec > numrec:
        torec = numrec - 1
    fromid = idlist[i]
    toid = idlist[torec]
    where = "{} >= {} and {} <= {}".format(idfield, fromid, idfield, toid)
    print ("  {}".format(where))
    urlstring = baseURL + "/query?where={}&returnGeometry=true&outFields={}&f=json".format(where,fields)
    # build that feature set!
    fs[i] = arcpy.FeatureSet()
    fs[i].load(urlstring)

# Save features
print("Saving features...")
fslist = []
for key,value in fs.items():
    fslist.append(value)
arcpy.Merge_management(fslist, outfc)
print("Done")

Record extract limit: 1000
Number of target records: 188673
Gathering records...
  OBJECTID >= 1 and OBJECTID <= 1000
  OBJECTID >= 1001 and OBJECTID <= 2000
  OBJECTID >= 2001 and OBJECTID <= 3000
  OBJECTID >= 3001 and OBJECTID <= 4000
  OBJECTID >= 4001 and OBJECTID <= 5000
  OBJECTID >= 5001 and OBJECTID <= 6000
  OBJECTID >= 6001 and OBJECTID <= 7000
  OBJECTID >= 7001 and OBJECTID <= 8000
  OBJECTID >= 8001 and OBJECTID <= 9000
  OBJECTID >= 9001 and OBJECTID <= 10000
  OBJECTID >= 10001 and OBJECTID <= 11000
  OBJECTID >= 11001 and OBJECTID <= 12000
  OBJECTID >= 12001 and OBJECTID <= 13000
  OBJECTID >= 13001 and OBJECTID <= 14000
  OBJECTID >= 14001 and OBJECTID <= 15000
  OBJECTID >= 15001 and OBJECTID <= 16000
  OBJECTID >= 16001 and OBJECTID <= 17000
  OBJECTID >= 17001 and OBJECTID <= 18000
  OBJECTID >= 18001 and OBJECTID <= 19000
  OBJECTID >= 19001 and OBJECTID <= 20000
  OBJECTID >= 20001 and OBJECTID <= 21000
  OBJECTID >= 21001 and OBJECTID <= 22000
  OBJECTID >= 220

Done


### Select Parcels to Keep for each County

In [34]:
# list of parcel staging layers to trim
parcelLayers = ["Parcel_CC_Extracted",
                "Parcel_DG_Extracted",
                "Parcel_EL_Extracted",
                "Parcel_PL_Extracted",
                "Parcel_WA_Extracted"]

# delete BS Parcels
parcelDelete = "ParcelDelete"

for parcel in parcelLayers:
    # Run MakeFeatureLayer
    arcpy.management.MakeFeatureLayer(parcel, parcelDelete)
    # select within clementini 
    arcpy.management.SelectLayerByLocation(parcelDelete, 
                                           "INTERSECT", 
                                           # includes TRPA Boundary and Olympic Valley Wateshed
                                           parcelAOI, '0', 
                                           "NEW_SELECTION", "INVERT")

    # Run GetCount and if some features have been selected, then 
    #  run DeleteFeatures to remove the selected features.
    deleteCount=arcpy.management.GetCount(parcelDelete)[0]
    if int(deleteCount) > 0:
        arcpy.management.DeleteFeatures(parcelDelete)
    # delete feature layer
    arcpy.management.Delete(parcelDelete)
    print("{} features deleted".format(deleteCount))

21112 features deleted
23843 features deleted
84158 features deleted
177778 features deleted
179257 features deleted


## Transform

### Combine

In [ ]:
# 
# get staging feature class and name output transformed feature class
in_features = "Parcel_CC_Extracted"
parcel_out  = "Parcel_CC_Transformed"

# in-memory feature class
carsonParcel = r"in_memory/inMemoryFeatureClass"

# copy feature class into in-memory feature class to work on
arcpy.management.CopyFeatures(in_features, carsonParcel)

# Add TRPA base fields
arcpy.management.AddFields(carsonParcel, baseFields)


# Do work.
with arcpy.da.UpdateCursor(carsonParcel, [
                                        ## TRPA base schema ##
                                        'APN_TRPA',                 #0
                                        'PPNO_TRPA',                #1
                                        'JURISDICTION_TRPA',        #2
                                         # parcel address   
                                        'HSE_NUMBR_TRPA',           #3
                                        'STR_DIR_TRPA',             #4
                                        'STR_NAME_TRPA',            #5
                                        'STR_SUFFIX_TRPA',          #6
                                        'UNIT_NUMBR_TRPA',          #7
                                        'APO_ADDRESS_TRPA',         #8
                                        'PSTL_TOWN_TRPA',           #9
                                        'PSTL_STATE_TRPA',          #10
                                        'PSTL_ZIP5_TRPA',           #11
                                        # owner fields
                                            # no first and last fields
                                        'OWN_FULL_TRPA',            #12
                                        'MAIL_ADD1_TRPA',           #13
                                        'MAIL_CITY_TRPA',           #14
                                        'MAIL_STATE_TRPA',          #15
                                        'MAIL_ZIP5_TRPA',           #16
                                        # value fields  
                                        'AS_LANDVALUE_TRPA',        #17
                                        'AS_IMPROVALUE_TRPA',       #18
                                        'AS_SUM_TRPA',              #19
                                        'TAX_LANDVALUE_TRPA',       #20 
                                        'TAX_IMPROVALUE_TRPA',      #21
                                        'TAX_SUM_TRPA',             #22
                                        'TAX_YEAR_TRPA',            #23
                                        # land use fields 
                                        'COUNTY_LANDUSE_CODE_TRPA', #24
                                        'COUNTY_LANDUSE_TRPA',      #25
                                        # Fields for building info
                                        "YEAR_BUILT_TRPA",          #26
                                        'UNITS_TRPA',               #27
                                        'BEDROOMS_TRPA',            #28
                                        'BATHROOMS_TRPA',           #29
                                        'BUILDING_SQFT_TRPA',       #30
                                        'VHR_TRPA',                 #31
                                        'HOA_TRPA',                 #32
                                        ###-------------------------###
                                        # County Fields to get data from
                                        'APN',   # apn              #33
                                        'APN_NUM',   # ppno         #34
                                        'Phy_Addr', #full adr       #35
                                        'Loc1', # house number      #36
                                        'Dir',# street dir          #37
                                        'Street_Name',# street name #38
                                        'Unit',  # unite Number     #39
                                        'Legal_Owner',  # Owner     #40
                                        'Mail_Addr',# mail address1 #41
                                        'Mail2_Addr',#mail address2 #42
                                        'MCity', # Mailing City     #43
                                        'MZip',  # Mailing Zip      #44
                                        'Land_Value',# land value   #45
                                        'Improv_Val',# improvedvalue#46
                                        'LU',    # land use code    #47
                                        'Total_DWUnits',  # units   #48

]) as cursor:
    # loop through each record and transform the values
    for row in cursor:
        # Set APN
        apn = row[33]
        if not (apn is None or apn == "" or apn.isspace() == True):
            row[0] = (apn[:3] + "-" + apn[3:6] + "-" + apn[6:8])
        else:
            row[0] = ''
            
        #PPNO
        ppno = row[34]
        if not (ppno is None):
            row[1] = int(ppno)
        else:
            row[1] = ''
            
        # Jurisdiction
        row[2] = "CC"
        
        # APO Address
        full_address = row[35]
        if not (full_address is None or full_address=='' or full_address.isspace()==True):
            row[8] = full_address
        else:
            row[8] = ''
        
        # House Number
        house = row[36]
        if not (house is None):
            row[3] = str(house)
        else:
            row[3] = ''
        
        # Street Direction
        street_direction = row[37]
        if not (street_direction is None or street_direction=='' or street_direction.isspace()==True):
            row[4] = street_direction
        else:
            row[4] = ''
            
        # Street Name
        street_name = row[38]
        if not (street_name is None or street_name =='' or street_name.isspace()==True):
            row[5] = street_name.split(" ",-1)[0]
        else:
            row[5] = ''
            
        # Street Suffix
        street_suffix = row[38]
        if not (street_suffix is None or street_suffix =='' or street_suffix.isspace()==True):
            row[6] = street_suffix.split(" ")[-1]
        else:
            row[6] = ''
            
        # Unit Number
        unit= row[39]
        if not (unit is None or unit=='' or unit.isspace()==True):
            row[7] = unit
        else:
            row[7] = ''
                    
        # Postal Town - see Search/Update Cursor below
        
        # Postal State
        row[10] = 'NV'
        
        # Postal Zip - See Search/Update Cursor below
        row[11] = ''    
        
        # Owner Name
        owner = row[40]
        if not (owner is None or owner == '' or owner.isspace()==True):
            row[12] = owner.strip()
        else:
            row[12] = ''

        # Mailing Address
        address1 = row[41]
        address2 = row[42]
        if not (address2 is None or address2 == '' or address2.isspace()==True):
            row[13] = str(address2).strip()
        elif (address2 is None or address2 == '' or address2.isspace()==True and address1 is None or address1 == '' or address1.isspace()==True):
            row[13] = str(address1).strip()
        else:
            row[13] = '' 
           
        # Mailing City
        mail_city = row[43]        
#         mail_city.split(',',1)[0]
        if not (mail_city is None or mail_city=='' or mail_city.isspace()==True):
            row[14] = mail_city.strip().split(',',1)[0]
        else:
            row[14] = ''
            
        # Mailing State
        mail_state = row[43]
        if not (mail_state is None or mail_state=='' or mail_state.isspace()==True):
            row[15] = mail_state.strip().rsplit(',')[-1]
        else:
            row[15] = ''
        
        # Mailing Zipcode
        mail_zip = row[44] 
        if (mail_zip is not None and len(mail_zip)>=5):
            row[16] = mail_zip[:5]
        else:
            row[16] = ''
            
        # Assessed Land Value
        land_value = row[45]
        if not(land_value is None):
            row[17] = land_value
        else:
            row[17] = 0
        
        # Assessed Improved Value
        improved_value = row[46]
        if not (improved_value is None):
            row[18] = improved_value
        else:
            row[18] = 0
                
        # Assessed Sum
        if not (land_value is None or improved_value is None):
            assessed_sum = improved_value + land_value
            row[19] = assessed_sum
        else:
            row[19] = None
        
        # Tax  Land Value
        taxland_value = row[45]
        if not(taxland_value is None):
            row[20] = taxland_value/0.35
        else:
            row[20] = None
        
        # Tax Improved Value
        taximproved_value = row[46]
        if not (taximproved_value is None):
            row[21] = taximproved_value/0.35
        else:
            row[21] = None
        
        # Tax Sum
        if not (land_value is None or improved_value is None):
            tax_sum = row[20]+row[21]
            row[22] = tax_sum
        else:
            row[22] = None
        
        # Tax Year
        tax_year =  datetime.now().year

        if not (tax_year is None):
            row[23] = tax_year
        else:
            row[23] = ''
            
        # County Land Use Code
        county_luc = row[47]
        if not (county_luc is None):
            row[24] = str(county_luc)
        else:
            row[24] = '' 
            
        # Units
        units = row[48]
        if not (units is None):
            row[27] = units
        else:
            row[27] = None
        
#         Update the row.
        cursor.updateRow(row)
del cursor

#arcpy.management.CopyFeatures(carsonParcel, parcel_out)

out_coordinate_system = arcpy.SpatialReference('NAD 1983 UTM Zone 10N') 
arcpy.Project_management(carsonParcel, parcel_out, out_coordinate_system)

print('New Carson Parcels transformed')

### Carson City County

In [35]:
# get staging feature class and name output transformed feature class
in_features = "Parcel_CC_Extracted"
parcel_out  = "Parcel_CC_Transformed"

# in-memory feature class
carsonParcel = r"in_memory/inMemoryFeatureClass"

# copy feature class into in-memory feature class to work on
arcpy.management.CopyFeatures(in_features, carsonParcel)

# Add TRPA base fields
arcpy.management.AddFields(carsonParcel, baseFields)


# Do work.
with arcpy.da.UpdateCursor(carsonParcel, [
                                        ## TRPA base schema ##
                                        'APN_TRPA',                 #0
                                        'PPNO_TRPA',                #1
                                        'JURISDICTION_TRPA',        #2
                                         # parcel address   
                                        'HSE_NUMBR_TRPA',           #3
                                        'STR_DIR_TRPA',             #4
                                        'STR_NAME_TRPA',            #5
                                        'STR_SUFFIX_TRPA',          #6
                                        'UNIT_NUMBR_TRPA',          #7
                                        'APO_ADDRESS_TRPA',         #8
                                        'PSTL_TOWN_TRPA',           #9
                                        'PSTL_STATE_TRPA',          #10
                                        'PSTL_ZIP5_TRPA',           #11
                                        # owner fields
                                            # no first and last fields
                                        'OWN_FULL_TRPA',            #12
                                        'MAIL_ADD1_TRPA',           #13
                                        'MAIL_CITY_TRPA',           #14
                                        'MAIL_STATE_TRPA',          #15
                                        'MAIL_ZIP5_TRPA',           #16
                                        # value fields  
                                        'AS_LANDVALUE_TRPA',        #17
                                        'AS_IMPROVALUE_TRPA',       #18
                                        'AS_SUM_TRPA',              #19
                                        'TAX_LANDVALUE_TRPA',       #20 
                                        'TAX_IMPROVALUE_TRPA',      #21
                                        'TAX_SUM_TRPA',             #22
                                        'TAX_YEAR_TRPA',            #23
                                        # land use fields 
                                        'COUNTY_LANDUSE_CODE_TRPA', #24
                                        'COUNTY_LANDUSE_TRPA',      #25
                                        # Fields for building info
                                        "YEAR_BUILT_TRPA",          #26
                                        'UNITS_TRPA',               #27
                                        'BEDROOMS_TRPA',            #28
                                        'BATHROOMS_TRPA',           #29
                                        'BUILDING_SQFT_TRPA',       #30
                                        'VHR_TRPA',                 #31
                                        'HOA_TRPA',                 #32
                                        ###-------------------------###
                                        # County Fields to get data from
                                        'APN',   # apn              #33
                                        'APN_NUM',   # ppno         #34
                                        'Phy_Addr', #full adr       #35
                                        'Loc1', # house number      #36
                                        'Dir',# street dir          #37
                                        'Street_Name',# street name #38
                                        'Unit',  # unite Number     #39
                                        'Legal_Owner',  # Owner     #40
                                        'Mail_Addr',# mail address1 #41
                                        'Mail2_Addr',#mail address2 #42
                                        'MCity', # Mailing City     #43
                                        'MZip',  # Mailing Zip      #44
                                        'Land_Value',# land value   #45
                                        'Improv_Val',# improvedvalue#46
                                        'LU',    # land use code    #47
                                        'Total_DWUnits',  # units   #48

]) as cursor:
    # loop through each record and transform the values
    for row in cursor:
        # Set APN
        apn = row[33]
        if not (apn is None or apn == "" or apn.isspace() == True):
            row[0] = (apn[:3] + "-" + apn[3:6] + "-" + apn[6:8])
        else:
            row[0] = ''
            
        #PPNO
        ppno = row[34]
        if not (ppno is None):
            row[1] = int(ppno)
        else:
            row[1] = ''
            
        # Jurisdiction
        row[2] = "CC"
        
        # APO Address
        full_address = row[35]
        if not (full_address is None or full_address=='' or full_address.isspace()==True):
            row[8] = full_address
        else:
            row[8] = ''
        
        # House Number
        house = row[36]
        if not (house is None):
            row[3] = str(house)
        else:
            row[3] = ''
        
        # Street Direction
        street_direction = row[37]
        if not (street_direction is None or street_direction=='' or street_direction.isspace()==True):
            row[4] = street_direction
        else:
            row[4] = ''
            
        # Street Name
        street_name = row[38]
        if not (street_name is None or street_name =='' or street_name.isspace()==True):
            row[5] = street_name.split(" ",-1)[0]
        else:
            row[5] = ''
            
        # Street Suffix
        street_suffix = row[38]
        if not (street_suffix is None or street_suffix =='' or street_suffix.isspace()==True):
            row[6] = street_suffix.split(" ")[-1]
        else:
            row[6] = ''
            
        # Unit Number
        unit= row[39]
        if not (unit is None or unit=='' or unit.isspace()==True):
            row[7] = unit
        else:
            row[7] = ''
                    
        # Postal Town - see Search/Update Cursor below
        
        # Postal State
        row[10] = 'NV'
        
        # Postal Zip - See Search/Update Cursor below
        row[11] = ''    
        
        # Owner Name
        owner = row[40]
        if not (owner is None or owner == '' or owner.isspace()==True):
            row[12] = owner.strip()
        else:
            row[12] = ''

        # Mailing Address
        address1 = row[41]
        address2 = row[42]
        if not (address2 is None or address2 == '' or address2.isspace()==True):
            row[13] = str(address2).strip()
        elif (address2 is None or address2 == '' or address2.isspace()==True and address1 is None or address1 == '' or address1.isspace()==True):
            row[13] = str(address1).strip()
        else:
            row[13] = '' 
           
        # Mailing City
        mail_city = row[43]        
#         mail_city.split(',',1)[0]
        if not (mail_city is None or mail_city=='' or mail_city.isspace()==True):
            row[14] = mail_city.strip().split(',',1)[0]
        else:
            row[14] = ''
            
        # Mailing State
        mail_state = row[43]
        if not (mail_state is None or mail_state=='' or mail_state.isspace()==True):
            row[15] = mail_state.strip().rsplit(',')[-1]
        else:
            row[15] = ''
        
        # Mailing Zipcode
        mail_zip = row[44] 
        if (mail_zip is not None and len(mail_zip)>=5):
            row[16] = mail_zip[:5]
        else:
            row[16] = ''
            
        # Assessed Land Value
        land_value = row[45]
        if not(land_value is None):
            row[17] = land_value
        else:
            row[17] = 0
        
        # Assessed Improved Value
        improved_value = row[46]
        if not (improved_value is None):
            row[18] = improved_value
        else:
            row[18] = 0
                
        # Assessed Sum
        if not (land_value is None or improved_value is None):
            assessed_sum = improved_value + land_value
            row[19] = assessed_sum
        else:
            row[19] = None
        
        # Tax  Land Value
        taxland_value = row[45]
        if not(taxland_value is None):
            row[20] = taxland_value/0.35
        else:
            row[20] = None
        
        # Tax Improved Value
        taximproved_value = row[46]
        if not (taximproved_value is None):
            row[21] = taximproved_value/0.35
        else:
            row[21] = None
        
        # Tax Sum
        if not (land_value is None or improved_value is None):
            tax_sum = row[20]+row[21]
            row[22] = tax_sum
        else:
            row[22] = None
        
        # Tax Year
        tax_year =  datetime.now().year

        if not (tax_year is None):
            row[23] = tax_year
        else:
            row[23] = ''
            
        # County Land Use Code
        county_luc = row[47]
        if not (county_luc is None):
            row[24] = str(county_luc)
        else:
            row[24] = '' 
            
        # Units
        units = row[48]
        if not (units is None):
            row[27] = units
        else:
            row[27] = None
        
#         Update the row.
        cursor.updateRow(row)
del cursor

#arcpy.management.CopyFeatures(carsonParcel, parcel_out)

out_coordinate_system = arcpy.SpatialReference('NAD 1983 UTM Zone 10N') 
arcpy.Project_management(carsonParcel, parcel_out, out_coordinate_system)

print('New Carson Parcels transformed')

New Carson Parcels transformed


### Douglas County

In [61]:
# get staging feature class and name output trnasformed feature class
in_features = "Parcel_DG_Extracted"
parcel_out  = "Parcel_DG_Transformed"

# in-memory feature class
douglasParcel = r"in_memory/inMemoryFeatureClass"

# copy feature class into in-memory feature class to work on
arcpy.management.CopyFeatures(in_features, douglasParcel)

# Add TRPA base fields
arcpy.management.AddFields(douglasParcel, baseFields)


# Do work.
with arcpy.da.UpdateCursor(douglasParcel, [
                                        ## TRPA base schema ##
                                        'APN_TRPA',                 #0
                                        'PPNO_TRPA',                #1
                                        'JURISDICTION_TRPA',        #2
                                         # parcel address   
                                        'HSE_NUMBR_TRPA',           #3
                                        'STR_DIR_TRPA',             #4
                                        'STR_NAME_TRPA',            #5
                                        'STR_SUFFIX_TRPA',          #6
                                        'UNIT_NUMBR_TRPA',          #7
                                        'APO_ADDRESS_TRPA',         #8
                                        'PSTL_TOWN_TRPA',           #9
                                        'PSTL_STATE_TRPA',          #10
                                        'PSTL_ZIP5_TRPA',           #11
                                        # owner fields
                                            # no own first and last for DG
                                        'OWN_FULL_TRPA',            #12
                                        'MAIL_ADD1_TRPA',           #13
                                        'MAIL_CITY_TRPA',           #14
                                        'MAIL_STATE_TRPA',          #15
                                        'MAIL_ZIP5_TRPA',           #16
                                        # value fields  
                                        'AS_LANDVALUE_TRPA',        #17
                                        'AS_IMPROVALUE_TRPA',       #18
                                        'AS_SUM_TRPA',              #19
                                        'TAX_LANDVALUE_TRPA',       #20 
                                        'TAX_IMPROVALUE_TRPA',      #21
                                        'TAX_SUM_TRPA',             #22
                                        'TAX_YEAR_TRPA',            #23
                                        # land use fields 
                                        'COUNTY_LANDUSE_CODE_TRPA', #24
                                        'COUNTY_LANDUSE_TRPA',      #25
                                        # Fields for building info
                                        "YEAR_BUILT_TRPA",          #26
                                        'UNITS_TRPA',               #27
                                        'BEDROOMS_TRPA',            #28
                                        'BATHROOMS_TRPA',           #29
                                        'BUILDING_SQFT_TRPA',       #30
                                        'VHR_TRPA',                 #31
                                        'HOA_TRPA',                 #32
                                        ###-------------------------###
                                        # County Fields to get data from
                                        'APN',   # apn,ppno         #33
                                        'PLOC_', # house number     #34
                                        'PLOCDR',# street dir       #35
                                        'PLOCNM',# street name      #36
                                        'PLOCTP',# street suffix    #37
                                        'PLOCU_',# unit number      #38
                                        'PANAME',# owner name       #39
                                        'PMADD1',# mailing addr1    #40
                                        'PMADD2',# mailing addr2    #41
                                        'PMCTST',# city,state       #42
                                        'PZIP',  # zip              #43
                                        'YYEAR', # tax year         #44
                                        'YLDUSE',# land use code    #45
                                        'YLANDV',# land value       #46
                                        'YIMPRV',# improved value   #47
                                        'YEXMP', # tax exempt value #48
                                        'YNETV', # tax net value    #49
                                        'PCONYR',# year built       #50
                                        'PBEDS', # bedrooms         #51
                                        'PBATHS',# bathrooms        #52

    ### These are missing from the new service
    #                                         'PBLDSF',# building sqft    #
    #                                         'STREETADDR', #full adr     #
    #                                         'P_DWEL',# units            #
    #                                         'VHR',   # vhr yes?         #
    #                                         'HOA',   # hoa name         #
]) as cursor:
    # loop through each record and transform the values
    for row in cursor:
        # APN field
        # Get County value
        apn = str(row[33])
        if not (apn is None or apn == ""):
            row[0] =(apn[:4] + "-" + apn[4:6] + "-" + apn[6:9] + "-" + apn[9:12])
        else:
            row[0] = ""
            
        #PPNO
        ppno = row[33]
        if not (ppno is None):
            row[1] = int(ppno)
        else:
            row[1] = ''
            
        # Jurisdiction
        row[2] = "DG"
        
        # APO Address
        house            = str(row[34]).strip()
        street_direction = str(row[35]).strip()
        street_name      = str(row[36]).strip()
        street_suffix    = str(row[37]).strip()
        unit             = str(row[38]).strip()
        if not (street_name is None or street_name=='' or street_name.isspace()==True):
            row[8] = re.sub(" +"," ", (house + " " + street_direction +" " + street_name+" " + street_suffix+" " + unit).strip())
        else:
            row[8] = ''
        
        # House Number
        house = row[34]
        if not (house is None):
            row[3] = str(house)
        else:
            row[3] = ''
        
        # Street Direction
        street_direction = row[35]
        if not (street_direction is None or street_direction=='' or street_direction.isspace()==True):
            row[4] = street_direction
        else:
            row[4] = ''
            
        # Street Name
        street_name = row[36]
        if not (street_name is None or street_name =='' or street_name.isspace()==True):
            row[5] = street_name
        else:
            row[5] = ''
            
        # Street Suffix
        street_suffix = row[37]
        if not (street_suffix is None or street_suffix =='' or street_suffix.isspace()==True):
            row[6] = street_suffix
        else:
            row[6] = ''
            
        # Unit Number
        unit= row[38]
        if not (unit is None or unit=='' or unit.isspace()==True):
            row[7] = unit
        else:
            row[7] = ''
                    
        # Postal Town - see Search/Update Cursor below
        
        # Postal State
        row[10] = 'NV'
        
        # Postal Zip - See Search/Update Cursor below
        row[11] = ''    
        
        # Owner Name
        owner = row[39]
        if not (owner is None or owner == '' or owner.isspace()==True):
            row[12] = owner.strip()
        else:
            row[12] = ""

        # Mailing Address
        address1 = row[40].strip()
        address2 = row[41].strip()
        if not (address1 is None or address1=='' or address1.isspace()==True):
            row[13] = str(address1 + " " + address2)
        elif (address2 is None):
            row[13] = address1
        else:
            row[13] = ''
                   
        # Mailing City
        mail_city = str(row[42]).split(',',1)[0].strip()
        
        if not (mail_city is None or mail_city=='' or mail_city.isspace()==True):
            row[14] = mail_city
        else:
            row[14] = ''
            
        # Mailing State - Added logic to set anything that isn't 2 characters long to '' 
        mail_state = str(row[42]).rsplit(',')[-1].strip().split(' ',1)[0].strip()
        if not (mail_state is None or mail_state=='' or mail_state.isspace()==True or len(mail_state)!=2):
            row[15] = mail_state
        else:
            row[15] = ''
        
        # Mailing Zipcode
        mail_zip = row[43].strip()
        if not (mail_zip is None or mail_zip=='' or mail_zip.isspace()==True):
            row[16] = mail_zip
        else:
            row[16] = ''
            
        # Assessed Land Value
        land_value = row[46]
        if not(land_value is None):
            row[17] = land_value
        else:
            row[17] = ''
        
        # Assessed Improved Value
        improved_value = row[47]
        if not (improved_value is None):
            row[18] = improved_value
        else:
            row[18] = None
                
        # Assessed Sum
        if not (land_value is None or improved_value is None):
            assessed_sum = improved_value + land_value
            row[19] = assessed_sum
        else:
            row[19] = None
        
        # Tax  Land Value
        taxland_value = row[46]
        if not(taxland_value is None):
            row[20] = taxland_value/0.35
        else:
            row[20] = None
        
        # Tax Improved Value
        taximproved_value = row[47]
        if not (taximproved_value is None):
            row[21] = taximproved_value/0.35
        else:
            row[21] = None
        
        # Tax Sum
        if not (land_value is None or improved_value is None):
            tax_sum = row[49]
            row[22] = tax_sum
        else:
            row[22] = None
        
        # Tax Year
        tax_year = row[44]
        if not (tax_year is None):
            row[23] = tax_year
        else:
            row[23] = ''
            
        # County Land Use Code
        county_luc = row[45]
        if not (county_luc is None):
            row[24] = str(county_luc)
        else:
            row[24] = '' 
        
        # Year Built
        year_built = row[50]
        if not (year_built is None or year_built==''):
            row[26] = year_built
        else:
            row[26] = None
        
        # Bedrooms
        bedrooms = row[51]
        if not (bedrooms is None or bedrooms==''):
            row[28] = bedrooms
        else:
            row[28] = None
        # Bathrooms
        baths = row[52]
        if not (baths is None or baths==''):
            row[29] = baths
        else:
            row[29] = None
            
#         Update the row.
        cursor.updateRow(row)
del cursor

out_coordinate_system = arcpy.SpatialReference('NAD 1983 UTM Zone 10N') 
arcpy.Project_management(douglasParcel, parcel_out, out_coordinate_system)

print('New Douglas Parcels transformed')

New Douglas Parcels transformed


### Eldorado County

In [40]:
# get staging feature class to transform
in_features = "Parcel_EL_Extracted"
parcel_out  = "Parcel_EL_Transformed"

# in-memory feature class
eldoradoParcel = r"in_memory/inMemoryFeatureClass"

# copy feature class into in-memory feature class to work on
arcpy.management.CopyFeatures(in_features, eldoradoParcel)

# Add TRPA base fields
arcpy.management.AddFields(eldoradoParcel, baseFields)

# Set up the regex queries for the data.
# cityStateZipRegex = r'(.+?)\s([A-Z]{1,2})\s(.+?)$' - Keep in case new one doesn't work out long term.
cityStateZipRegex = r'(.+?)\s([A-Z]{1,2})\s(?=\d)(.*)'
poBoxRegex = r'([^x]+)\W(P\s*O BOX\W*[0-9]{1,6})'
addressRegex = r'(\d{1,5}\D+.+)'
canadaRegex = r'(.+?)\s([A-Z]{1,2})\s(CANADA)\s(.*)'
brazilRegex = r'(.+?)\s(BRAZIL)\s(.*)'

# Set up list for addresses with a country name in the mail_addr4 column.
countriesList = ['japan','canada']

# Transform County data to TRPA Schema
with arcpy.da.UpdateCursor(eldoradoParcel, [
                                        ## TRPA base schema ##
                                        'APN_TRPA',                 #0
                                        'PPNO_TRPA',                #1
                                        'JURISDICTION_TRPA',        #2
                                         # parcel address   
                                        'HSE_NUMBR_TRPA',           #3
                                        'STR_DIR_TRPA',             #4
                                        'STR_NAME_TRPA',            #5
                                        'STR_SUFFIX_TRPA',          #6
                                        'UNIT_NUMBR_TRPA',          #7
                                        'APO_ADDRESS_TRPA',         #8
                                        'PSTL_TOWN_TRPA',           #9
                                        'PSTL_STATE_TRPA',          #10
                                        'PSTL_ZIP5_TRPA',           #11
                                        # owner fields
                                            # no first and last fields
                                        'OWN_FULL_TRPA',            #12
                                        'MAIL_ADD1_TRPA',           #13
                                        'MAIL_CITY_TRPA',           #14
                                        'MAIL_STATE_TRPA',          #15
                                        'MAIL_ZIP5_TRPA',           #16
                                        # value fields  
                                        'AS_LANDVALUE_TRPA',        #17
                                        'AS_IMPROVALUE_TRPA',       #18
                                        'AS_SUM_TRPA',              #19
                                        'TAX_LANDVALUE_TRPA',       #20 
                                        'TAX_IMPROVALUE_TRPA',      #21
                                        'TAX_SUM_TRPA',             #22
                                        'TAX_YEAR_TRPA',            #23
                                        # land use fields 
                                        'COUNTY_LANDUSE_CODE_TRPA', #24
                                        'COUNTY_LANDUSE_TRPA',      #25
                                        # Fields for building info
                                        "YEAR_BUILT_TRPA",          #26
                                        'UNITS_TRPA',               #27
                                        'BEDROOMS_TRPA',            #28
                                        'BATHROOMS_TRPA',           #29
                                        'BUILDING_SQFT_TRPA',       #30
                                        'VHR_TRPA',                 #31
                                        'HOA_TRPA',                 #32
                                        ###-------------------------###
                                        # County Fields to get data from
                                        'PRCL_ID',                  #33
                                        'OWNER_NAME',               #34
                                        'MAIL_ADDR1',               #35
                                        'MAIL_ADDR2',               #36
                                        'MAIL_ADDR3',               #37
                                        'MAIL_ADDR4',               #38
                                        'ADDRSTNBR',                #39
                                        'ADDRSTDIR',                #40
                                        'ADDRSTNAME',               #41
                                        'ADDRSTTYPE',               #42
                                        'ADDRUNITNB',               #43
                                        'PRCL_ADDR',                #44
                                        'USECD_1',                  #45
                                        'USECDLIT_1',               #46
                                        'STRUCT_VAL',               #47
                                        'LAND_VAL',                 #48
                                        'YR_BUILT',                 #49
                                        'DWELLUNITS',               #50
                                        'BEDROOMS',                 #51
                                        'ADDRSTPRFX'                 #52
]) as cursor:
    # transform each row
    for row in cursor:   
        # Set APN
        apn = row[33]
        if not (apn is None or apn == "" or apn.isspace() == True or 'UN' in apn):
            row[0] = (apn[:3] + "-" + apn[3:6] + "-" + apn[6:9])
        else:
            row[0] = ''
            
        # Set PPNO
        ppno = row[33]
        if not (apn is None or apn == "" or apn.isspace() == True or 'UN' in apn or 'NP' in apn):
            try:
                row[1] = float(ppno)
            except ValueError:
                row[1] = 0
        else:
            row[1] = 0
        # Set County
        row[2] = 'EL'
        
        # APO Address
        full_address = row[44]
        if not (full_address is None or full_address=='' or full_address.isspace()==True):
            
            row[8] = full_address
        else:
            row[8] = ''
        
        # House Number
        house = row[39]
        if not (house is None):
            # convert house number to integer type
            row[3] = str(int(house))
        else:
            row[3] = ''
        
        # Street Direction
        street_direction = row[40]
        if not (street_direction is None or street_direction=='' or street_direction.isspace()==True
                or street_direction == 'UNASSIGNED'):
            # get the first character
            row[4] = street_direction[0]
        else:
            row[4] = ''
            
        # Street Name
        street_name = row[41]
        street_prefix = row[52]
        if not (street_name is None or street_name =='' or street_name.isspace()==True):
            if not (street_prefix is None or street_prefix =='' or street_prefix.isspace()==True
                   or street_prefix == 'UNASSIGNED'):
                row[5]= street_prefix + ' ' + street_name
            else:
                row[5] = street_name
        else:
            row[5] = ''
            
        # Street Suffix
        street_suffix = row[42]
        if not (street_suffix is None or street_suffix =='' or street_suffix.isspace()==True               
                or street_direction == 'UNASSIGNED'):
            row[6] = street_suffix
        else:
            row[6] = ''
            
        # Unit Number
        unit= row[43]
        if not (unit is None or unit=='' or unit.isspace()==True):
            row[7] = ("#" + str(unit))
        else:
            row[7] = ''
                    
        # Postal Town - see Search/Update Cursor below
        row[9] = ''
        
        # Postal State
        row[10] = 'CA'
        
        # Postal Zip - See Search/Update Cursor below
        row[11] = ''    
        
        # Set Mailing Owner, Address, City, State, Zip
        if row[38] != ' ':
#             print("Working on MAIL_ADDR4")
            if row[38] != 'UNKNOWN' and row[38].lower() not in countriesList:
                # Parse out city, state, and zip code and assign variables.
                cityStateZip = re.search(cityStateZipRegex, str(row[38]))
                if cityStateZip is not None:
                    city = cityStateZip.group(1)
                    state = cityStateZip.group(2)
                    zipCode = cityStateZip.group(3)
                    country = ''
                else:
                    continue
                # Check to see if address starts with PO Box and assign variable.
                if str(row[37]).startswith('PO') or str(row[37]).startswith('P O'):
                    address = str(row[37])
                elif "PO BOX" in str(row[37]) or "P O BOX" in str(row[37]) or "P.O. BOX" in str(row[37]):
                    address = str(row[37])

                # Parse out address that doesn't have PO Box and assign variable.
                else:
                    add = re.search(addressRegex,str(row[37]))
                    address = add.group(1)

                # Assign owner variable.
                owner = str(row[34])+' '+str(row[35])+' '+str(row[36])
            elif row[38].lower() in countriesList:
                country = str(row[38])
                state = str(row[37])
                city = str(row[36])
                address = str(row[35])
                owner = str(row[34])
                zipCode = ''
            elif row[38] == "CANADA": # temporary patch for incorrectly entered Canadian address
                canadaZip = re.search(r'[ABCEGHJKLMNPRSTVXY][0-9][ABCEGHJKLMNPRSTVWXYZ] ?[0-9][ABCEGHJKLMNPRSTVWXYZ][0-9]', str(row[3]))
                canadaProvZip = re.search(r'(.*?)\s(N[BLSTU]|[AMN]B|[BQ]C|ON|PE|SK)',str(row[36]))
                if canadaZip != None:
                    zipCode = str(canadaZip.group(0))
                else:
                    zipCode =''
                    address = str(row[35])
                    city = str(canadaProvZip.group(1))
                    state = str(canadaProvZip.group(2))
                    country = str(row[38])
            else:
                owner = str(row[34])
                address = ''
                city = ''
                state = ''
                zipCode = ''
                country = ''

        # If mail_addr4 is "empty".
        elif row[37] != ' ':
#             print("Working on MAIL_ADDR3")
            # Parse out city, state, and zip code.
            cityStateZip = re.search(cityStateZipRegex, str(row[37]))

            # Foreign addresses won't parse so assign country, owner, address, and city variables. Set state and zip to blanks.
            if cityStateZip is None:
                country = str(row[37])
                owner = str(row[34])
                address = str(row[35])
                city = str(row[36])
                state = ''
                zipCode = ''
            else:
                country = ''
                row2 = str(row[2])

                # Sanitize rows that start with a space.
                if str(row[36]).startswith(' '):
                    row2 = str(row[36])[1:]

                # Parse out city, state, and zip code and assign variables.
                city = cityStateZip.group(1)
                state = cityStateZip.group(2)
                zipCode = cityStateZip.group(3)

                # Check to see if address starts with PO Box and assign variable.
                if row2.startswith('PO') or row2.startswith('P O') or row2.startswith('P.O.'):
                    address = row2

                # Sometimes there may be a word in front of PO Box and parse that out and assign variable.
                elif "PO BOX" in row2 or "P O BOX" in row2 or row2.startswith('ONE ') or row2.startswith('TWO '):
                    address = row2
                else:
                    # Parse out address that doesn't have PO Box and assign variable, sometimes there no address so set variable to None.
                    add = re.search(addressRegex,row2)
                    if add is None:
                        address = 'None'
                    else:
                        address = add.group(1)

                # Assign owner variable.
                owner = str(row[34])+' '+str(row[35])

        # Before moving to mail_addr2 must capture "blanks" and USA owned parcels and insert blanks.
        elif row[0] == 'UNITED STATES OF AMERICA':
            cityStateZip = re.search(cityStateZipRegex, str(row[36]))
            owner = str(row[34])
            address = str(row[35])
            if cityStateZip is None:
                city = ''
                state = ''
                zipCode = ''
            else:
                city = cityStateZip.group(1)
                state = cityStateZip.group(2)
                zipCode = cityStateZip.group(3)
            country = ''
        elif row[34] == ' ':
            owner = ''
            address = ''
            city = ''
            state = ''
            zipCode = ''
            country = ''
        elif row[35] == ' ':
            owner = str(row[34])
            address = ''
            city = ''
            state = ''
            zipCode = ''
            country = ''

        # Parse the rest of the address info.
        else:
#             print("Working on MAIL_ADDR2")
            if str(row[36]) == ' ':
                owner = str(row[34])
                address = str(row[35])
                city = ''
                state = ''
                zipCode = ''
                country = ''
            else:
                row2 = str(row[36])

                # Parse out city, state, and zip code and assign variables.
                cityStateZip = re.search(cityStateZipRegex, row2)

                # if it can't parse it's a foreign address and assign country variable.
                if cityStateZip is None:
                    if "CANADA" in row2:
                        cityStateZip = re.search(canadaRegex, row2)
                        city = cityStateZip.group(1)
                        state = cityStateZip.group(2)
                        zipCode = cityStateZip.group(4)
                        country = cityStateZip.group(3)
                    if "BRAZIL" in row2:
                        cityStateZip = re.search(brazilRegex, row2)
                        city = cityStateZip.group(1)
                        state = ''
                        zipCode = cityStateZip.group(3)
                        country = cityStateZip.group(2)
                else:
                    row1 = str(row[35])
                    country = ''
                    city = cityStateZip.group(1)
                    state = cityStateZip.group(2)
                    zipCode = cityStateZip.group(3)

                    # Sanitize rows that start with a space.
                    if row1.startswith(' '):
                        row1 = row1[1:]

                    # Check to see if address starts with PO Box and assign variable.
                    if row1.startswith('PO') or row1.startswith('P.O.') or row1.startswith('P O') or row1.startswith('P  O'):
                        address = str(row[35])

                    # Sometimes there may be a word in front of PO Box and parse that out and assign variable.
                    elif "PO BOX" in row1 or "P O BOX" in row1:
                        poBox = re.search(poBoxRegex,row1)

                        # If it can't be parsed assign variable.
                        if poBox is None:
                            address = row1
                        else:
                            address = poBox.group(2)
                    else:
                        # Parse out address that doesn't have PO Box and assign variable, sometimes there no address so set variable to None.
                        add = re.search(addressRegex,row1)

                        # Have exception for addresses that spell out 'one' instead of '1'.
                        if add is None or row1.startswith('ONE'):
                            address = row1
                        else:
                            address = add.group(1)

                # Set owner variable.
                owner = str(row[34])

        # Set Owner
        row[12] = owner
        
        # Set Mailing Address
        row[13] = address
        
        # Set Mailing City
        row[14] = city
        
        # Set Mailing State
        row[15] = state
        
        # Set Mailing ZIP
        row[16] = zipCode
#         row[10] = country

        # Assessed Land Value
        land_value = row[48]
        if not(land_value is None):
            row[17] = land_value
        else:
            row[17] = ''
        
        # Assessed Improved Value
        improved_value = row[47]
        if not (improved_value is None):
            row[18] = improved_value
        else:
            row[18] = None
                
        # Assessed Sum
        if not (land_value is None or improved_value is None):
            assessed_sum = improved_value + land_value
            row[19] = assessed_sum
        else:
            row[19] = None
        
        # Tax  Land Value
        taxland_value = row[48]
        if not(taxland_value is None):
            row[20] = taxland_value
        else:
            row[20] = None
        
        # Tax Improved Value
        taximproved_value = row[47]
        if not (taximproved_value is None):
            row[21] = taximproved_value
        else:
            row[21] = None
        
        # Tax Sum
        if not (land_value is None or improved_value is None):
            tax_sum = taximproved_value + taxland_value
            row[22] = tax_sum
        else:
            row[22] = None
        
        # Tax Year
        row[23] = datetime.now().year # get current year
            
        # County Land Use Code
        county_luc = row[45]
        if not (county_luc is None):
            row[24] = str(county_luc)
        else:
            row[24] = '' 
        
        # County Land Use - See Search/Update Cursor Below
        county_landuse = row[46]
        if not (county_landuse is None or county_landuse=='' or county_landuse.isspace()==True):
            row[25] = county_landuse
        else:
            row[25] = '' 
        
        # Year Built
        year_built = row[49]
        if not (year_built is None):
            row[26] = year_built
        else:
            row[26] = None
            
        # Units
        units = row[50]
        if not (units is None):
            row[27] = units
        else:
            row[27] = None
        
        # Bedrooms
        bedrooms = row[51]
        if not (bedrooms is None):
            row[28] = bedrooms
        else:
            row[28] = None

        # Update the row.
        cursor.updateRow(row)
del cursor

out_coordinate_system = arcpy.SpatialReference('NAD 1983 UTM Zone 10N') 



CombineAPNs(eldoradoParcel, 'APN_TRPA')


arcpy.Project_management(eldoradoParcel, parcel_out, out_coordinate_system)



print('New El Dorado Parcels transformed')

Started combining APNs: 2023-05-04 15:56:34
{'', '029-630-021', '029-630-008', '029-630-017', '029-630-015', '029-630-001', '029-630-004', '029-670-001', '029-630-019', '029-630-023', '029-630-020', '029-630-009', '029-630-022', '027-010-016', '029-630-010', '920-000-740', '022-333-002', '029-630-005', '029-630-007', '029-630-011', '920-000-405', '022-312-017', '029-630-002', '029-630-026', '029-630-006', '029-630-024', '029-630-014', '029-630-016', '029-630-027', '029-630-003', '029-630-013', '029-630-028', '029-670-002', '920-000-739', '029-630-018', '029-630-025', '029-630-012'}
029-630-021
029-630-008
029-630-017
029-630-015
029-630-001
029-630-004
029-670-001
029-630-019
029-630-023
029-630-020
029-630-009
029-630-022
027-010-016
029-630-010
920-000-740
022-333-002
029-630-005
029-630-007
029-630-011
920-000-405
022-312-017
029-630-002
029-630-026
029-630-006
029-630-024
029-630-014
029-630-016
029-630-027
029-630-003
029-630-013
029-630-028
029-670-002
920-000-739
029-630-018
029

### Placer County

In [32]:
in_features = "Parcel_PL_Extracted"
parcel_out  = "Parcel_PL_Transformed"

# in-memory feature class
placerParcel = r"in_memory/inMemoryFeatureClass"

# copy feature class into in-memory feature class to work on
arcpy.management.CopyFeatures(in_features, placerParcel)

# Add TRPA base fields
arcpy.management.AddFields(placerParcel, baseFields)

# Transform County data to TRPA data.
with arcpy.da.UpdateCursor(placerParcel, ['APN_TRPA',               #0
                                        'PPNO_TRPA',                #1
                                        'JURISDICTION_TRPA',        #2
                                         # parcel address   
                                        'HSE_NUMBR_TRPA',           #3
                                        'STR_DIR_TRPA',             #4
                                        'STR_NAME_TRPA',            #5
                                        'STR_SUFFIX_TRPA',          #6
                                        'UNIT_NUMBR_TRPA',          #7
                                        'APO_ADDRESS_TRPA',         #8
                                        'PSTL_TOWN_TRPA',           #9
                                        'PSTL_STATE_TRPA',          #10
                                        'PSTL_ZIP5_TRPA',           #11
                                        # owner fields
                                        'OWN_FIRST_TRPA',           #12
                                        'OWN_LAST_TRPA',            #13
                                        'OWN_FULL_TRPA',            #14
                                        'MAIL_ADD1_TRPA',           #15
                                        'MAIL_CITY_TRPA',           #16
                                        'MAIL_STATE_TRPA',          #17
                                        'MAIL_ZIP5_TRPA',           #18
                                        # value fields  
                                        'AS_LANDVALUE_TRPA',        #19
                                        'AS_IMPROVALUE_TRPA',       #20
                                        'AS_SUM_TRPA',              #21
                                        'TAX_LANDVALUE_TRPA',       #22 
                                        'TAX_IMPROVALUE_TRPA',      #23
                                        'TAX_SUM_TRPA',             #24
                                        'TAX_YEAR_TRPA',            #25
                                        # land use fields 
                                        'COUNTY_LANDUSE_CODE_TRPA', #26
                                        'COUNTY_LANDUSE_TRPA',      #27
                                        # Fields for building info
                                        "YEAR_BUILT_TRPA",          #28
                                        'UNITS_TRPA',               #29
                                        'BEDROOMS_TRPA',            #30
                                        'BATHROOMS_TRPA',           #31
                                        'BUILDING_SQFT_TRPA',       #32
                                        'VHR_TRPA',                 #33
                                        'HOA_TRPA',                 #34
                                        ###-------------------------###
                                        # County Fields to get data from
                                        'APN',   # apn                #35
                                        'GISAPN',   # ppno            #36
                                        'STREETNUM', # house number   #37
                                        'STREETDIR',# street dir      #38
                                        'STREETNAME',# street name    #39
                                        'STREETTYPE',# street suffix  #40
                                        'SP_APT',  # unit number      #41
                                        'OWNER1',# owner name         #42
                                        'OWNER2',# owner 2            #43
                                        'ADR1',  # mailing addr1      #44
                                        'ADR2',  # mailing addr2      #45
                                        'CITY',  # city               #46 
                                        'STATE', # state              #47
                                        'ZIP',  # zip                 #48
                                        'USE_CD', # land use code     #49
                                        'USE_CD_N', # land use desc   #50
                                        'LANDVALUE',# land value      #51
                                        'STRUCTURE',# improved value  #52
                                        'EFFECTIVEY',# year built     #53
                                        'STR_SQFT'  # build sqft      #54
                                    
]) as cursor:   
    # loop through each record to transform values to TRPA schema values
    for row in cursor:
        # set APN
        apn = row[36]
        if not (apn is None or apn == "" or apn.isspace() == True or "ROW" in apn or len(apn) < 8):
            row[0] =(apn[:3] + "-" + apn[3:6] + "-" + apn[6:9])
        else:
            row[0] = ""
            
        # set PPNO
        ppno = row[36]
        if not (ppno is None or ppno == "" or "ROW" in ppno or len(ppno) < 8):
            row[1] = ppno
        else:
            row[1] = 0
            
        # Jurisdiction
        row[2] = "PL"
        
        # House Number
        house = row[37]
        if not (house is None or house=='' or house.isspace()==True):
            row[3] = house
        else:
            row[3] = ''
        
        # Street Direction
        street_direction = row[38]
        if not (street_direction is None or street_direction=='' or street_direction.isspace()==True):
            row[4] = street_direction
        else:
            row[4] = ''
            
        # Street Name
        street_name = row[39]
        if not (street_name is None or street_name =='' or street_name.isspace()==True):
            row[5] = street_name
        else:
            row[5] = ''
            
        # Street Suffix
        street_suffix = row[40]
        if not (street_suffix is None or street_suffix =='' or street_suffix.isspace()==True):
            row[6] = street_suffix
        else:
            row[6] = ''
            
        # Unit Number
        unit= row[41]
        if not (unit is None or unit=='' or unit.isspace()==True):
            row[7] = str(unit)
        else:
            row[7] = ''
        
        # APO Address
        full_address = [house, street_direction, street_name, street_suffix, unit]
        adr = str(' '.join(filter(None, full_address))).strip()
        
        if not (adr is None or adr=='' or adr.isspace()==True):
            row[8] = adr
        else:
            row[8] = ''
            
        # Postal Town - See TRPA ATTRIBUTION section
            
        # Postal State
        row[10] = 'CA'

        # Postal City - See TRPA ATTRIBUTION section
        
        # Owner Name
        owner1 = row[42]
        owner2 = row[43]
        # own first
        if not (owner1 is None or owner1 == "" or owner1.isspace() == True):
            row[12] = owner1.strip()
        else:
            row[12] = ''
        # own last
        if not (owner2 is None or owner2 == "" or owner2.isspace() == True):
            row[13] = owner2.strip()
        else:
            row[13] = ''    
        # own full
        if not (owner2 is None or owner2 == "" or owner2.isspace() == True):
            row[14] = (owner1+" " + owner2).strip()
        elif not (owner1 is None or owner1 == ""):
            row[14] = owner1.strip()
        else:
            row[14] = ''
            
        # Mailing Address
        address1 = row[44]
        address2 = row[45]
        if not (address1 is None or address1=='' or address1.isspace()==True):
            row[15] = str(address1).strip()
        else:
            row[15] = ''
                  
        # Mailing City
        mail_city = row[46]
        
        if not (mail_city is None or mail_city=='' or mail_city.isspace()==True):
            row[16] = mail_city
        else:
            row[16] = ''
            
        # Mailing State
        mail_state = row[47]
        if not (mail_state is None or mail_state=='' or mail_state.isspace()==True):
            row[17] = mail_state
        else:
            row[17] = ''
        
        # Mailing Zipcode
        mail_zip = row[48]
        if not (mail_zip is None or mail_zip=='' or mail_zip.isspace()==True):
            row[18] = mail_zip[:5]
        else:
            row[18] = ''
            
        # Assessed Land Value
        land_value = row[51]
        if not(land_value is None or land_value==''):
            row[19] = int(land_value)
        else:
            row[19] = None
       
        # Assessed Improved Value    
        improved_value = row[52]
        if not (improved_value is None or improved_value==''):
            row[20] = int(improved_value)
        else:
            row[20] = None

        # Assessed Sum
        if not (row[19] is None and row[20] is None):
            assessed_sum = improved_value + land_value
            row[21] = assessed_sum
        else:
            row[21] = None
        
        # Tax Land Value
        taxland_value = row[51]
        if not(taxland_value is None):
            row[22] = int(taxland_value)
        else:
            row[22] = None
        
        # Tax Improved Value
        taximproved_value = row[52]
        if not (taximproved_value is None):
            row[23] = int(taximproved_value)
        else:
            row[23] = None
        
        # Tax Sum
        if not (row[22] is None and row[23] is None):
            tax_sum = taximproved_value + taxland_value
            row[24] = tax_sum
        else:
            row[24] = None
        
        # Tax Year
        row[25] = datetime.now().year # get current year
            
        # County Land Use Code
        county_luc = row[49]
        if not (county_luc is None or county_luc=='' or county_luc.isspace()==True):
            row[26] = county_luc
        else:
            row[26] = '' 
        
        # County Land Use
        county_landuse = row[50]
        if not (county_landuse is None or county_landuse=='' or county_landuse.isspace()==True):
            row[27] = county_landuse
        else:
            row[27] = ''
            
        # Year Built
        year_built = row[53]
        if not (year_built is None):
            row[28] = year_built
        else:
            row[28] = None
            
        # Building SQFT
        bldsqft = row[54]
        if not (bldsqft is None):
            row[32] = bldsqft
        else:
            row[32] = None
             
        # Update the row.
        cursor.updateRow(row)
del cursor

# combine duplicate APNs 
### some shoreline parcels are split by the highway and have two features for the same APN
CombineAPNs(placerParcel, 'APN_TRPA')

# project to our projected coordinate system
out_coordinate_system = arcpy.SpatialReference('NAD 1983 UTM Zone 10N') 
arcpy.Project_management(placerParcel, parcel_out, out_coordinate_system)

# done with the transormations for Placer
print('New Placer Parcels transformed')

Started combining APNs: 2023-05-09 20:23:02
{'', '084-192-001', '090-041-014', '096-440-014', '096-440-028', '094-070-016', '094-121-001', '090-233-036', '096-540-014', '083-062-008', '096-440-019', '083-074-019', '094-540-023', '117-010-012', '112-180-039', '090-133-003', '091-070-001', '098-165-007', '085-040-024', '110-051-031', '085-260-039', '090-324-004', '096-440-030', '095-380-025', '090-041-020', '090-074-026', '117-110-038', '085-343-026', '096-221-029', '091-090-001', '092-234-024', '091-050-004', '080-221-006', '096-380-022', '090-052-026', '095-110-003', '085-215-006', '083-162-036', '085-040-039', '094-010-007', '094-180-022', '096-440-011', '096-103-037', '090-064-037', '098-167-019', '085-083-016', '085-050-032', '090-064-001', '117-020-022', '090-041-037', '096-380-021', '091-070-002', '094-110-018', '096-440-013', '091-080-002', '090-055-037', '085-040-027', '117-020-021', '091-050-008', '094-190-004', '083-062-045', '095-170-001', '096-380-019', '095-190-005', '090-1

### Washoe County

In [33]:
# input/output
in_features = "Parcel_WA_Extracted"
parcel_out  = "Parcel_WA_Transformed"

# in-memory feature class
washoeParcels = r"in_memory/inMemoryFeatureClass"

# copy features to in-memory feature class
arcpy.CopyFeatures_management(in_features, washoeParcels)

# Add TRPA base fields
arcpy.management.AddFields(washoeParcels,baseFields)

# Tansform County Data to TRPA Data.
with arcpy.da.UpdateCursor(washoeParcels, ['APN_TRPA',              #row[0]
                                        'PPNO_TRPA',                #row[1]
                                        'JURISDICTION_TRPA',        #row[2]
                                         # parcel address   
                                        'HSE_NUMBR_TRPA',           #3
                                        'STR_DIR_TRPA',             #4
                                        'STR_NAME_TRPA',            #5
                                        'STR_SUFFIX_TRPA',          #6
                                        'UNIT_NUMBR_TRPA',          #7
                                        'APO_ADDRESS_TRPA',         #8
                                        'PSTL_TOWN_TRPA',           #9
                                        'PSTL_STATE_TRPA',          #10
                                        'PSTL_ZIP5_TRPA',           #11
                                        # owner fields
                                        'OWN_FIRST_TRPA',           #12
                                        'OWN_LAST_TRPA',            #13
                                        'OWN_FULL_TRPA',            #14
                                        'MAIL_ADD1_TRPA',           #15
                                        'MAIL_CITY_TRPA',           #16
                                        'MAIL_STATE_TRPA',          #17
                                        'MAIL_ZIP5_TRPA',           #18
                                        # value fields  
                                        'AS_LANDVALUE_TRPA',        #19
                                        'AS_IMPROVALUE_TRPA',       #20
                                        'AS_SUM_TRPA',              #21
                                        'TAX_LANDVALUE_TRPA',       #22 
                                        'TAX_IMPROVALUE_TRPA',      #23
                                        'TAX_SUM_TRPA',             #24
                                        'TAX_YEAR_TRPA',            #25
                                        # land use fields 
                                        'COUNTY_LANDUSE_CODE_TRPA', #26
                                        'COUNTY_LANDUSE_TRPA',      #27
                                        # Fields for building info
                                        "YEAR_BUILT_TRPA",          #28
                                        'UNITS_TRPA',               #29
                                        'BEDROOMS_TRPA',            #30
                                        'BATHROOMS_TRPA',           #31
                                        'BUILDING_SQFT_TRPA',       #32
                                        'VHR_TRPA',                 #33
                                        'HOA_TRPA',                 #34
                                        ###-------------------------###
                                        # County Fields to get data from
                                        'PIN',   # apn              #35
                                        'APN',   # ppno             #36
                                        'FullAddress',#full adrress #37
                                        'STREETNUM', # house number #38
                                        'STREETDIR',# street dir    #39
                                        'STREET',# street name      #40
                                        'CITY',    # postal town    #41
                                        'SITUSZIP', # postal zip    #42
                                        'SQFEET',# building sqft    #43
                                        'FIRSTNAME',# first name    #44
                                        'LASTNAME', # last name     #45
                                        'MAILING1',# mailing addr1  #46
                                        'MAILING2',# mailing addr2  #47
                                        'MAILCITY',# city           #48
                                        'MAILSTATE', # mailing state#49
                                        'MAILZIP',  # zip           #50
                                        'TAXYEAR', # tax year       #51
                                        'LAND_USE',# land use code  #52
                                        'LANDASS',# land value      #53
                                        'BUILDASS',# improved value #54
                                        'TOTALASS', # total assesed #55
                                        'LANDAPR',  # land apr      #56
                                        'BUILDAPR', # building apr  #57
                                        'TOTALAPR', # total apr     #58
                                        'YEARBLT',# year built      #59
                                        'STORIES',# stories         #60
                                        'BEDROOMS', # bedrooms      #61      
                                        'BATHS',# bathrooms         #62
                                        'UNITS'   # units           #63
]) as cursor:
    # loop through each record and transform the values
    for row in cursor:
        # APN field
        # Get County value
        apn  = row[35]
        if not (apn is None or apn == "" or apn.isspace() == True):
            # set TRPA value
            row[0] = apn
        else:
            row[0] = ''
            
        #PPNO
        ppno = row[36]
        if not (ppno is None or ppno == ""):
            row[1] = int(ppno)
        else:
            row[1] = None
            
        # Jurisdiction
        row[2] = "WA"
                
        # APO Address
        fulladdress = row[37]
        if not (fulladdress is None or fulladdress=='' or fulladdress.isspace()==True):
            row[8] = fulladdress
        else:
            row[8] = ''
        
        # House Number
        house = row[38]
        if not (house is None or house=='' or house.isspace()==True):
            row[3] = house
        else:
            row[3] = ''
            
           
        # Unit Number
        if not (fulladdress is None or fulladdress == ""):
            if fulladdress.strip()[-1].isdigit():
                if not ('STATE ROUTE 28' in fulladdress):
                    row[7] = (fulladdress.rsplit(' ')[-1].strip())
                else:
                    if not (fulladdress.strip().rsplit(' ')[-1] == '28'):
                        row[7] = (fulladdress.rsplit(' ')[-1].strip())
                    else:
                        if not ('STATE ROUTE 28 28' in fulladdress): 
                            row[7] = ""
                        else:
                            row[7] = (fulladdress.rsplit(' ')[-1].strip())
            else:
                if not ('US HIGHWAY 395' in fulladdress):
                    if len(fulladdress.rsplit(' ')[-1]) == 1:
                        row[7] = (fulladdress.rsplit(' ')[-1].strip())
                    elif not (len(fulladdress.rsplit(' ')[-1]) == 1):
                        if fulladdress[-2].isdigit():
                            row[7] = (fulladdress.rsplit(' ')[-1].strip())
                        else:
                            row[7] = ""
                    else:
                        row[7] = ""
                else:
                    row[7] = ""
        else:
            row[7] = ""
            
        # Street Direction
        stdir = row[39]
        if not (stdir is None):
            row[4] = (stdir.strip())
        else:
            row[4] = ""
            
        # Street Name    
        stname = row[40]
        if not stname in ('CROSS BOW', 'ENTERPRISE', 'STATE ROUTE 28', 'UNSPECIFIED', 'US HIGHWAY 395', ''):
            if stname[:2] in ('N ', 'S ', 'E ', 'W '):
                row[5] = stname.rsplit(' ',1)[0].strip().split(' ',1)[1].strip()
            elif not (stname is None or stname == "" or stname.isspace() == True):
                row[5] = (stname.rsplit(' ',1)[0].strip())
            #Currently the only example of this is two blanks in Incline Village with no info
            elif stname is None or stname == "" or stname.isspace() == True:
                if fulladdress[0].isdigit():
                    row[5] = (fulladdress.rsplit(' ')[-1].strip())
                else:
                    row[5] = ""
            else:
                logging.info("Error parsing washoe street name")
        elif stname in ('CROSS BOW', 'ENTERPRISE', 'STATE ROUTE 28', 'UNSPECIFIED', ''):
                row[5] = (stname.strip())
        else:
            row[5] = ""
            
        # Street Suffix
        if not stname in ('CROSS BOW', 'ENTERPRISE', 'STATE ROUTE 28', 'UNSPECIFIED', 'US HIGHWAY 395', ''):
            if not (stname is None or stname == "" or stname.isspace() == True):
                row[6] = (stname.rsplit(' ')[-1].strip())
            elif stname is None or stname == "" or stname.isspace() == True:
                if fulladdress[0].isdigit():
                    row[6] = (fulladdress.rsplit(' ')[-1].strip())
                else:
                    row[6] = ""
            else:
                logging.info("Error parsing washoe street suffix")
        else:
            row[11] = ""

        # Postal Town
        postal_town = row[41]
        if not (postal_town is None or postal_town == '' or postal_town.isspace()==True):
            row[9] = postal_town
        else:
            row[9] = ''
            
        # Postal State
        row[10] = 'NV'
        
        # Postal Zip
        postal_zip = row[42]
        if not (postal_zip is None or postal_zip == '' or postal_zip.isspace()==True):
            row[11] = postal_zip
        else:
            row[11] = ''
            
        # Owner Name
        # set owner first name
        ownfirst = row[44]
        if not (ownfirst is None or ownfirst.isspace() == True):
            row[12] = ownfirst
        else:
            row[12] = ""
        
        # own last
        ownlast = row[45]
        if not (ownlast is None or ownlast.isspace() == True):
            row[13] = ownlast
        else:
            row[13] = ""
        
        # own full
        if not (ownfirst is None and ownlast is None):
            row[14] = (ownfirst + " " + ownlast).strip()
        else:
            row[14] = ""
            
        # Mailing Address  
        address1 = row[46].strip()
        address2 = row[47].strip()
        if not (address1 is None or address1=='' or address1.isspace()==True):
            row[15] = str((address1 + " " + address2).strip())
        elif (address2 is None):
            row[15] = address1
        else:
            row[15] = ''
           
        # Mailing City
        mail_city = row[48]
        if not (mail_city is None or mail_city=='' or mail_city.isspace()==True):
            row[16] = mail_city
        else:
            row[16] = ''
            
        # Mailing State
        mail_state = row[49]
        if not (mail_state is None or mail_state=='' or mail_state.isspace()==True):
            row[17] = mail_state
        else:
            row[17] = ''
        
        # Mailing Zipcode
        mail_zip = row[50].strip()
        if not (mail_zip is None or mail_zip=='' or mail_zip.isspace()==True):
            row[18] = mail_zip[:5]
        else:
            row[18] = ''
            
        # Assessement Value
        land_value = row[53]
        if not(land_value is None or land_value==''):
            row[19] = land_value
        else:
            row[19] = None
        
        improved_value = row[54]
        if not (improved_value is None or improved_value==''):
            row[20] = improved_value
        else:
            row[20] = None
                
        assessed_sum = row[55]
        if not (assessed_sum is None or assessed_sum==''):
            row[21] = assessed_sum
        else:
            row[21] = None
        
        # Tax Value
        taxland_value = row[56]
        if not(taxland_value is None or taxland_value==''):
            row[22] = taxland_value
        else:
            row[22] = None
        
        taximproved_value = row[57]
        if not (taximproved_value is None or taximproved_value==''):
            row[23] = taximproved_value
        else:
            row[23] = None
        
        tax_sum = row[58]
        if not (tax_sum is None or tax_sum==''):
            row[24] = tax_sum
        else:
            row[24] = None
        
        # Tax Year
        tax_year = row[51]
        if not (tax_year is None or tax_year=='' or tax_year.isspace()==True):
            row[25] = tax_year
        else:
            row[25] = None
            
        # County Land Use Code
        county_luc = row[52]
        if not (county_luc is None or county_luc=='' or county_luc.isspace()==True):
            row[26] = int(county_luc.split(",",1)[0].strip())
        else:
            row[26] = None 
        
        # Year Built
        year_built = row[59]
        if not (year_built is None or year_built==''):
            row[28] = year_built
        else:
            row[28] = None
            
        # Units
        units = row[63]
        if not (units is None or units==''):
            row[29] = int(units)
        else:
            row[29] = None
        
        # Bedrooms
        bedrooms = row[61]
        if not (bedrooms is None or bedrooms==''):
            row[30] = bedrooms
        else:
            row[30] = None
        
        # Bathrooms
        bathrooms = row[62]
        if not (bathrooms is None or bathrooms==''):
            row[31] = bathrooms
        else:
            row[31] = None
            
        # Building Square Feet
        building_sqft = row[43]
        if not (building_sqft is None or building_sqft==''):
            row[32] = building_sqft
        else:
            row[32] = None

        # Update the row.
        cursor.updateRow(row)
del cursor

# create a spatial reference object for the output coordinate system 
out_coordinate_system = arcpy.SpatialReference('NAD 1983 UTM Zone 10N') 
arcpy.Project_management(washoeParcels, parcel_out, out_coordinate_system)

print('New Washoe Parcels transformed')

New Washoe Parcels transformed


In [25]:
fields = arcpy.ListFields(washoeParcels)
for field in fields:
    print("{0}, {1} ,{2}"
          .format(field.name, field.type, field.length))

OBJECTID, OID ,4
Shape, Geometry ,0
APN, Integer ,4
REGION, String ,6
PIN, String ,10
RECMAP, String ,6
BOOK, String ,3
PAGE, String ,2
BLOCK, String ,1
PARCEL, String ,2
SUBNAME, String ,50
TOWNSHIP, String ,50
RANGE, String ,50
SECTION_, String ,50
FLOOR, Double ,8
MAPLINK, String ,80
FLR, Double ,8
STREETNUM, String ,10
STREETDIR, String ,10
STREET, String ,50
CITY, String ,50
SITUSZIP, String ,5
FIRSTNAME, String ,100
LASTNAME, String ,100
MAILING1, String ,100
MAILING2, String ,100
MAILCITY, String ,50
MAILSTATE, String ,10
MAILZIP, String ,16
LAND_USE, String ,16
WATER, String ,4
SEWER, String ,4
ACREAGE, Double ,8
TAXDIST, String ,4
BEDROOMS, Integer ,4
BATHS, Double ,8
YEARBLT, Integer ,4
LANDASS, Integer ,4
BUILDASS, Integer ,4
TOTALASS, Integer ,4
LANDAPR, Integer ,4
BUILDAPR, Integer ,4
TOTALAPR, Integer ,4
DEPRECIATION, Double ,8
SALEDATE, String ,10
SALEPRICE, Integer ,4
OCCUPANCY, String ,4
PROPCODE, String ,20
TAXYEAR, String ,4
STORIES, String ,50
TOWNHOUSE, String ,1
S

### Merge

In [35]:
# delete in-memory
arcpy.Delete_management("memory")
print("Deleted Memory Workspace: " + strftime("%Y-%m-%d %H:%M:%S"))

# out merge fc
parcel_out = "Parcel_Staging"

# input feature classes
ccParcel = "Parcel_CC_Transformed"
dgParcel = "Parcel_DG_Transformed"
elParcel = "Parcel_EL_Transformed"
plParcel = "Parcel_PL_Transformed"
waParcel = "Parcel_WA_Transformed"

# Create FieldMappings object to manage merge output fields
fieldMappings = arcpy.FieldMappings()
# Add all fields from all parcel staging layers
fieldMappings.addTable(ccParcel)
fieldMappings.addTable(dgParcel)
fieldMappings.addTable(elParcel)
fieldMappings.addTable(plParcel)
fieldMappings.addTable(waParcel)

# Remove all output fields from the field mappings, except fields in field_master list
for field in fieldMappings.fields:
    if field.name not in [  'OBJECTID',
                            'APN_TRPA',                 #0
                            'PPNO_TRPA',                #1
                            'JURISDICTION_TRPA',        #2
                            'COUNTY_TRPA',
                             # parcel address   
                            'HSE_NUMBR_TRPA',           #3
                            'STR_DIR_TRPA',             #4
                            'STR_NAME_TRPA',            #5
                            'STR_SUFFIX_TRPA',          #6
                            'UNIT_NUMBR_TRPA',          #7
                            'APO_ADDRESS_TRPA',         #8
                            'PSTL_TOWN_TRPA',           #9
                            'PSTL_STATE_TRPA',          #10
                            'PSTL_ZIP5_TRPA',           #11
                            # owner fields
                            'OWN_FIRST_TRPA',           #12
                            'OWN_LAST_TRPA',            #13
                            'OWN_FULL_TRPA',            #14
                            'MAIL_ADD1_TRPA',           #15
                            'MAIL_CITY_TRPA',           #16
                            'MAIL_STATE_TRPA',          #17
                            'MAIL_ZIP5_TRPA',           #18
                            # value fields  
                            'AS_LANDVALUE_TRPA',        #19
                            'AS_IMPROVALUE_TRPA',       #20
                            'AS_SUM_TRPA',              #21
                            'TAX_LANDVALUE_TRPA',       #22 
                            'TAX_IMPROVALUE_TRPA',      #23
                            'TAX_SUM_TRPA',             #24
                            'TAX_YEAR_TRPA',            #25
                            # land use fields 
                            'COUNTY_LANDUSE_CODE_TRPA', #26
                            'COUNTY_LANDUSE_TRPA',      #27
                            # Fields for building info
                            "YEAR_BUILT_TRPA",          #28
                            'UNITS_TRPA',               #29
                            'BEDROOMS_TRPA',            #30
                            'BATHROOMS_TRPA',           #31
                            'BUILDING_SQFT_TRPA',       #32
                            'VHR_TRPA',                 #33
                            'HOA_TRPA',                 #34
                            'SHAPE@']:
        # remove everything else
        fieldMappings.removeFieldMap(fieldMappings.findFieldMapIndex(field.name)) 
    
# Use Merge tool to move features into single dataset
arcpy.management.Merge([ccParcel, dgParcel, elParcel, plParcel, waParcel ], parcel_out, fieldMappings)
print("Transformed Parcel Datasets Merged")

# out merge fc
parcel_out = "Parcel_Staging"
result = arcpy.GetCount_management(parcel_out)
print('{} has {} records'.format(parcel_out, result[0]))

# out merge fc
parcel_out = "Parcel_Staging"

# delete unneccesary parcels
parcelDelete = "ParcelDelete"

# Run MakeFeatureLayer
arcpy.management.MakeFeatureLayer(parcel_out, parcelDelete)
 
arcpy.management.SelectLayerByAttribute(parcelDelete, 'NEW_SELECTION', 
                                        "APN_TRPA = '' Or APN_TRPA LIKE '920%' Or APN_TRPA LIKE '910%' OR APN_TRPA LIKE '%NP%' OR APN_TRPA LIKE '%ROW%' OR APN_TRPA LIKE '%UN%'")

# Run GetCount and if some features have been selected, then 
#  run DeleteFeatures to remove the selected features.
deleteCount = arcpy.management.GetCount(parcelDelete)[0]
if int(deleteCount) > 0:
    arcpy.management.DeleteFeatures(parcelDelete)
    print('{} records deleted'.format(deleteCount))
    
result = arcpy.GetCount_management(parcel_out)
print('{} has {} records now.'.format(parcel_out, result[0]))

Deleted Memory Workspace: 2023-05-09 20:44:43
Transformed Parcel Datasets Merged
Parcel_Staging has 67108 records
2504 records deleted
Parcel_Staging has 64604 records now.


### Additional Transformation of County Data

In [ ]:
def UpdateFieldFromDictionary(featureclass, field, update_dictionary):
    with arcpy.da.UpdateCursor(featureclass, field) as cursor:
        for row in cursor:
            key_field_value = row[0]
            if key_field_value in update_dictionary:
                row[0] = update_dictionary[key_field_value]
                cursor.updateRow(row)

suffix_dict = {
    'CI':'CIR',
    'BL':'BLVD',
    'TR':'TRL',
    'WY':'WAY',
    'E': '',
    'L':'',
    'AV':'AVE',
    'LP':'LOOP',
    'HY':'HWY',
    'PY':'PKWY',
    'PKY':'PKWY',
    'DRIVE': 'DR'
}

#We can ignore
jurisdiction_dict= {
    'EL': 'El Dorado County',
    'CSLT': 'City of South Lake Tahoe',
    'WA': 'Washoe County',
    'PL':'Placer County',
    'DG':'Douglas County'
    'CC': 'Carson City County',
    'SL': 'City of South Lake Tahoe'
}

suffix_field = ["NewSuffix"]

UpdateFieldFromDictionary('Parcel_Staging', suffix_field, suffix_dict)

replacement_values = ['UNIT','SUITE','SPACE','NULL']
set_to_blank_values = ['0', '0 NULL']
with arcpy.da.UpdateCursor(Parcel_Staging, ["STR_NAME_TRPA"]) as cursor:
    for row in cursor:
        row[0]=row[0].upper()        
        for replacement_value in replacement_values:
            row[0] = row[0].replace(replacement_value, '')
        row[0] = row[0].replace('  ', ' ')
        if row[0].isin(set_to_blank_values)
            row[0]=''
        if row[0].startswith('0 '):
            row[0]=row[0][2:]
        if (row[0] == '0 NO ADDRESS ON FILE')| (row[0] == 'NO ADDRESS ON FILE'):
            NewStreet='NO ADDRESS ON FILE'
            
        cursor.updateRow(row)

                


### TRPA Attribution

In [63]:
print("Starting TRPA Attribution: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting TRPA Attribution: " + strftime("%Y-%m-%d %H:%M:%S"))

# in and out with the same name overwrite == True
ParcelStaging = "Parcel_Staging"
ParcelPoint   = "Parcel_Point"
ParcelNew     = 'Parcel_Staging_Attributed'

# copy data into an in_memory feature class for warp speed.
ParcelLayer = r"memory/ParcelLayer"
arcpy.CopyFeatures_management(ParcelStaging, ParcelLayer)

# Add TRPA fields.
arcpy.management.AddFields(ParcelLayer,trpaFields)

# copy shapes to points in new parcel point layer
arcpy.FeatureToPoint_management(ParcelLayer, ParcelPoint, "INSIDE")

print("Copied features to points: "+ strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Copied features to points: "+ strftime("%Y-%m-%d %H:%M:%S"))


# ### County Atribute Update -------------------------------------------------------------------------------------###

print("Starting the County attribute update: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the County attribute update: " + strftime("%Y-%m-%d %H:%M:%S"))

with arcpy.da.UpdateCursor(ParcelLayer, ["JURISDICTION_TRPA", "COUNTY_TRPA"]) as cursor:
    for row in cursor:
        # set county field before changing EL to CSLT in Jurisdiction field
        row[1] = row[0] 
        cursor.updateRow(row)
del cursor
print("County Attribute Updated")
### Ownership Type Attribute Update ------------------------------------------------------------------------------###
print("Starting the Ownership Type attribute update: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Ownership Type attribute update: " + strftime("%Y-%m-%d %H:%M:%S"))

# owner name lists - this is a stupid way to figure this out
fedOwnList = ("USA FOREST SERVICE", "USDA FOREST SERVICE", "USDA - FOREST SERVICE", "UNITED STATES POSTAL", 
              "UNITED STATES OF AMERICA", "UNITED STATES FOREST SERVICE", "U S POSTAL SERVICE", "U S COAST GUARD",
              "U S A FOREST SERVICE``", "U S A FOREST SERVICE", "LAKE VALLEY RANGER STA", "DEPT OF VETRANS AFFAIRS%", 
              "DEPT OF VETERANS AFFAIRS%", "DEPT OF VETERANS AFFAIRS %", "BUREAU OF LAND MANAGEMENT", "U S FOREST SERVICE",
              "DEPT OF VETERANS AFFAIRS  & ERSKINE NEIL H TR", "DEPT OF VETERANS AFFAIRS  & MASTERS DANE C", 
              "DEPT OF VETERANS AFFAIRS  & SLEZAK FRANK J CO TR", "DEPT OF VETERANS AFFAIRS & RIVES DONALD E JR"
              "DEPT OF VETERANS AFFAIRS & WILLIAMS MATTHEW G DBA WILLIAMS VACATION HOME", "DEPT OF VETRANS AFFAIRS & WILSON VIVIAN M",
              "DEPARTMENT OF TRANSPORTATION", "USA FOREST SERVICE & OWNERSHIP UNVERIFIED", "U S A FOREST SERVICE & LAKE TAHOE BASIN MNGMT UNIT",
              "U S A FOREST SERVICE & OWNERSHIP UNVERIFIED", "U S D A FOREST SERVICE", "UNITED STATES & DEPT OF AGRICULTURE",
              "UNITED STATES OF AMERICA & ATTN RICHARD T FLYNN", "UNITED STATES OF AMERICA & DEPARTMENT OF AGRICULTU", "UNITED STATES OF AMERICA & F/S DEPT OF AGRICULTURE",
              "UNITED STATES OF AMERICA & FOREST SER. DEPT OF AG.", "UNITED STATES OF AMERICA & FOREST SERVICE", "UNITED STATES OF AMERICA & FOREST SERVICE (USDA)",
              "UNITED STATES OF AMERICA & FOREST SERVICE DEPT OF", "UNITED STATES OF AMERICA & FOREST SERVICE TAHOE BA", "UNITED STATES OF AMERICA & FOREST SERVICE USDA",
              "UNITED STATES OF AMERICA & FOREST SVC/DEPT OF AGRI", "UNITED STATES OF AMERICA & LAKE TAHOE BASIN MANAGM",
              "UNITED STATES OF AMERICA & LAKE TAHOE BASIN MGT UN", "UNITED STATES OF AMERICA & REGIONAL LAND ADJUSTMEN",
              "UNITED STATES OF AMERICA & U S FOREST SERVICE", "UNITED STATES OF AMERICA & U S FOREST SERVIE",
              "UNITED STATES OF AMERICA & USDA FOREST SER LAKE TA", "UNITED STATES OF AMERICA & USDA FOREST SERVICE",
              "USDA - FOREST SERVICE & LAKE TAHOE BASIN MGMT UNIT")

stateOwnList = ("TAHOE CONSERVANCY", "STATE OF NEVADA FOREST SERVICE", "STATE OF NEVADA", "STATE OF CALIFORNIA THE", 
                "STATE OF CALIFORNIA (EASEMENT)", "STATE OF CALIFORNIA", "STATE OF CA", "REGENTS OF UNIV OF CALIF",
                "UNIVERSITY CALIFORNIA REGENTS", "UNIVERSITY OF NEVADA RENO", "NEVADA, STATE OF", "NEVADA STATE OF", 
                "CALIFORNIA TAHOE CONSERVANCY ET AL", "CALIFORNIA TAHOE CONSERVANCY", "CALIFORNIA STATE OF THE", 
                "CALIFORNIA STATE OF ET AL", "CALIFORNIA STATE OF", "CA STATE DEPT TRANSPORTATION", 
                "CA TAHOE CONSERVANCY", "CALIFORNIA STATE OF TAHOE CONSERVANCY", "CALIFORNIA STATE OF THE", 
                "NEVADA DEPT OF TRANSPORTATION", "STATE OF CALIFORNIA & CALIFORNIA TAHOE CONSERVANCY", "STATE OF CALIFORIA & CALIFORNIA TAHOE CONSERVANCY",
                "STATE OF CALIFORNIA & CA TAHOE CONSERVANCY", "STATE OF CALIFORNIA & CALIFORNIA TAHOE CONSERVANCY CALIFORNIA TAHOE CONSERVANCY",
                "STATE OF CALIFORNIA & CALIFORNIA TAHOE CONSEVANCY", "STATE OF CALIFORNIA & DEPART OF TRANSPORTATION", "STATE OF CALIFORNIA & DEPARTMENT OF GENERAL SERVIC", 
                "STATE OF CALIFORNIA & DEPARTMENT OF TRANSPORTATION", "STATE OF CALIFORNIA & DEPT OF GEN SRVS R E DIV", "STATE OF CALIFORNIA & DEPT OF GENERAL SERVICES",
                "STATE OF CALIFORNIA & DEPT OF PARKS & RECREATION", "STATE OF CALIFORNIA & DEPT OF TRANSPORTATION", "STATE OF CALIFORNIA & PARKS & RECREATION",
                "STATE OF CALIFORNIA (EASEMENT) & CALIFORNIA TAHOE", "CALIFORNIA STATE PARKS AND RECREATION",
                "CALIFORNIA STATE OF & DEPT GEN SERVICES REAL ESTAT")

localOwnList = ("ZEPHYR COVE GENERAL IMP DIST", "WASHOE COUNTY SCHOOL DISTRICT BOARD", "WASHOE COUNTY", "WASHOE TRIBE OF NV & CA", 
                "TALMONT RESORT IMPROVEMENT DISTRICT", "TALMONT RESORT IMPR DIST", "TALMONT RESORT IMP DISTRICT",
                "TALMONT RESORT IMP DIST", "TAHOE PARADISE RESORT IMP DIST", "TAHOE PARADISE RES IMP DST",
                "TAHOE FOREST HOSPITAL DISTRICT", "TAHOE TRUCKEE UNIFIED SCHOOL DISTRICT", "TAHOE TRUCKEE UNIFIED SCH DIST", 
                "TAHOE DOUGLAS FIRE PROTECT DIST", "TAHOE DOUGLAS SEWER DIST", "TAHOE DOUGLAS DISTRICT", 
                "TAHOE CITY PUBLIC UTILITY DISTRICT", "TAHOE CITY PUBLIC UTILITY DIST", "TAHOE CITY PUBLIC UTILDIST", 
                "TAHOE CITY PUB UTILITY DST", "TAHOE CITY PUB UTILITY DIS", "TAHOE CITY P U D", "TAHOE CITY CEMETERY DIST", 
                "SOUTH TAHOE REDEVELP AGENCY", "SOUTH TAHOE REFUSE CO", "SOUTH TAHOE PUD", "SOUTH TAHOE PUBLIC UTL DST",
                "SOUTH TAHOE PUBLIC UTILITYDIST", "SOUTH TAHOE PUBLIC UTILITY DST", "SOUTH TAHOE PUBLIC UTILITY DIS", 
                "SOUTH TAHOE PUBLIC UTILITY", "SOUTH TAHOE PUBLIC UTIL DT", "SOUTH TAHOE PUBLIC UTIL DIST", 
                "SOUTH TAHOE PUBLIC", "SOUTH TAHOE PUB UTIL DIST", "SOUTH LAKE TAHOE CTYOF 1/3", "SOUTH LAKE TAHOE CITY OF", 
                "SO TAHOE PUBLIC UTILITY DIST", "SO TAHOE PUB UTIL DIST", "SIERRA NEVADA COLLEGE", "ROUND HILL GEN IMP DIST",
                "PLACER COUNTY REDEVELOPMENT AGENCY", "PLACER COUNTY OF", "PLACER COUNTY", "NORTH TAHOE PUBLIC UTL DIST",
                "NORTH TAHOE PUBLIC UTILITY DISTRICT", "NORTH TAHOE PUBLIC UTILITY DIST", "NORTH TAHOE PUBLIC UTILITY DIS", 
                "NORTH TAHOE PUBLIC UTILITIES DIST", "NORTH TAHOE PUBLIC UTILIITY DISTRICT", "NORTH TAHOE P U D",
                "NORTH TAHOE FIRE PROTECTION DISTRICT", "NORTH TAHOE FIRE PROTECTION", "NORTH TAHOE FIRE DIST",
                "NORTH LAKE TAHOE FIRE PROTECTION DIST", "N TAHOE FIRE PROTECTION DIST", "MEEKS BAY FIRE PROT DIST", 
                "LAKERIDGE GENERAL IMP DIST", "LAKE VALLEY FIRE PROTECTION", "LAKE VALLEY FIRE PROT DST", "LAKE VALLEY FIRE PROT DIST", 
                "LAKE VALLEY FIRE DISTRICT", "LAKE TAHOE UNIFIED SCHOOL DIST", "LAKE TAHOE SCHOOL", "LAKERIDGE GENERAL IMP DIST", 
                "LAKE TAHOE FIRE PROTECTION DIST", "LAKE TAHOE FIRE PROTECT DIST", "LAKE TAHOE COMM COLLEGE DIST",
                "LAKE TAHOE COMM COL DIST", "KINGSBURY GENERAL IMP DISTRICT", "KINGSBURY GENERAL IMP DIST",
                "INCLINE VILLAGE GENERAL IMPROVEMENT DISTRICT", "INCLINE VILLAGE GENERAL IMPROVEMENT DIST", 
                "DOUGLAS COUNTY SEWER DIST", "DOUGLAS COUNTY SCHOOL DIST", "DOUGLAS COUNTY", "DOUGLAS CO SEWER IMP DIST #1", 
                "COUNTY OF EL DORADO", "CITY OF SOUTH LAKE TAHOE", "EL DORADO IRRIGATION DISTRICT", 
                "HAPPY HOMESTEAD CEMETERY DIST", "WASHOE TRIBE", "SOUTHTAHOE PUBLIC UTILITY DIST", "DOUGLAS COUNTY TRUSTEE", 
                "DOUGLAS COUNTY TRUSTEE (HOLD)", "WASHOE TRIBE OF NEVADA AND CALIFORNIA", "ALPINE SPRINGS CO WATER DIST", 
                "ALPINE SPRINGS COUNTY WATER DISTRICT", "ALPINE SPRINGS WATER DISTRICT", "NORTHSTAR COMMUNITY SERVICE DISTRICT",
                "SQUAW VALLEY CO WATER DIST", "SQUAW VALLEY PUBLIC SERVICE DISTRICT", "TRUCKEE TAHOE AIRPORT DISTRICT", 
                "COUNTY OF EL DORADO & ATTEN: PAUL MCINTOSH", "COUNTY OF EL DORADO & BOARD OF SUPERVISORS", 
                "COUNTY OF EL DORADO & BOARD OF SUPERVISORS", "COUNTY OF EL DORADO & C/O BOARD OF SUPERVISORS", "COUNTY OF EL DORADO & COUNSEL",
                "COUNTY OF EL DORADO & COUNSEL'S OFFICE", "COUNTY OF EL DORADO & DEPARTMENT OF PUBLIC WORKS", "COUNTY OF EL DORADO & DEPARTMENT OF TRANSPORTATION",
                "COUNTY OF EL DORADO & DEPT OF PUBLIC WORKS", "COUNTY OF EL DORADO & DEPT OF TRANSPORTATION", "COUNTY OF EL DORADO & GENERAL SERVICES DEPARTMENT",
                "COUNTY OF EL DORADO & OF EL DORADO", "COUNTY OF EL DORADO & PUBLIC WORKS DEPARTMENT", "EL DORADO CO OFFICE EDUCATION", "EL DORADO COUNTY & SUPERINTENDENT OF SCHOOLS",
                "EL DORADO COUNTY & BOARD OF SUPERVISORS", "LAKE TAHOE COMMUNITY COLLEGE DIST", "LAKE VALLEY RANGER STA & U S FOREST SERVICE",
                "LAKE VALLEY FIRE PROTECTION & DISTRICT POLITICAL S", "SOUTH TAHOE PUBLIC & UTILITY DISTRICT",
                "SOUTH TAHOE PUBLIC UTILITY &  DISTRIC", "SOUTH TAHOE PUBLIC UTIL DIST & CA MUNICIPAL CORP", "TAHOE CITY PUBLIC UTIL DST",
                "TAHOE RESOURCE CONSERVATION &  DISTRIC", "TAHOE RESOURCE CONSERVATION DIST  C/O DISTRICT MANAGER", "FALLEN LEAF COMM SERVICES DIST",
                "FALLEN LEAF LAKE COMM SERVDIST", "TAHOE DOUGLAS VISITORS AUTH", "KINGSBURY GENARAL IMP DIST", "ALPINE SPRINGS CO WTR DIST FIN CORP",
                "COUNTY OF PLACER", "MCKINNEY WATER DISTRICT", "NORTHSTAR COMMUNITY SERVICES DISTRICT", "PLACER COUNTY PUBLIC WORKS",
                "REDEVELOPMENT AGENCY OF THE COUNTY OF PL", "TRUCKEE DONNER PUBLIC UTILITY DISTRICT", "TRUCKEE SANITARY DISTRICT",
                "TAHOE TRANSPORTATION DISTRICT") 

with arcpy.da.UpdateCursor(ParcelLayer, ["OWN_FULL_TRPA", "OWNERSHIP_TYPE_TRPA"]) as cursor:
    for row in cursor:
        # set ownership type
        own = row[0]
        if not (own is None or own == "" or own.isspace() == True):
            if own in fedOwnList:
                row[1] = "Federal"
            elif own in localOwnList:
                row[1] = "Local"
            elif own in stateOwnList:
                row[1] = "State"
            elif not own in (fedOwnList, localOwnList, stateOwnList):
                row[1] = "Private" 
            cursor.updateRow(row)
del cursor
print ("The 'OWNERSHIP_TYPE' field in the parcel data has been updated")
# log.info("The 'Owernshipe Type' field in the parcel data has been updated")

### Existing Landuse Attribute Update ----------------------------------------------------------------------------###
fields = ("COUNTY_LANDUSE_CODE_TRPA",
          "COUNTY_LANDUSE_TRPA",  
          "EXISTING_LANDUSE_TRPA", 
          'JURISDICTION_TRPA')

with arcpy.da.UpdateCursor(ParcelLayer, fields) as cursor:
    for row in cursor:
        ctyluc = row[0]
        cty = row[3]
        # set Washoe county land use
        # set TRPA Land Use Description
        if (row[0] != None or row[0] != "") and (row[3] == 'WA'):
            if ctyluc in ('400', '410', '440', '500', '510', '520', '630', '640', '670', '720'):
                row[2] = "Commercial"
            elif ctyluc in ('210', '250'):
                row[2] = "Condominium"
            elif ctyluc in ('240'):
                row[2] = "Condominium Common Area"
            elif ctyluc in ('220', '230', '300', '310', '320', '330', '340', '350', '360'):
                row[2] = "Multi-Family Residential"
            elif ctyluc in ('600', '620'):
                row[2] = "Open Space"
            elif ctyluc in ('700', '710', 'PBRD'):
                row[2] = "Public Service"
            elif ctyluc in ('190'):
                row[2] = "Recreation"
            elif ctyluc in ('200'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('420', '430'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('100', '110', '120', '130', '140', '150', '160', '170', '180'):
                row[2] = "Vacant"            
            elif ctyluc is None:
                row[2] == ''
        if (row[0] != None or row[0] != "") and (row[3] == 'WA'):
            if ctyluc in ('710'):
                row[1] = "Intracounty public utility"
            elif ctyluc == '700':
                row[1] = 'Centrally assessed public utility'
            elif ctyluc == '510':
                row[1] = 'Commercial Industrial: retail or office with Indus'
            elif ctyluc == '500':
                row[1] = 'General industrial: light indust, trucking, warehs'
            elif ctyluc == '440':
                row[1] = 'Resort commercial: ski, golf, sports, etc.'
            elif ctyluc == '430':
                row[1] = 'Commercial hotel or motel'
            elif ctyluc == '420':
                row[1] = 'Casino or hotel casino'
            elif ctyluc == '410':
                row[1] = 'Offices, professional and business, banks, etc.'
            elif ctyluc == '400':
                row[1] = 'General Commercial: retail, mixed, parking, school'
            elif ctyluc == '340':
                row[1] = 'Ten or more units'
            elif ctyluc == '330':
                row[1] = 'Five to Nine Units'
            elif ctyluc == '320':
                row[1] = 'Three or four Units'
            elif ctyluc == '310':
                row[1] = 'Two Single Family Units'
            elif ctyluc == '300':
                row[1] = 'Duplex'
            elif ctyluc == '250':
                row[1] = 'Condo or Townhouse valued as apartment use'
            elif ctyluc == '240':
                row[1] = 'Common Area'
            elif ctyluc == '210':
                row[1] = 'Condominium or Townhouse'
            elif ctyluc == '200':
                row[1] = 'Single Family Residence'
            elif ctyluc == '190':
                row[1] = 'Public Parks: vacant or improved'
            elif ctyluc == '170':
                row[1] = 'Other, unbuildable: roads, restrictions, terrain'
            elif ctyluc == '160':
                row[1] = 'Splinter, unbuildable: small size or shape'
            elif ctyluc == '140':
                row[1] = 'Vacant, commercial'
            elif ctyluc == '130':
                row[1] = 'Vacant, multi-residential'
            elif ctyluc == '120':
                row[1] = 'Vacant, single family'
            elif ctyluc == '110':
                row[1] = 'Vacant, under development'
            elif ctyluc == '100':
                row[1] = 'Vacant, other or unknown'            
            elif ctyluc is None:
                row[1] == ''
            cursor.updateRow(row)
        # Set Carson City County Land Use Descriptions
        if (row[0] != None or row[0] != "") and (row[3] == 'CC'):
            if ctyluc in ('400', '401', '402', '403', '404', '408', '410', '411', 
                          '412', '440', '441', '460', '470', '480', '482', '490', 
                          '500', '501', '510', '511', '512', '513', '520', '521', 
                          '560', '570', '580', '582', '590', '624', '625', '694', 
                          '800', '820', '830', '840', '880', '882', '890', '920', 
                          '921', '930', '960', '980', '990'):
                row[2] = "Commercial"
            elif ctyluc in ('210', '211'):
                row[2] = "Condominium"
            elif ctyluc == '970':
                row[2] = "Condominium Common Area"
            elif ctyluc in ('240', '241', '300', '301', '310', '311', '313', '320', 
                            '321', '330', '331', '333', '340', '341', '350', '360', 
                            '370', '380', '382', '390', '698'):
                row[2] = "Multi-Family Residential"
            elif ctyluc in ('190', '600', '610', '612', '613', '614', '615', '616', 
                            '618', '620', '695', '696', '697', '810'):
                row[2] = "Open Space"
            elif ctyluc in ('190', '700', '710', '711', '720', '731', '732', '733', '780', 
                            '790', '910', '922'):
                row[2] = "Public Service"
            elif ctyluc in ('450', '900'):
                row[2] = "Recreation"
            elif ctyluc in ('200', '201', '220', '222', '230', '231', '232', '260', 
                            '270', '280', '282', '290', '622', '692', '693'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('420', '421', '430', '431', '432', '514'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('100', '108', '110', '117', '120', '130', '140', '150', '160'):
                row[2] = "Vacant"
            elif ctyluc is None:
                row[2] == ''
        if (row[0] != None or row[0] != "") and (row[3] == 'CC'):
            if ctyluc == '980':
                row[1] = 'Special Purpose with Minor Improvements'
            elif ctyluc == '320':
                row[1] = 'Three to Four Units'
            elif ctyluc == '280':
                row[1] = 'Single Family Residential with Minor Improvements'
            elif ctyluc == '190':
                row[1] = 'Vacant - Public Use Lands'
            elif ctyluc == '120':
                row[1] = 'Vacant - Single Family Residential' 
            elif ctyluc is None:
                row[1] == ''
            cursor.updateRow(row)           
        # update Douglas Land Use descriptions        
        if (row[0] != None or row[0] != "") and (row[3] == 'DG'):
            if ctyluc in ('400', '402', '410', '411', '412', 
                          '440', '460', '470', '480', '500', 
                          '510', '560', '580', '582'):
                row[2] = "Commercial"
            elif ctyluc in ('210', '211'):
                row[2] = "Condominium"
            elif ctyluc == '270':
                row[2] = "Condominium Common Area"
            elif ctyluc in ('300', '310', '320', '330', '350', '390'):
                row[2] = "Multi-Family Residential"
            elif ctyluc == '190':
                row[2] = "Open Space"
            elif ctyluc in ('700', '710', '711', '910', '980', '970'):
                row[2] = "Public Service"
            elif ctyluc in ('450', '900', '970'):
                row[2] = "Recreation"
            elif ctyluc in ('200', '220', '230', '236', '240', '280', '282'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('420', '430'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('100', '110', '117', '120', '130', '140'):
                row[2] = "Vacant"
            elif ctyluc is None:
                row[2] == ''
        if (row[0] != None or row[0] != "" or ctyluc.isspace() != True) and (row[3] == 'DG'):
            if ctyluc == '980':
                row[1] = 'Special Purpose with Minor Improvements'
            elif ctyluc == '970':
                row[1] = 'Special Purpose Common Area'
            elif ctyluc == '910':
                row[1] = 'Cemeteries'
            elif ctyluc == '900':
                row[1] = 'Parks for Public Use'
            elif ctyluc == '711':
                row[1] = 'Communication, Transportation, and Utility Property of a Local Nature Under Construction'
            elif ctyluc == '710':
                row[1] = 'Communication, Transportation, and Utility Property of a Local Nature'
            elif ctyluc == '700':
                row[1] = 'Operating Communication, Transportation, and Utility Property of an Interstate or Intercounty Nature'
            elif ctyluc == '582':
                row[1] = 'Industrial with Minor Improvements - with structures insufficient to determine intended use'
            elif ctyluc == '580':
                row[1] = 'Industrial with Minor Improvements'
            elif ctyluc == '560':
                row[1] = 'Industrial Auxiliary Area'
            elif ctyluc == '510':
                row[1] = 'Commercial Industrial - retail or office use combined with Industrial use'
            elif ctyluc == '500':
                row[1] = 'General Industrial - light industry, trucking and warehousing, service, repair, etc.'
            elif ctyluc == '480':
                row[1] = 'Commercial with Minor Improvements'
            elif ctyluc == '470':
                row[1] = 'Commercial Common Area'
            elif ctyluc == '460':
                row[1] = 'Commercial Auxiliary Area'
            elif ctyluc == '450':
                row[1] = 'Golf Course'
            elif ctyluc == '440':
                row[1] = 'Commercial Recreation'
            elif ctyluc == '430':
                row[1] = 'Commercial Living Accommodations'
            elif ctyluc == '420':
                row[1] = 'Casino or Hotel Casino'
            elif ctyluc == '410':
                row[1] = 'Offices, Professional and Business Services'
            elif ctyluc == '402':
                row[1] = 'Parking and/or Parking Structures'
            elif ctyluc == '400':
                row[1] = 'General Commercial'
            elif ctyluc == '390':
                row[1] = 'Mixed Use with Multi-Family Residential as primary use'
            elif ctyluc == '382':
                row[1] = 'Multi-Family Residential with Minor Improvements - No livable structures'
            elif ctyluc == '380':
                row[1] = 'Multi-Family Residential with Minor Improvements'
            elif ctyluc == '370':
                row[1] = 'Multi-Family Residential Common Area'
            elif ctyluc == '360':
                row[1] = 'Multi-Family Residential Auxiliary Area'
            elif ctyluc == '350':
                row[1] = 'Manufactured Home Park - Ten or More Manufactured Home Units'
            elif ctyluc == '341':
                row[1] = 'Five or More Units - High Rise Under Construction'
            elif ctyluc == '340':
                row[1] = 'Five or More Units - High Rise'
            elif ctyluc == '333':
                row[1] = 'Exempt or Partially Exempt Apartment Building'
            elif ctyluc == '331':
                row[1] = 'Five or More Units - Low Rise Under Construction'
            elif ctyluc == '330':
                row[1] = 'Five or More Units - Low Rise'
            elif ctyluc == '321':
                row[1] = 'Three to Four Units Under Construction'
            elif ctyluc == '320':
                row[1] = 'Three to Four Units'
            elif ctyluc == '313':
                row[1] = 'Multi-Family Residence with Manufactured Home Conversion'
            elif ctyluc == '311':
                row[1] = 'Two Single Family Units Under Construction'
            elif ctyluc == '310':
                row[1] = 'Two Single Family Units'
            elif ctyluc == '301':
                row[1] = 'Duplex Under Construction'
            elif ctyluc == '300':
                row[1] = 'Duplex'
            elif ctyluc == '290':
                row[1] = 'Mixed Use with Single Family Residential as primary use'
            elif ctyluc == '282':
                row[1] = 'Single Family Residential with Minor Improvements - No livable structures'
            elif ctyluc == '280':
                row[1] = 'Single Family Residential with Minor Improvements'
            elif ctyluc == '270':
                row[1] = 'Single Family Residential Common Area'
            elif ctyluc == '260':
                row[1] = 'Single Family Residential Auxiliary Area'
            elif ctyluc == '240':
                row[1] = 'Individual Residential Unit - Townhouse or Row House'
            elif ctyluc == '236':
                row[1] = 'Personal Property Manufactured Home Secured'
            elif ctyluc == '233':
                row[1] = 'Secured Manufactured Home with Site Built Additions (Not Converted)'
            elif ctyluc == '232':
                row[1] = 'Manufactured Home - Unsecured with Site Built Additions'
            elif ctyluc == '231':
                row[1] = 'Manufacture Home Conversions Pending'
            elif ctyluc == '230':
                row[1] = 'Personal Property Manufactured Home on the Unsecured Roll'
            elif ctyluc == '222':
                row[1] = 'Manufactured Home (Converted) with Site Built Additions'
            elif ctyluc == '220':
                row[1] = 'Manufactured Home Converted to Real Property'
            elif ctyluc == '211':
                row[1] = 'Individual Unit in a Multiple Unit Building Under Construction'
            elif ctyluc == '210':
                row[1] = 'Individual Unit in a Multiple Unit Building'
            elif ctyluc == '201':
                row[1] = 'Single Family Residence Under Construction'
            elif ctyluc == '200':
                row[1] = 'Single Family Residence'
            elif ctyluc == '190':
                row[1] = 'Vacant - Public Use Lands'
            elif ctyluc == '150':
                row[1] = 'Vacant - Industrial'
            elif ctyluc == '140':
                row[1] = 'Vacant - Commercial'
            elif ctyluc == '130':
                row[1] = 'Vacant - Multi-Residential'
            elif ctyluc == '120':
                row[1] = 'Vacant - Single Family Residential'
            elif ctyluc == '117':
                row[1] = 'Vacant - Roads/Easements'
            elif ctyluc == '110':
                row[1] = 'Vacant - Splinter and Other Unbuildable'
            elif ctyluc == '108':
                row[1] = 'Vacant - Patented Mining Claim, Not Mined'
            elif ctyluc == '100':
                row[1] = 'Vacant - Unknown/Other'
            elif ctyluc is None:
                row[1] == ''
            cursor.updateRow(row)        
        # Set El Dorado County Land Use Description fields
        if (row[0] != None or row[0] != "") and (row[3] == 'EL'):
            if ctyluc in ('03', '29', '31', '32', '34', '36', '37', '38', '39', '41', '42', '43', '44', '45', '46', '47', '48', 
                          '65', '67', '68', '82', '91', '93'):
                row[2] = "Commercial"
            elif ctyluc == '14':
                row[2] = "Condominium"
            elif ctyluc == '89':
                row[2] = "Condominium Common Area"
            elif ctyluc in ('01', '07', '12', '13', '16', '18', '19', '28', '35'):
                row[2] = "Multi-Family Residential"
            elif ctyluc in ('25', '26', '50', '51', '52', '55', '56', '60', '70', '75', '79'):
                row[2] = "Open Space"
            elif ctyluc in ('90', '92', '94', '96', '97', '98', '99'):
                row[2] = "Public Service"
            elif ctyluc in ('61', '62', '63', '64'):
                row[2] = "Recreation"
            elif ctyluc in ('06', '11', '15', '22', '23'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('33', '80', '81'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('00', '02', '05', '17', '21', '24', '30', '40'):
                row[2] = "Vacant"
            elif ctyluc is None:
                row[2] == ''
        if (row[0] != None or row[0] != "") and (row[3] == 'EL'):
            if ctyluc == '98':
                row[1] = 'DEV MSC FIRE SUPPRESSION FACILITIES'
            elif ctyluc == '96':
                row[1] = 'DEV MSC CEMETERIES'
            elif ctyluc == '94':
                row[1] = 'DEV MSC SCHOOLS - LARGE (101+ STUDENTS)'
            elif ctyluc == '93':
                row[1] = 'DEV MSC SCHOOLS - MEDIUM (13-100 STUDENTS)'
            elif ctyluc == '92':
                row[1] = 'DEV MSC SCHOOLS - SMALL (1-12 STUDENTS)'
            elif ctyluc == '90':
                row[1] = 'UTL IND PUBLIC UTILITY (ON STATE ASSESSED ROLL)'
            elif ctyluc == '84':
                row[1] = 'DEV MSC TEMPORARY USE CODE FOR PROJECT 184'
            elif ctyluc == '82':
                row[1] = 'DEV COM PARKING LOT'
            elif ctyluc == '81':
                row[1] = 'DEV MSC UNDERLYING INTEREST IN TIME SHARE PROJ'
            elif ctyluc == '79':
                row[1] = 'RLU MSC ENV. SENSITIVE LAND - RESTRICTED USE'
            elif ctyluc == '68':
                row[1] = 'DEV COM MARINAS'
            elif ctyluc == '65':
                row[1] = 'DEV COM RESTAURANT'
            elif ctyluc == '64':
                row[1] = 'DEV MSC SKI RESORTS'
            elif ctyluc == '63':
                row[1] = 'DEV MSC CAMPGROUNDS'
            elif ctyluc == '62':
                row[1] = 'DEV MSC COMMUNITY ORIENTED FACILITIES'
            elif ctyluc == '61':
                row[1] = 'DEV MSC MISC. IMPROVED RECREATIONAL'
            elif ctyluc == '60':
                row[1] = 'VAC MSC VACANT RECREATIONAL LAND'
            elif ctyluc == '50':
                row[1] = 'TPZ MSC TIMBER PRESERVE ZONING - ACTIVE'
            elif ctyluc == '48':
                row[1] = 'DEV IND OFFICES'
            elif ctyluc == '47':
                row[1] = 'DEV IND HOSPITALS & CONVALESCENT HOSPITALS'
            elif ctyluc == '46':
                row[1] = 'DEV IND MEDICAL/DENTAL/VET OFFICES'
            elif ctyluc == '45':
                row[1] = 'DEV IND LIGHT MANUFACTURING'
            elif ctyluc == '43':
                row[1] = 'DEV IND WAREHOUSES'
            elif ctyluc == '42':
                row[1] = 'DEV IND MINI-WAREHOUSES (MINI-STORAGE)'
            elif ctyluc == '41':
                row[1] = 'DEV IND MISC. IMPROVED INDUSTRIAL PROPERTY'
            elif ctyluc == '40':
                row[1] = 'VAC IND VACANT INDUSTRIAL LAND'
            elif ctyluc == '39':
                row[1] = 'DEV COM SUPERMARKETS'
            elif ctyluc == '38':
                row[1] = 'DEV COM RETAIL STORES >15,000 SQ. FT.'
            elif ctyluc == '37':
                row[1] = 'DEV COM RETAIL STORES 5,001-15,000 SQ. FT.'
            elif ctyluc == '36':
                row[1] = 'DEV COM RETAIL STORES <=5,000 SQ. FT.'
            elif ctyluc == '35':
                row[1] = 'DEV COM MOBILE HOME PARKS'
            elif ctyluc == '34':
                row[1] = 'DEV COM SERVICE STATION'
            elif ctyluc == '33':
                row[1] = 'DEV COM MOTEL, HOTEL'
            elif ctyluc == '31':
                row[1] = 'DEV COM MISC. IMPROVED COMMERCIAL'
            elif ctyluc == '30':
                row[1] = 'VAC COM VACANT COMMERCIAL LAND'
            elif ctyluc == '29':
                row[1] = 'DEV MSC RURAL NON-RES. IMPROVEMENT 2.51-20.0 AC.'
            elif ctyluc == '26':
                row[1] = 'AGP MSC RURAL RESTRICTIVE ZONING - NON-RENEWAL'
            elif ctyluc == '25':
                row[1] = 'AGP MSC RURAL RESTRICTIVE ZONING - CLCA (ACTIVE)'
            elif ctyluc == '24':
                row[1] = 'VAC RES RURAL RES. LAND 20+ MINOR NON-RES IMPR'
            elif ctyluc == '23':
                row[1] = 'DEV RES RURAL RES. 20+ AC. 1 RES. UNIT'
            elif ctyluc == '22':
                row[1] = 'DEV RES RURAL RES. 2.51-20.0 AC. 1 SF UNIT'
            elif ctyluc == '21':
                row[1] = 'VAC RES VAC RURAL RES LAND 2.51-20.0 AC. 1 UNIT'
            elif ctyluc == '17':
                row[1] = 'VAC MSC SUBJ. TO OPEN SPACE CONTRACT (NOT CLCA)'
            elif ctyluc == '16':
                row[1] = 'DEV RES MOBILE HOME ON RENTED LAND'
            elif ctyluc == '15':
                row[1] = 'DEV RES RESIDENCE ON LEASED LAND'
            elif ctyluc == '14':
                row[1] = 'DEV MFR CONDOMINIUMS & TOWNHOUSES'
            elif ctyluc == '13':
                row[1] = 'DEV MFR MULTI-RESIDENTIAL 4+ UNITS'
            elif ctyluc == '12':
                row[1] = 'DEV MFR MULTI-RESIDENTIAL 2-3 UNITS'
            elif ctyluc == '11':
                row[1] = 'DEV RES SINGLE FAM. RES. <=2.5 AC.(INC. MAN. HMS'
            elif ctyluc == '07':
                row[1] = 'DEV MFR RETIREMENT HOUSING'
            elif ctyluc == '05':
                row[1] = 'VAC MFR VACANT MULTI-RES. LAND 4+ UNITS ALLOWED'
            elif ctyluc == '03':
                row[1] = 'DEV COM PLACE OF WORSHIP'
            elif ctyluc == '02':
                row[1] = 'VAC RES NON-RES. IMPROVEMENTS <=2.5 AC.'
            elif ctyluc == '00':
                row[1] = 'VAC RES VACANT RES. LAND <=2.5 AC. 1-3 UNITS'
            elif ctyluc is None:
                row[1] == ''
            cursor.updateRow(row)
        # set Placer TRPA land use description
        if (row[0] != None or row[0] != "") and (row[3] == 'PL'):
            if ctyluc in ('07', '11', '12', '13', '14', '15', '17', '19', '21', '22', '23', 
                          '24', '25', '26', '27', '29', '31', '32', '36', '37', '38', 
                          '39', '62', '63', '71', '88'):
                row[2] = "Commercial"
            elif ctyluc == ('04'):
                row[2] = "Condominium"
            elif ctyluc == '89':
                row[2] = "Condominium Common Area"
            elif ctyluc in ('02', '03', '04', '05', '09', '28'):
                row[2] = "Multi-Family Residential"
            elif ctyluc in ('56', '55', '60', '61', '87', '90'):
                row[2] = "Open Space"
            elif ctyluc in ('72', '76', '77', '81'):
                row[2] = "Public Service"
            elif ctyluc in ('65', '66', '67', '68', '69'):
                row[2] = "Recreation"
            elif ctyluc in ('01', '08', '16'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('06', '18', '64'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('00', '10', '20', '30'):
                row[2] = "Vacant"
            elif ctyluc is None:
                row[2] == ''
        if (row[0] != None or row[0] != "") and (row[3] == 'PL'):
            if ctyluc == '90':
                row[1] = 'GREENBELT'
            elif ctyluc == '89':
                row[1] = 'COMMON AREA'
            elif ctyluc == '88':
                row[1] = 'HIGHWAYS, ROADS, STREETS'
            elif ctyluc == '87':
                row[1] = 'RIVERS, LAKES, RESERVOIR, CANAL'
            elif ctyluc == '81':
                row[1] = 'UTILITIES, PUBLIC & PRIVATE'
            elif ctyluc == '77':
                row[1] = 'CEMETERIES'
            elif ctyluc == '76':
                row[1] = 'MISC. PUBLIC BUILDINGS'
            elif ctyluc == '72':
                row[1] = 'SCHOOLS'
            elif ctyluc == '71':
                row[1] = 'CHURCHES'
            elif ctyluc == '69':
                row[1] = 'MISCELLANEOUS RECREATIONAL'
            elif ctyluc == '68':
                row[1] = 'CAMPS & PARKS, GENERAL'
            elif ctyluc == '67':
                row[1] = 'SKI FACILITY'
            elif ctyluc == '66':
                row[1] = 'GOLF COURSE'
            elif ctyluc == '65':
                row[1] = 'TENNIS, SWIMMING CLUBS'
            elif ctyluc == '64':
                row[1] = 'LODGES, HALLS'
            elif ctyluc == '63':
                row[1] = 'MARINA, PIER'
            elif ctyluc == '62':
                row[1] = 'THEATER, BOWLING ALLEY'
            elif ctyluc == '61':
                row[1] = 'NON-PROFIT CAMPS/PARKS'
            elif ctyluc == '60':
                row[1] = 'CONSERVATION EASEMENT RESTRICTIONS'
            elif ctyluc == '56':
                row[1] = 'TIMBERLAND, ZONED TPZ'
            elif ctyluc == '55':
                row[1] = 'TIMBERLAND, UNRESTRICTED'
            elif ctyluc == '39':
                row[1] = 'MISCELLANEOUS INDUSTRIAL'
            elif ctyluc == '38':
                row[1] = 'WAREHOUSE'
            elif ctyluc == '37':
                row[1] = 'MINI-STORAGE, COVERED STORAGE'
            elif ctyluc == '36':
                row[1] = 'UNCOVERED STORAGE, WRECKING YARD'
            elif ctyluc == '32':
                row[1] = 'HEAVY INDUSTRIAL'
            elif ctyluc == '31':
                row[1] = 'LIGHT INDUSTRIAL'
            elif ctyluc == '30':
                row[1] = 'VACANT INDUSTRIAL'
            elif ctyluc == '29':
                row[1] = "MISCELLANEOUS COMM'L"
            elif ctyluc == '28':
                row[1] = 'MOBILE HOME PARK'
            elif ctyluc == '27':
                row[1] = 'PARKING LOTS'
            elif ctyluc == '26':
                row[1] = 'AUTO SALES, REPAIR'
            elif ctyluc == '25':
                row[1] = 'SERVICE STATION'
            elif ctyluc == '24':
                row[1] = 'MINI-MARKET WITH GAS'
            elif ctyluc == '23':
                row[1] = "BANKS, S&L'S, CREDIT UNION"
            elif ctyluc == '22':
                row[1] = 'FAST FOOD RESTAURANT'
            elif ctyluc == '21':
                row[1] = 'RESTAURANTS, COCKTAIL LOUNGES'
            elif ctyluc == '20':
                row[1] = 'VACANT, COMMERCIAL'
            elif ctyluc == '19':
                row[1] = 'OFFICE MEDICAL/DENTAL'
            elif ctyluc == '18':
                row[1] = 'HOTELS, MOTELS, RESORTS'
            elif ctyluc == '17':
                row[1] = 'OFFICE GENERAL'
            elif ctyluc == '16':
                row[1] = 'RESIDENCE ON COMMERCIAL LAND'
            elif ctyluc == '15':
                row[1] = 'SHOPPING CENTER'
            elif ctyluc == '14':
                row[1] = 'OFFICE CONDO'
            elif ctyluc == '13':
                row[1] = 'MINI-MARKETS, NO GAS'
            elif ctyluc == '12':
                row[1] = 'SUBURBAN STORE'
            elif ctyluc == '11':
                row[1] = 'COMMERCIAL STORE'
            elif ctyluc == '10':
                row[1] = 'VACANT, SUBDIVIDED RESIDENTIAL'
            elif ctyluc == '09':
                row[1] = 'MOBILE HOME IN M H PARK'
            elif ctyluc == '08':
                row[1] = 'MOBILE HOME OUTSIDE OF PARK'
            elif ctyluc == '07':
                row[1] = 'RESIDENTIAL, AUXILIARY IMP'
            elif ctyluc == '06':
                row[1] = 'TIMESHARES'
            elif ctyluc == '05':
                row[1] = 'APARTMENTS, 4 UNITS OR MORE'
            elif ctyluc == '04':
                row[1] = 'SINGLE FAM RES, CONDO'
            elif ctyluc == '03':
                row[1] = '3 SINGLE FAM RES, TRIPLEX'
            elif ctyluc == '02':
                row[1] = '2 SINGLE FAM RES, DUPLEX'
            elif ctyluc == '01':
                row[1] = 'SINGLE FAM RES, HALF PLEX'
            elif ctyluc == '00':
                row[1] = 'VACANT, ALL TYPES-NOT ASGND'
            elif ctyluc is None:
                row[1] == ''
            cursor.updateRow(row)
# delete cursor
del cursor
print ("The 'EXISTING_LANDUSE' field in the parcel data has been updated")
# log.info("The 'EXISTING_LANDUSE' field in the parcel data has been updated")


### Regional Landuse Update --------------------------------------------------------------------------------------###
print("Starting the Regional Land Use Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Regional Land Use Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_RegionalLandUse, ParcelPoint_RegionalLandUse, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Regional Land Use Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Regional Land Use Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'REGIONAL_LANDUSE_TRPA'], 
              ParcelPoint_RegionalLandUse, ['APN_TRPA', 'Description'])

print ("The 'REGIONAL_LANDUSE' field in the parcel data has been updated")
# log.info("The 'REGIONAL_LANDUSE' field in the parcel data has been updated")

## Estimated Coverage Allowed Attirbute Update ------------------------------------------------------------------###
print("Starting the Estimated Coverage Allowed Identity Overlay: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Estimated Coverage Allowed Identity Overlay: " + strftime("%Y-%m-%d %H:%M:%S"))

# create out table for the stats sum
outTable =  memory + "id_Parcel_Bailey_Table"

# Create Identity Output Layer
id_ParcelLyr_BaileyLyr = memory + "id_Parcel_Bailey"

# Create Impervious Layer
Bailey_lyr = memory + "Bailey_lyr"

# Create Identity Layer
identity_layer = memory + "bailey_identity_layer"

# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(sde_Bailey, Bailey_lyr)
print ("Created feature layer of Bailey Soils")

# Process: Use the Identity function
print ("Starting Identity: "+ strftime("%Y-%m-%d %H:%M:%S"))
arcpy.Identity_analysis (ParcelLayer, Bailey_lyr, id_ParcelLyr_BaileyLyr)
print ("Finished Identity: "+ strftime("%Y-%m-%d %H:%M:%S"))

# Add SqFt field
arcpy.management.AddField(id_ParcelLyr_BaileyLyr, "SqFt", "DOUBLE", "", "", "", 
                          "Square Feet", "NULLABLE", "NON_REQUIRED", "")

# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(id_ParcelLyr_BaileyLyr, identity_layer, 
                                  where_clause = "NOT CAPABILITY in ('WB', '-1', '0')")

# calculate geometry of output identity
arcpy.CalculateField_management(identity_layer, "SqFt", "!shape.area@SQUAREFEET!", "PYTHON3", "")

# multiply square footage by bailey coefficents
with arcpy.da.UpdateCursor(identity_layer, ['CAPABILITY', 'SqFt', 'PERCENT_COVERAGE_ALLOWED']) as cur:
    for row in cur:
        if row[0] != ('','WB'):
            row[1] = row[1]*row[2]
        else:
            row[1] == 0
        cur.updateRow(row)
del cur    
# Sum the square footage
arcpy.Statistics_analysis(identity_layer, outTable, [["SqFt", "SUM"]], "APN_TRPA")

print("Finsished the Estimated Coverage Allowed Identity Overlay: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finsished the Estimated Coverage Allowed Identity Overlay: " + strftime("%Y-%m-%d %H:%M:%S"))

## Join parcel sums back to parcel layer and calculate field
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'ESTIMATED_COVERAGE_ALLOWED_TRPA'], 
              outTable, ['APN_TRPA', 'SUM_SqFt'])

print ("The 'ESTIMATED_COVERAGE_ALLOWED' field in the parcel data has been updated")

### Impervious Surface Attrigute Update --------------------------------------------------------------------------###
# create out table for the stats sum
outTable =  memory +"id_Parcel_Imp_Table"

# Create Identity Output Layer
id_ParcelLyr_ImperviousLyr = memory + "id_Parcel_Impervious"

# Create Impervious Layer
Impervious_lyr = memory + "Impervious_lyr"

# Create Identity Layer
identity_layer = memory + "identity_layer"
    
# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(sde_Impervious, Impervious_lyr)

# Process: Use the Identity function
print ("Starting Identity of Imperviuos Surface by parcel: "+ strftime("%Y-%m-%d %H:%M:%S"))
arcpy.Identity_analysis (ParcelLayer, Impervious_lyr, id_ParcelLyr_ImperviousLyr)
print ("Finished Identity of Imperviuos Surface by parcel:: "+ strftime("%Y-%m-%d %H:%M:%S"))

# Add SqFt field
arcpy.management.AddField(id_ParcelLyr_ImperviousLyr, 
                          "SqFt", "DOUBLE", "", "", "", "Square Feet", "NULLABLE", "NON_REQUIRED", "")

# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(id_ParcelLyr_ImperviousLyr, identity_layer, 
                                  where_clause = "Feature IN ('Building', 'Road', 'Other', 'Driveway')")

# calculate geometry of output identity
arcpy.CalculateField_management(identity_layer, "SqFt", "!shape.area@SQUAREFEET!", "PYTHON3", "")
                                                           
# Sum the square footage of buildings and other by APN
arcpy.Statistics_analysis(identity_layer, outTable, [["SqFt", "SUM"]], "APN_TRPA")

# Join parcel sums back to parcel layer and calculate field "Impervious Surface Sq Ft"
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'IMPERVIOUS_SURFACE_SQFT_TRPA'], 
              outTable, ['APN_TRPA', 'SUM_SqFt'])

print ("The 'ImperviousCoverage_SqFt' field in the parcel data has been updated")

### Fire District Attribute Update -------------------------------------------------------------------------------###
print("Starting the Fire District Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Fire District Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_FireDistrict, ParcelPoint_FireDistrict, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Fire District Spatial Join: "  + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Fire District Spatial Join: "  + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'FIREPD_TRPA'], 
              ParcelPoint_FireDistrict, ['APN_TRPA', 'DISTRICT'])

print ("The 'FIRE_PD' field has been updated")
# log.info("The 'FIRE_PD' field has been updated")

### Soil 1974 Attribute Update ------------------------------------------------------------------------------------### 
print("Starting the SOIL_1974 Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the SOIL_1974 Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_NRCSSoils1974, ParcelPoint_Soils74, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the SOIL_1974 Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the SOIL_1974 Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'SOIL_1974_TRPA'], 
              ParcelPoint_Soils74, ['APN_TRPA', 'MUSYM_74'])

print ("The 'SOIL_1974' field in the parcel data has been updated")
# log.info("The 'SOIL_1974' field in the parcel data has been updated")

### Soil 2003 Attribute Update -----------------------------------------------------------------------------------###
print("Starting the SOIL_2003 Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the SOIL_2003 Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_NRCSSoils2003, ParcelPoint_Soils03, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the SOIL_2003 Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the SOIL_2003 Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'SOIL_2003_TRPA'], 
              ParcelPoint_Soils03, ['APN_TRPA', 'MUSYM_03'])

print ("The 'SOIL_2003' field in the parcel data has been updated.")
# log.info("The 'SOIL_2003' field in the parcel data has been updated: "  + strftime("%Y-%m-%d %H:%M:%S"))

### HRA Attribute Upate -------------------------------------------------------------------------------------###
print("Starting the Hydrologic Area Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Hydrologic Area Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_HydroArea, ParcelPoint_HydroArea, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Hydrologic Area Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Hydrologic Area Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'HRA_NAME_TRPA'], 
              ParcelPoint_HydroArea, ['APN_TRPA', 'HRA_NAME'])

print ("The 'HRA_NAME' field in the parcel data has been updated")
# log.info("The 'HRA_NAME' field in the parcel data has been updated")

### Watshed Attribute Update -------------------------------------------------------------------------------###
print("Starting the Watershed Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Watershed Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Watershed, ParcelPoint_Watershed, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Watershed Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Watershed Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'WATERSHED_NUMBER_TRPA'], 
              ParcelPoint_Watershed, ['APN_TRPA', 'NUMBER'])

print ("The 'WATERSHED_NUMBER' field in the parcel data has been updated")
# log.info("The 'WATERSHED_NUMBER' field in the parcel data has been updated")
#
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'WATERSHED_NAME_TRPA'], 
              ParcelPoint_Watershed, ['APN_TRPA', 'NAME'])

print ("The 'WATERSHED_NAME' field in the parcel data has been updated")
# log.info("The 'WATERSHED_NAME' field in the parcel data has been updated")
#
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'PRIORITY_WATERSHED_TRPA'], 
              ParcelPoint_Watershed, ['APN_TRPA', 'PRIORITY'])

print ("The 'PRIORITY_WATERSHED' field in the parcel data has been updated")
# log.info("The 'PRIORITY_WATERSHED' field in the parcel data has been updated")

### Local Plan Attribute Update -----------------------------------------------------------------------------###
print("Starting the Local Plan Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Local Plan Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_LocalPlan, ParcelPoint_LocalPlan, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Local Plan Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Local Plan Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'PLAN_ID_TRPA'], 
              ParcelPoint_LocalPlan, ['APN_TRPA', 'PLAN_ID'])

print ("The 'PLAN_ID' field in the parcel data has been updated")
# log.info("The 'PLAN_ID' field in the parcel data has been updated")
#
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'PLAN_NAME_TRPA'], 
              ParcelPoint_LocalPlan, ['APN_TRPA', 'PLAN_NAME'])

print ("The 'PLAN_NAME' field in the parcel data has been updated")
# log.info("The 'PLAN_NAME' field in the parcel data has been updated")
#
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'PLAN_TYPE_TRPA'], 
              ParcelPoint_LocalPlan, ['APN_TRPA', 'PLAN_TYPE'])

print ("The 'PLAN_TYPE' field in the parcel data has been updated")
# log.info("The 'PLAN_NAME' field in the parcel data has been updated")
#
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'LOCAL_PLAN_HYPERLINK_TRPA'], 
              ParcelPoint_LocalPlan, ['APN_TRPA', 'File_URL'])

print ("The 'LOCAL_PLAN_HYPERLINK' field in the parcel data has been updated")
# log.info("The 'LOCAL_PLAN_HYPERLINK' field in the parcel data has been updated")

### Town Center Attribute Update --------------------------------------------------------------------------------### 
print("Starting the Town Center Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Town Center Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_TownCenter, ParcelPoint_TownCenter, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")

print("Finished the Town Center Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Town Center Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'TOWN_CENTER_TRPA'], 
              ParcelPoint_TownCenter, ['APN_TRPA', 'NAME'])

print("The 'TOWN_CENTER' field in the parcel data has been updated")
# log.info("The 'TOWN_CENTER' field in the parcel data has been updated")

### Town Center Buffer Attribute Update --------------------------------------------------------------------------###
print("Starting the Town Center Buffer Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Town Center Buffer Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_TownCenterBuffer, ParcelPoint_TownCenterBuffer, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Town Center Buffer Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Town Center Buffer Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'LOCATION_TO_TOWNCENTER_TRPA'], 
              ParcelPoint_TownCenterBuffer, ['APN_TRPA', 'BUFFER_NAME'])

print ("The 'LOCATION_TO_TOWNCENTER' field in the parcel data has been updated")
# log.info("The 'LOCATION_TO_TOWNCENTER' field in the parcel data has been updated")

### Catchment Attribute Update ------------------------------------------------------------------------------------### 
print("Starting the Catchment Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Catchment Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Catchment, ParcelPoint_Catchment, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print("Finished the Catchment Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Catchment Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'CATCHMENT_TRPA'], 
              ParcelPoint_Catchment, ['APN_TRPA', 'Name'])

print ("The 'Catchment' field in the parcel data has been updated")
# log.info("The 'Catchment' field in the parcel data has been updated")

### Littoral Parcel Attribute Update -----------------------------------------------------------------------------###
print("Identifying Littoral parcels: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info('Identifying Littoral parcels: ' + strftime("%Y-%m-%d %H:%M:%S"))

# Select parcels that have their center within
littoralSelect = arcpy.SelectLayerByLocation_management(ParcelLayer, 
                                                          'HAVE_THEIR_CENTER_IN', 
                                                           sde_Littoral, 
                                                           0, 
                                                          'NEW_SELECTION')
# Update field 1= yes 0 = no
with arcpy.da.UpdateCursor(littoralSelect, ['LITTORAL_TRPA']) as cursor:
    for row in cursor:
        row[0] = '1'
        cursor.updateRow(row) 
del cursor 

# switch the selection
litSelect = arcpy.SelectLayerByAttribute_management(littoralSelect,'SWITCH_SELECTION')

# update other parcels
with arcpy.da.UpdateCursor(litSelect, ['LITTORAL_TRPA']) as cursor:
    for row in cursor:
        row[0] = '0'
        cursor.updateRow(row)
del cursor
print("Littoral parcels updated: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Littoral parcels Updated: " + strftime("%Y-%m-%d %H:%M:%S"))

### Tolerance ID -------------------------------------------------------------------------------------------------###
print("Starting the Tolerance District Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Tolerance, ParcelPoint_Tolerance, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "INTERSECT", "", "")
print ("Finished the Tolerance District Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'TOLERANCE_ID_TRPA'], 
              ParcelPoint_Tolerance, ['APN_TRPA', 'DISTRICT'])

print ("The Tolerance ID field in the parcel data has been updated")

### Index 1987 Attribute Update ----------------------------------------------------------------------------------###
print("Starting the 1987 Index Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the 1987 Index Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Index1987, ParcelPoint_Index1987, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the 1987 Index Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the 1987 Index Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'INDEX_1987_TRPA'], 
              ParcelPoint_Index1987, ['APN_TRPA', 'MAP_NUMBER'])

print("The 'INDEX_1987' field in the parcel data has been updated")
# log.info("The 'INDEX_1987' field in the parcel data has been updated")
#
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'INDEX_1987_HYPERLINK_TRPA'], 
              ParcelPoint_Index1987, ['APN_TRPA', 'MAP_PATH'])

print ("The 'INDEX_1987_HYPERLINK' field in the parcel data has been updated")
# log.info("The 'INDEX_1987_HYPERLINK' field in the parcel data has been updated")

### Postal Town Field --------------------------------------------------------------------------------------------### 
print("Starting the Postal Town Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Postal Town Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Zip, ParcelPoint_PstlTown, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Postal Town Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Postal Town Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# Transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'PSTL_TOWN_TRPA'], 
              ParcelPoint_PstlTown, ['APN_TRPA', 'PO_NAME'])

print ("The 'PSTL_TOWN' field in the parcel data has been updated")
# log.info("The 'PSTL_TOWN' field in the parcel data has been updated")

### Postal ZIP ---------------------------------------------------------------------------------------------------###
print("Starting the Postal Zip Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Postal Zip Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Zip, ParcelPoint_PstlZip, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print("Finished the Postal Zip Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Postal Zip Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# Transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'PSTL_ZIP5_TRPA'], 
              ParcelPoint_PstlZip, ['APN_TRPA', 'ZIP_CODE'])

print("The 'PSTL_ZIP5' field in the parcel data has been updated")
# log.info("The 'PSTL_ZIP5' field in the parcel data has been updated")

### CSLT Jurisdiction Update -------------------------------------------------------------------------------------###
print("Starting to select parcels within City of South Lake Tahoe: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting to select parcels within City of South Lake Tahoe: " + strftime("%Y-%m-%d %H:%M:%S"))

# select by location
csltParcels = arcpy.SelectLayerByLocation_management(ParcelLayer, "HAVE_THEIR_CENTER_IN", sde_CSLT, 0,   
                                                     "NEW_SELECTION")
# update jurisdcition field
with arcpy.da.UpdateCursor(csltParcels, ["JURISDICTION_TRPA"]) as cursor:
    for row in cursor:
        row[0] = "City of South Lake Tahoe"
        # update all rows
        cursor.updateRow(row)
del cursor 
print("Finished updating parcels within City of South Lake Tahoe: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished updating parcels within City of South Lake Tahoe:  " + strftime("%Y-%m-%d %H:%M:%S"))
print("JURISDCITION field update with 'CSLT' values ")
# log.info("JURISDCITION field update with 'CSLT' values ")

### Zoning Attribute Update --------------------------------------------------------------------------------------###
# Spatial Join
print("Starting the Zoning Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Zoning Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Zoning, ParcelPoint_Zoning, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print("Finished the Zoning Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Zoning Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'ZONING_ID_TRPA'],
              ParcelPoint_Zoning, ['APN_TRPA', 'ZONING_ID'])

print("The Zoning ID field in the parcel data has been updated")
# log.info("The Zoning ID field in the parcel data has been updated")
#
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'ZONING_DESCRIPTION_TRPA'], 
              ParcelPoint_Zoning, ['APN_TRPA', 'ZONING_DESCRIPTION'])

print("The Zoning Description field in the parcel data has been updated")
# log.info("The Zoning Description field in the parcel data has been updated")
#
fieldJoinCalc(ParcelLayer, ['APN_TRPA', "DESIGN_GUIDELINES_HYPERLINK_TRPA"], 
              ParcelPoint_Zoning, ['APN_TRPA', "DESIGN_GUIDELINES_HYPERLINK"])

print("The DESIGN_GUIDELINES_HYPERLINK field in the parcel data has been updated")
# log.info("The DESIGN_GUIDELINES_HYPERLINK_TRPA field in the parcel data has been updated")

### TAZ Attirbute Update --------------------------------------------------------------------------------------###
# Spatial Join
print("Starting the TAZ Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the TAZ Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
arcpy.SpatialJoin_analysis(ParcelPoint, sde_TAZ, ParcelPoint_TAZ, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print("Finished the TAZ Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the TAZ Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN_TRPA', 'TAZ_TRPA'],
              ParcelPoint_TAZ, ['APN_TRPA', 'TAZ'])

print("The TAZ field in the parcel data has been updated")

### LTinfo Parcel Details Hyperlink Attribute Update -------------------------------------------------------------###
print("Creating LTinfo Hyperlinks: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Creating LTinfo Hyperlinks: " + strftime("%Y-%m-%d %H:%M:%S"))

# create ltinfo hyper link
with arcpy.da.UpdateCursor(ParcelLayer, ["APN_TRPA","LTINFO_HYPERLINK_TRPA"]) as cursor:
    for row in cursor:
        if not (row[0] == None):
            row[1] = 'https://parcels.laketahoeinfo.org/Parcel/Detail/'+ row[0]
        else:
            row[1] = ''
        cursor.updateRow(row)
del cursor
print("The LTINFO_HYPERLINK field in the parcel data has been updated")

### set within TRPA boundary -------------------------------------------------------------------------------------###
print("Identifying parcels within TRPA Boundary: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info('Identifying parcels within TRPA Boundary: ' + strftime("%Y-%m-%d %H:%M:%S"))

# Select all new parcels that have their center within
parcelSelect = arcpy.SelectLayerByLocation_management(ParcelLayer, 
                                                          'INTERSECT', 
                                                           sde_TRPAboundary, 
                                                           0, 
                                                          'NEW_SELECTION')

# Update field 1= yes 0 = no
with arcpy.da.UpdateCursor(parcelSelect, ['WITHIN_TRPA_BNDY_TRPA']) as cursor:
    for row in cursor:
        row[0] = '1'
        cursor.updateRow(row) 
del cursor        
# switch the selection
parcelSelect = arcpy.SelectLayerByAttribute_management(parcelSelect,'SWITCH_SELECTION')

# update other parcels
with arcpy.da.UpdateCursor(parcelSelect, ['WITHIN_TRPA_BNDY_TRPA']) as cursor:
    for row in cursor:
        row[0] = '0'
        cursor.updateRow(row)
del cursor
print("Within TRPA Boundary Updated: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Within TRPA Boundary Updated: " + strftime("%Y-%m-%d %H:%M:%S"))

### set within Bonus Unit Boundary -------------------------------------------------------------------------------###
print("Identifying parcels within bonus unit boundary: "  + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Identifying parcels within bonus unit boundary: " + strftime("%Y-%m-%d %H:%M:%S"))

# Select all new parcels that have their center within
parcelSelect = arcpy.SelectLayerByLocation_management(ParcelLayer, 
                                                          'HAVE_THEIR_CENTER_IN', 
                                                           sde_BonusUnitboundary, 
                                                           0, 
                                                          'NEW_SELECTION')

with arcpy.da.UpdateCursor(parcelSelect, ['WITHIN_BONUSUNIT_BNDY_TRPA']) as cursor:
    for row in cursor:
        row[0] = '1'
        cursor.updateRow(row) 
del cursor   
# switch the selection
parcelSelect = arcpy.SelectLayerByAttribute_management(parcelSelect,'SWITCH_SELECTION')

with arcpy.da.UpdateCursor(parcelSelect, ['WITHIN_BONUSUNIT_BNDY_TRPA']) as cursor:
    for row in cursor:
        row[0] = '0'
        cursor.updateRow(row)
del cursor     
print("Bonus Unit Boundary Updated: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Bonus Unit Boundary Updated: " + strftime("%Y-%m-%d %H:%M:%S"))

### Calculate Area Field------------------------------------------------------------------------------------------###
print("Calculating Acres..." + strftime("%Y-%m-%d %H:%M:%S"))
with arcpy.da.UpdateCursor(ParcelLayer, ['PARCEL_ACRES_TRPA', 'SHAPE@']) as cursor:
    for row in cursor:
        row[0] = row[1].getArea('PLANAR', 'ACRES')
        cursor.updateRow(row)
del cursor

# calculate square feet
print("Calculating Square Feet..." + strftime("%Y-%m-%d %H:%M:%S"))
with arcpy.da.UpdateCursor(ParcelLayer, ['PARCEL_SQFT_TRPA', 'SHAPE@']) as cursor:
    for row in cursor:
        row[0] = row[1].getArea('PLANAR', 'SquareFeetUS')
        cursor.updateRow(row)
del cursor

### Copy to Feature Class ----------------------------------------------------------------------------------------###
print("Copying memory features to staging: " + strftime("%Y-%m-%d %H:%M:%S"))
# copy in-memory features to staging feature class
arcpy.CopyFeatures_management(ParcelLayer, ParcelNew)

print("Copied in-memory features, parcel staging new is set: " + strftime("%Y-%m-%d %H:%M:%S"))

arcpy.Delete_management("memory")
print("Deleted Memory Workspace: " + strftime("%Y-%m-%d %H:%M:%S"))

Starting TRPA Attribution: 2023-05-04 20:13:13
Copied features to points: 2023-05-04 20:13:45
Starting the County attribute update: 2023-05-04 20:13:45
County Attribute Updated
Starting the Ownership Type attribute update: 2023-05-04 20:13:48
The 'OWNERSHIP_TYPE' field in the parcel data has been updated
The 'EXISTING_LANDUSE' field in the parcel data has been updated
Starting the Regional Land Use Spatial Join: 2023-05-04 20:13:56
Finished the Regional Land Use Spatial Join: 2023-05-04 20:14:22
Started data transfer: 2023-05-04 20:14:22
Finished data transfer: 2023-05-04 20:14:27
The 'REGIONAL_LANDUSE' field in the parcel data has been updated
Starting the Estimated Coverage Allowed Identity Overlay: 2023-05-04 20:14:27
Created feature layer of Bailey Soils
Starting Identity: 2023-05-04 20:14:29
Finished Identity: 2023-05-04 20:15:17
Finsished the Estimated Coverage Allowed Identity Overlay: 2023-05-04 20:15:51
Started data transfer: 2023-05-04 20:15:51
Finished data transfer: 2023-05

In [ ]:
result = arcpy.GetCount_management("Parcel_Staging")
print('{} has {} records'.format("Parcel_Staging", result[0]))
result = arcpy.GetCount_management("Parcel_Staging_Attributed")
print('{} has {} records'.format("Parcel_Staging_Attributed", result[0]))

In [11]:
# Describe the feature class and get its spatial reference   
desc = arcpy.Describe("Parcel_Staging_Attributed")
spatialRef = desc.spatialReference
 
# Print the spatial reference name
print (spatialRef.Name)

NAD_1983_UTM_Zone_10N


### Create Parcel County Staging Feature Class

In [12]:
staging_fc     = "Parcel_Staging_Attributed"
new_fc         = "Parcel_County_Staging" 

# Create FieldMappings object to manage merge output fields
fieldMappings = arcpy.FieldMappings()
# # Add all fields from all parcel staging layers
# fieldMappings.addTable(fc)

for field in arcpy.ListFields(staging_fc):
    if not field.name == "OBJECTID" and not field.name == "Shape":
        old_name = field.name

        #Rename if necessary
        if old_name.endswith("_TRPA"):
            new_name = old_name[:-5]
        else:
            new_name = old_name

        #Create new FieldMap object    
        new_f = arcpy.FieldMap()
        new_f.addInputField(staging_fc, old_name) # Specify the input field to use

        #Rename output field
        new_f_name = new_f.outputField
        new_f_name.name = new_name
        new_f_name.aliasName = new_name
        new_f.outputField = new_f_name

        #Add field to FieldMappings object
        fieldMappings.addFieldMap(new_f)

#Convert fc using new field names
arcpy.FeatureClassToFeatureClass_conversion(staging_fc, 
                                            os.path.dirname(new_fc), 
                                            os.path.basename(new_fc), 
                                            field_mapping=fieldMappings)

arcpy.DeleteField_management("Parcel_County_Staging", 
                             ["OBJECTID_1"])

<Result '//Trpa-fs01/GIS/PARCELUPDATE/Workspace/ParcelStaging.gdb\\Parcel_County_Staging'>

## QA QC

In [12]:
fc = "Parcel_County_Staging"

apn = 'APN'


# Create an expression with proper delimiters
expression = u"{} = '034-402-001'".format(arcpy.AddFieldDelimiters(fc, apn))

# Create a search cursor using an SQL expression
with arcpy.da.SearchCursor(fc, ['APO_ADDRESS'],
                           where_clause=expression) as cursor:
    for row in cursor:
        # Print the name of the residential road
        print(row)

('2950 US HWY 50',)


In [13]:
fc = "Parcel_County_Staging"

apn = 'APN'


# Create an expression with proper delimiters
expression = u"{} = '094-520-001'".format(arcpy.AddFieldDelimiters(fc, apn))

# Create a search cursor using an SQL expression
with arcpy.da.SearchCursor(fc, ['APO_ADDRESS'],
                           where_clause=expression) as cursor:
    for row in cursor:
        # Print the name of the residential road
        print(row)

('NO ADDRESS ON FILE',)


## Load

### Obsolete and New Parcels

In [10]:
#Function definition
def make_old_new_dataframe(old_feature_class, new_feature_class, TRPA_boundary, prefix_remove)
    df_old = pd.DataFrame.spatial.from_featureclass(old_feature_class)
    df_new = pd.DataFrame.spatial.from_featureclass(new_feature_class)
    merge_df = pd.merge(df_old, df_new,  how='outer', on=['APN'], indicator=True)
    merge_df.query('_merge!="both"', inplace=True)
    df_merge.loc[df_merge['_merge']=='right_only', 'Status']='New APN'
    # define Left Only as Old APNs
    df_merge.loc[df_merge['_merge']=='left_only', 'Status']='Old APN'
    df_merge.dropna(subset=['APN'], inplace=True) 
    df_merge = df_merge.loc[~df_merge['APN'].str.startswith(prefix_remove)]

    date = datetime.date.today().strftime("%m/%d/%Y")
    df_merge['DiscoveryDate'] = date

    # final list of fields
    df_merge = df_merge[['APN','Status','DiscoveryDate','WITHIN_TRPA_BOUNDARY']]
    if TRPA_boundary == 'Yes':
        df_merge = df_merge.loc[df_merge['WITHING_TRPA_BOUNDARY']==1]
    return df_merge
        
def old_new_parcels_list(old_feature_class, new_feature_class, TRPA_boundary, prefix_remove, old_new):
    df_merge = make_old_new_dataframe(old_feature_class, new_feature_class, TRPA_boundary, prefix_remove)
    parcel_list = df_merge.loc[df_merge['Status']==old_new,'APN'].tolist()
    return parcel_list

#Identify differences between APNs that haven't changed
def return_matching_apns(feature_class_old, feature_class_new, parcels_ignore):
    dfOld = pd.DataFrame.spatial.from_featureclass(feature_class_old)
    dfOld = dfOld[~dfOld['APN'].isin(parcels_ignore['APN'])]
    dfNew = pd.DataFrame.spatial.from_featureclass(feature_class_new)
    matching_apns  = pd.merge(dfOld, dfNew,  how='inner', on=['APN'])
    matching_apns =pd.unique(matching_apns['APN'])
    return matching_apns

def delete_old_parcels(featureLayer, oldAPNs):
    delete_count = 0
    with arcpy.da.UpdateCursor(featureLayer, ["APN"]) as cursor:
        for row in cursor:
            apn = row[0]
            if apn in oldAPNs:
                cursor.deleteRow()
                delete_count +=1
    print(f"{delete_count} rows deleted from {featureLayer}.")

def insert_new_parcels(featureLayer, new_APNs, new_parcels, fields):
    new_count = 0
    where_clause = f"{arcpy.AddFieldDelimiters(featureLayer, 'APN')} IN " + str(tuple(new_APNs))
    with arcpy.da.SearchCursor(new_parcels, "*", where_clause) as search_cursor:
    # Open an insert cursor to the destination feature class
        with arcpy.da.InsertCursor(featureLayer, fields) as insert_cursor:
            # Loop through the selected rows and insert them into the destination feature class
            for row in search_cursor:
                insert_cursor.insertRow(row)
                new_count +=1
                print(f"{new_count} rows inserted into {featureLayer}.")
            
def update_parcel_geometry(featureLayer, new_parcels):
    newShapes = arcpy.management.SelectLayerByLocation(
    in_layer=new_parcels,
    overlap_type="ARE_IDENTICAL_TO",
    select_features=featureLayer,
    search_distance=None,
    selection_type="NEW_SELECTION",
    invert_spatial_relationship="INVERT"
    )

    # update SHAPE object with new 
    fieldJoinCalc(featureLayer,['APN','SHAPE@'],newShapes,['APN','SHAPE@'])

    # Get the count of selected features
    result = arcpy.management.GetCount(newShapes)
    count = int(result.getOutput(0))
    # number of shapes shifted
    print(f"{count} shapes shifted.")



In [11]:
#Generate old new apn lists
#Currently no sde.county_parcel_staging
parcelMaster   = sdeBase + "\\sde.SDE.Parcels\\sde.SDE.Parcel_Master"
parcelBase = sdeBase + "\\sde.SDE.Parcels\\sde.SDE.Parcel_Base"
parcelNew      = "Parcel_County_Staging"


prefix_remove = ('880','881','910','920')
parcel_master_new_apn = old_new_parcels_list(parcelMaster, parcelNew, 'Yes', prefix_remove,'New APN')
parcel_master_old_apn = old_new_parcels_list(parcelMaster, parcelNew, 'Yes', prefix_remove,'Old APN')
parcel_base_new_apn = old_new_parcels_list(parcelBase, parcelNew, 'Yes', prefix_remove,'New APN')
parcel_base_old_apn = old_new_parcels_list(parcelBase, parcelNew, 'Yes', prefix_remove,'Old APN')

### Version Management

#### Sign into Portal

In [67]:
## TRPA_ADMIN credentials 
portal_user = "TRPA_PORTAL_ADMIN"
portal_pwd = "@dmin6224"
portal_url = "https://maps.trpa.org/portal/"
# sign in
arcpy.SignInToPortal(portal_url, portal_user, portal_pwd)

{'token': 'G9qnwCRW2BwOGxr9kSunS2y-Cbpa2UuJp5uv7k_Th0ikfe7NkRpxiSxvnb-Lle00R40kga67w8f-a_DXnO_iASmSqy-IhkjgqmvI5y3vAYEHzAAHflbWlQqv1UfcfpLhsPlRE_1-aakVdSr5dj5LjcieWnbx_04Z8WZHFe2OuPlh7qh3drHAAPU5CmGtaNoidazrhjZJzsE4RVc9nPpS6UXYA03qxMKITGtdFK6BQa9ij6xN0hufrCflUyjZqqiI',
 'referer': 'http://www.esri.com/AGO/2F27EA4F-8CEF-4403-AEA5-C3F47E0D0D69',
 'expires': 1683304282}

#### Create New Version

In [64]:
## CREATE NEW VERSION - maybe do for each parcel update
# parent version
workspace_parent = r"https://maps.trpa.org/server/rest/services/Parcel_Edits/FeatureServer"
parent_version = "SDE.DEFAULT"
# Define the name of the new branch version and the access level

version_name = "Parcel_Update_" + strftime("%Y-%m-%d")
version_name_full = portal_user + "." + version_name
access = "PUBLIC"

#Create the new branch version
arcpy.CreateVersion_management(workspace_parent, parent_version, version_name, access)

<Result 'https://maps.trpa.org/server/rest/services/Parcel_Edits/FeatureServer'>

### Update_Parcel_Master

In [3]:




#This needs to be shifted over to SQL Table
df_special_parcels= pd.read_excel("//Trpa-fs01/GIS/PARCELUPDATE/Workspace/special_parcels.xlsx")

matching_apns_parcel_master = return_matching_apns(parcelMaster, parcelNew, df_special_parcels)

49067
49067


In [4]:
#Get a dictionary of all differences between parcel_master and parcel_new
dfparcelMaster = dfparcelMaster[dfparcelMaster['APN'].isin(matching_apns_parcel_master)]
dfparcelNew = dfparcelNew[dfparcelNew['APN'].isin(matching_apns_parcel_master)]
fields_to_ignore = ['SHAPE', 'ESTIMATED_COVERAGE_ALLOWED']
differences_master = differenceDictionary(df_master, df_new, 'APN', fields_to_ignore)


64
64
Updating Attributes stated: 2023-05-02 05:02:27
Updating Attributes Finished: 2023-05-02 05:02:31


In [10]:
# parcel master branch versioned feature service
parcelMaster = r"https://maps.trpa.org/server/rest/services/Parcel_Edits/FeatureServer/4"
# feature layer name
featureLayer= 'parcelMaster'
# make feature layer
arcpy.management.MakeFeatureLayer(parcelMaster, featureLayer)
# change to version to edit
arcpy.management.ChangeVersion(featureLayer, "BRANCH", version_full)
# function to update attributes
update_fc_from_dict(differences_master, 'APN', featureLayer)

Updating Attributes stated: 2023-05-02 05:16:08
Updating Attributes Finished: 2023-05-02 09:45:31


In [ ]:
#Update Geometry Parcels_Master
#Need to define field list for parcel master
fields = list(set(dfparcelMaster.columns) & set(dfparcelNew.columns))
delete_old_parcels(featureLayer, parcel_master_old_apn)
insert_new_parcels(featureLayer, parcel_master_new_apn, parcelNew, fields)
update_parcel_geometry(featureLayer, parcelNew)

### Update_Parcel_Base

In [ ]:
#Switch featureclasses
parcelBase = r"https://maps.trpa.org/server/rest/services/Parcel_Edits/FeatureServer/4"
# feature layer name
featureLayer= 'parcelBase'
# make feature layer
arcpy.management.MakeFeatureLayer(parcelBase, featureLayer)
# change to version to edit
arcpy.management.ChangeVersion(featureLayer, "BRANCH", version_full)

In [ ]:
dfparcelBase = dfparcelBase[dfparcelBase['APN'].isin(matching_apns_parcel_base)]
#This needs to be redefined
dfparcelNew = dfparcelNew[dfparcelNew['APN'].isin(matching_apns_parcel_base)]
fields_to_ignore = ['SHAPE']
differences_base = differenceDictionary(dfparcelBase, dfparcelNew, 'APN', fields_to_ignore)

update_fc_from_dict(differences_base, 'APN', featureLayer)

In [ ]:
#Geometry Updates
fields = ['APN','PPNO','PARCEL_ACRES','PARCEL_SQFT','JURISDICTION','SHAPE@']
delete_old_parcels(featureLayer, parcel_base_old_apn)
insert_new_parcels(featureLayer, parcel_base_new_apn, parcelNew, fields)
update_parcel_geometry(featureLayer, parcelNew)

### Update Geometry - Delete old, Insert new, Update existing.

#### Setup

#### Parcel Points Geometry Update

#### Parcel Master Geometry Updates

In [ ]:
def delete_old_parcels(featureLayer, oldAPNs):
    delete_count = 0
    with arcpy.da.UpdateCursor(featureLayer, ["APN"]) as cursor:
        for row in cursor:
            apn = row[0]
            if apn in oldAPNs:
                cursor.deleteRow()
                delete_count +=1
    print(f"{delete_count} rows deleted from {featureLayer}.")

def insert_new_parcels(featureLayer, new_APNs, new_parcels, fields):
    new_count = 0
    where_clause = f"{arcpy.AddFieldDelimiters(featureLayer, 'APN')} IN " + str(tuple(new_APNs))
    with arcpy.da.SearchCursor(new_parcels, "*", where_clause) as search_cursor:
    # Open an insert cursor to the destination feature class
        with arcpy.da.InsertCursor(featureLayer, fields) as insert_cursor:
            # Loop through the selected rows and insert them into the destination feature class
            for row in search_cursor:
                insert_cursor.insertRow(row)
                new_count +=1
                print(f"{new_count} rows inserted into {featureLayer}.")
            
def update_parcel_geometry(featureLayer, new_parcels):
    newShapes = arcpy.management.SelectLayerByLocation(
    in_layer=new_parcels,
    overlap_type="ARE_IDENTICAL_TO",
    select_features=featureLayer,
    search_distance=None,
    selection_type="NEW_SELECTION",
    invert_spatial_relationship="INVERT"
    )

    # update SHAPE object with new 
    fieldJoinCalc(featureLayer,['APN','SHAPE@'],newShapes,['APN','SHAPE@'])

    # Get the count of selected features
    result = arcpy.management.GetCount(newShapes)
    count = int(result.getOutput(0))
    # number of shapes shifted
    print(f"{count} shapes shifted.")

    
#Branch Version editing service
parcelMaster    = r"https://maps.trpa.org/server/rest/services/Parcel_Edits/FeatureServer/4"
# name the feature layer
featureLayer = 'parcelMaster'
# make feature layer 
arcpy.management.MakeFeatureLayer(parcelMaster, featureLayer)
# branch edit version
arcpy.management.ChangeVersion(featureLayer, "BRANCH", "TRPA_PORTAL_ADMIN.Base_Edits")







#### Parcel Base Geometry Updates

In [66]:
## Delete Obsolete APNs by using the OLD APN list
#Branch Version editing service
parcelBase    = r"https://maps.trpa.org/server/rest/services/Parcel_Edits/FeatureServer/1"
# # set workspace? why?
# env.workspace = parcelBase
# name the feature layer
featureLayer = 'parcelBase'
# make feature layer 
arcpy.management.MakeFeatureLayer(parcelBase, featureLayer)
# branch edit version
arcpy.management.ChangeVersion(featureLayer, "BRANCH", "TRPA_PORTAL_ADMIN.Base_Edits")

## DELETE OLD PARCELS
# loop on APN in Old APN to update the feature layer from the Branch Versioned Service
with arcpy.da.UpdateCursor(featureLayer, ["APN"]) as cursor:
    for row in cursor:
        apn = row[0]
#         print(apn)
        if apn in oldAPNs:
            print(apn)
            cursor.deleteRow()

# Print a message to indicate the number of rows deleted
num_deleted = len(oldAPNs)
print(f"{num_deleted} rows deleted.")

## INSERT NEW PARCELS
# where clause to create list of New APNs
where_clause = f"{arcpy.AddFieldDelimiters(featureLayer, 'APN')} IN " + str(tuple(newAPNs))

# parcel base fields
fields = ['APN','PPNO','PARCEL_ACRES','PARCEL_SQFT','JURISDICTION','SHAPE@']

# search curosr to narrow down the APNs to insert and insert cursor to insert those
#Change to use branch versioning  - we might have to remove the where clause
with arcpy.da.SearchCursor(parcelNew, fields, where_clause) as search_cursor:
    # Open an insert cursor to the destination feature class
    with arcpy.da.InsertCursor(featureLayer, fields) as insert_cursor:
        # Loop through the selected rows and insert them into the destination feature class
        for row in search_cursor:
            insert_cursor.insertRow(row)

# number of rows inserted
num_inserted = len(newAPNs)
print(f"{num_inserted} rows inserted into.")

## UPDATE EXISTING GEOMETRIES
            
# select parcels that are not identical to existing
newShapes = arcpy.management.SelectLayerByLocation(
    in_layer=parcelNew,
    overlap_type="ARE_IDENTICAL_TO",
    select_features=featureLayer,
    search_distance=None,
    selection_type="NEW_SELECTION",
    invert_spatial_relationship="INVERT"
)

# update SHAPE object with new
fieldJoinCalc(featureLayer,['APN','SHAPE@'],newShapes,['APN','SHAPE@'])

# Get the count of selected features
result = arcpy.management.GetCount(newShapes)
count = int(result.getOutput(0))
# number of shapes shifted
print(f"{count} shapes shifted.")

Old APNs:
['028-081-015', '110-070-008', '036-160-004', '110-070-015', '1318-25-110-009', '023-182-028', '026-031-014', '036-160-002', '036-300-000', '115-030-016', '036-170-010', '092-200-022', '098-180-016', '026-031-013', '117-190-060', '097-200-014', '048-140-01', '094-070-002', '029-095-026', '036-170-004', '036-160-005', '090-282-018', '090-225-002', '030-352-027', '1418-15-110-020', '110-070-014', '036-563-014', '029-412-006', '036-170-003', '036-170-021', '091-090-009', '097-091-009', '112-050-018', '036-170-015', '031-213-003', '092-200-009', '029-441-004', '116-080-003', '094-440-010', '122-181-64', '090-153-011', '1318-22-710-006', '122-181-65', '112-010-011', '025-744-011', '097-130-030', '036-170-006', '1318-22-002-113', '1318-22-710-010', '036-160-003', '029-240-011', '097-192-010', '094-470-026', '035-291-006', '036-170-013', '117-160-010', '1219-00-001-001', '036-170-011', '048-140-02', '036-170-007', '036-160-006', '036-160-001', '036-170-008', '112-190-048', '083-220-